# 07 — Canonical EDA and Exact Decomposition

**Notebook**: `notebooks/07_eda_and_decomposition.ipynb` · **Run id**: `NB07_CANONICAL_EDA_v001`

| record | value |
|---|---|
| `EXECUTION_BASE_TYPE` | `AUDITED_G5_HEAD` |
| `EXECUTION_BASE_SHA` | `9d99e13026b89dcf7d8846d0c105a811f64274bc` |
| G5 research head | `35a40e7c3541eff7a41ce853204409b128a6d676` |
| G5 independent audit | `9d99e13026b89dcf7d8846d0c105a811f64274bc` |
| branch | `research/nb07-canonical-eda-20260818` |
| worktree | `/home/sieg/projects-wsl/KOEN_nb07_20260818` |

## 진입 권한 (Entry authority)

직접 재독한 SSOT: `KOEN-TP-RS-001` (REDLINE) §5 · §6 · §7 · §8 · §9 · §12 · §13 · §15 · §16 · §17 · §20 · §21 · §31 · §32 · §35,
`RD-FAST-G5-01`, `RD-SSOT-CANONICAL-RETURN-01`, `RD-G5-ENTRY-GOVERNANCE-CLOSEOUT-01`,
`ssot_g5/02_G5_DIAGNOSTIC_PROTOCOL_v001.md`, G5 adjudication
(`2026-08-18_1325_..._G5_ANALYSIS_READINESS_ADJUDICATION.md`), G5 independent audit
(`2026-08-18_1344_..._G5_INDEPENDENT_AUDIT.md`).

감사 판정: `G5_INDEPENDENT_AUDIT_PASS` · `G5_ANALYSIS_READINESS_PASS_WITH_NOTES`.

G5 science/audit는 **닫혔다**. default-branch publish는 로컬 harness 제약으로
`CANONICALIZATION_PENDING_OPERATIONAL_ONLY` 상태이며, 이는 과학/Gate blocker가 아니다.
본 노트북은 stale local main이 아니라 **감사된 G5 head**에서 분기한 worktree에서 실행된다.

## 이 노트북의 과학 경계 (Science boundary)

| RQ | NB07에서 하는 일 | NB07에서 하지 않는 일 |
|---|---|---|
| RQ1 | NB08에서 이미 닫힌 primary inference를 **주석(annotate)** 한다 | 재검정하지 않는다 |
| RQ2 | exact decomposition — **NB07의 canonical 결과** | — |
| RQ3 | 표면형 기술 구조(descriptive)만 | 조건부 설명 결과는 NB09 M1 vs M0 |
| RQ4 | 형태소 분포만 | 증분 설명력은 NB09 M2 vs M1 |
| RQ5 | regex-chunk 구조만 | 조건부 mechanism 기여는 NB09 M3 vs M2 |
| RQ6 | 기술적 이질성(descriptive heterogeneity)만 | 모형 기반 효과는 이후 단계 |

**인과 표현은 이 노트북 전체에서 금지된다.** 어떤 계수도 추정하지 않는다.
모든 canonical 수치는 이 실행에서 새로 계산된다.

## Legacy notebook 정책

`notebooks/exploratory/EDA_representation_kiwi_o200k_casebook.ipynb` 는
`LEGACY_UNTRACKED_REFERENCE_ONLY` 로 분류된다. 편집·개명·수치 복사·해석 복사를 하지 않는다.
본 실행은 그 파일을 열지 않았고, 어떤 수치도 그 파일에서 가져오지 않았다.

```
LEGACY_CASEBOOK_NUMERICAL_SOURCE = NO
```

## 애드덤 (2026-08-18) — PRE-NB09 descriptive evidence

§14~§18 은 `DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW` 로 표기된 보완 증거다.
새로운 inference/model fitting은 없으며, NB09 decision이나 feature selection으로 승격하지 않는다.

## 01 — 실행 환경, 경로, 한국어 시각화 계약

왜: 모든 canonical figure는 한국어 title/axis/legend를 가져야 하고(§8),
실제로 어떤 font 파일이 사용되었는지 기록되어야 한다. silent fallback은 금지된다.
font 탐색 순서는 `Noto Sans CJK KR` → `NanumGothic` → `Malgun Gothic` 이다.

In [1]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import platform
import sys
import time
import warnings
from pathlib import Path
from zoneinfo import ZoneInfo

import duckdb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from matplotlib import font_manager
from matplotlib.colors import LogNorm
from matplotlib.ft2font import FT2Font
from scipy import stats

KST = ZoneInfo("Asia/Seoul")
RUN_ID = "NB07_CANONICAL_EDA_v001"
EXECUTION_BASE_SHA = "9d99e13026b89dcf7d8846d0c105a811f64274bc"
EXECUTION_BASE_TYPE = "AUDITED_G5_HEAD"
G5_RESEARCH_HEAD = "35a40e7c3541eff7a41ce853204409b128a6d676"

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, (ROOT / "src").as_posix())

REG = ROOT / "data" / "registry"
RUNTIME = ROOT / ".runtime" / "nb07"
FIGDIR = ROOT / "outputs" / "figures" / "nb07"
REPDIR = ROOT / "outputs" / "reports"
MANDIR = ROOT / "outputs" / "manifests"
PRIVDIR = ROOT / "outputs" / "manual_audit" / "nb07_private"
for d in (RUNTIME, FIGDIR, REPDIR, MANDIR, PRIVDIR):
    d.mkdir(parents=True, exist_ok=True)

from tokenization_premium.telemetry import RuntimeTelemetry  # noqa: E402

EXPECTED_N = 3_835_988
EXPECTED_PAIR_SET = "d9660d654ee449e4d0c23a0070225274"

# ---- 한국어 font: 우선순위대로 실제 glyph coverage를 확인한 뒤 확정한다 -------------
KOREAN_PRIORITY = ("Noto Sans CJK KR", "NanumGothic", "Malgun Gothic")
KOREAN_PROBE = "한글 토큰화 재현성 형태소 청크 프리미엄"

for _collection in Path("/usr/share/fonts").rglob("*.tt[cf]"):
    if "CJK" in _collection.name:                       # .ttc sub-face를 family로 등록한다
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            font_manager.fontManager.addfont(_collection.as_posix())

_by_family: dict[str, str] = {}
for _e in font_manager.fontManager.ttflist:
    _by_family.setdefault(_e.name, _e.fname)

KOREAN_PLOT_FONT, KOREAN_PLOT_FONT_PATH, FONT_TRACE = None, None, []
for _fam in KOREAN_PRIORITY:
    _path = _by_family.get(_fam)
    if _path is None:
        FONT_TRACE.append({"family": _fam, "status": "NOT_INSTALLED"})
        continue
    try:
        _cmap = FT2Font(_path).get_charmap()
    except (RuntimeError, OSError) as exc:
        FONT_TRACE.append({"family": _fam, "status": f"UNREADABLE: {exc}"})
        continue
    _missing = [c for c in KOREAN_PROBE if "가" <= c <= "힣" and ord(c) not in _cmap]
    if _missing:
        FONT_TRACE.append({"family": _fam, "status": f"MISSING_GLYPHS: {''.join(_missing)}"})
        continue
    FONT_TRACE.append({"family": _fam, "status": "SELECTED", "path": _path})
    KOREAN_PLOT_FONT, KOREAN_PLOT_FONT_PATH = _fam, _path
    break

if KOREAN_PLOT_FONT is None:                            # HARD WARNING — figure 생성을 금지한다
    raise RuntimeError(
        "HARD WARNING — 한글 glyph를 모두 제공하는 font를 찾지 못했습니다. "
        "final figure generation 전에 로컬에서 해결해야 합니다. trace=" + json.dumps(FONT_TRACE, ensure_ascii=False))

matplotlib.rcParams.update({
    "font.family": KOREAN_PLOT_FONT,
    "axes.unicode_minus": False,       # ASCII hyphen-minus를 사용해 minus glyph fallback을 차단한다
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "savefig.bbox": "tight",
    "svg.fonttype": "none",
    "svg.hashsalt": "koen-nb07-v001",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.titlesize": 11,
    "axes.labelsize": 9.5,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,
})

print(f"KOREAN_PLOT_FONT={KOREAN_PLOT_FONT}")
print(f"KOREAN_PLOT_FONT_PATH={KOREAN_PLOT_FONT_PATH}")
print("font resolution trace:")
for _t in FONT_TRACE:
    print("   ", _t)
print()
ENV = {
    "python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
    "scipy": scipy.__version__, "duckdb": duckdb.__version__, "matplotlib": matplotlib.__version__,
    "platform": platform.platform(),
}
print("environment:", json.dumps(ENV, indent=None))
print("ROOT =", ROOT)
NB07_STARTED_KST = dt.datetime.now(KST).isoformat(timespec="seconds")
print("started_kst =", NB07_STARTED_KST)

KOREAN_PLOT_FONT=Noto Sans CJK KR
KOREAN_PLOT_FONT_PATH=/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc
font resolution trace:
    {'family': 'Noto Sans CJK KR', 'status': 'SELECTED', 'path': '/usr/share/fonts/opentype/noto/NotoSansCJK-Bold.ttc'}

environment: {"python": "3.12.3", "numpy": "2.5.2", "pandas": "2.3.3", "scipy": "1.18.0", "duckdb": "1.5.5", "matplotlib": "3.11.1", "platform": "Linux-6.18.33.2-microsoft-standard-WSL2-x86_64-with-glibc2.39"}
ROOT = /home/sieg/projects-wsl/KOEN_nb07_20260818
started_kst = 2026-08-18T15:29:30+09:00


한국어 시각화 계약이 확정되었다. 선택된 font는 `.ttc` collection의 sub-face까지 등록한 뒤
**실제 cmap에서 한글 glyph 존재를 확인**하고 고른 것이며, 이름만 보고 고른 것이 아니다.
`axes.unicode_minus=False` 로 음수 축 label의 minus glyph fallback을 차단했다.
font를 찾지 못하면 위 셀은 HARD WARNING과 함께 즉시 중단되므로, 깨진 figure가
manifest에 들어갈 경로는 존재하지 않는다.

## 02 — Canonical artifact 신원 확인 (fail-closed)

왜: 어떤 기술통계도 artifact 신원이 확인되기 전에는 canonical이 아니다.
기대 SHA-256은 G5 `ssot_g5/03_G5_ARTIFACT_IDENTITY_v001.json` 에 동결되어 있고,
이 셀은 파일을 **다시 해싱**하여 대조한다. 하나라도 어긋나면 즉시 중단한다(fail closed).

D-05 `CHUNK_O200K_BASE_v001` 은 검증된 NB06/G5 원본에서 **read-only 복사본**으로 배치되어야
하며 hardlink여서는 안 된다 — 이 셀은 link count와 inode도 함께 기록한다.

In [2]:
IDENTITY_FROZEN = json.loads((ROOT / "ssot_g5" / "03_G5_ARTIFACT_IDENTITY_v001.json").read_text(encoding="utf-8"))
REQUIRED = {
    "D-02": "REP_FEATURES_v002.parquet",
    "D-03": "MORPH_FEATURES_KIWI_v001.parquet",
    "D-04": "TOKEN_O200K_BASE_v001.parquet",
    "D-05": "CHUNK_O200K_BASE_v001.parquet",
    "D-01": "PAIR_REGISTRY_v002.parquet",     # cohort 층화 변수의 출처
}


def sha256_file(path: Path, chunk: int = 1 << 22) -> str:
    h = hashlib.sha256()
    with path.open("rb") as fh:
        while block := fh.read(chunk):
            h.update(block)
    return h.hexdigest()


ARTIFACT_IDENTITY: dict[str, dict] = {}
_fail: list[str] = []
with RuntimeTelemetry(run_id=f"{RUN_ID}_ARTIFACT_SHA", stage="SHA256", total=len(REQUIRED),
                      abort_on_red=True) as tel:
    for _i, (_key, _fname) in enumerate(REQUIRED.items(), start=1):
        tel.set_stage(f"SHA256:{_key}")
        _p = REG / _fname
        _stat = _p.lstat()
        _real = _p.resolve()
        _digest = sha256_file(_real)
        _expected = IDENTITY_FROZEN["artifacts"][_key]["expected_sha256"]
        _match = _digest == _expected
        ARTIFACT_IDENTITY[_key] = {
            "filename": _fname, "sha256": _digest, "expected_sha256": _expected, "match": _match,
            "placement": "symlink" if _p.is_symlink() else "regular file",
            "resolved_path": _real.as_posix(), "size_bytes": _real.stat().st_size,
            "inode": _real.stat().st_ino, "link_count": _real.stat().st_nlink,
        }
        if not _match:
            _fail.append(f"{_key} SHA MISMATCH")
        tel.update(1)
        print(f"{_key} {_fname:34s} {'MATCH' if _match else 'MISMATCH'}  {_digest[:12]}…  "
              f"{ARTIFACT_IDENTITY[_key]['placement']}  inode={ARTIFACT_IDENTITY[_key]['inode']}  "
              f"nlink={ARTIFACT_IDENTITY[_key]['link_count']}")
    ARTIFACT_SHA_TELEMETRY = tel.summary()

D05 = ARTIFACT_IDENTITY["D-05"]
if D05["placement"] != "regular file" or D05["link_count"] != 1:
    _fail.append("D-05 must be an independent read-only copy, not a hardlink or symlink")
if _fail:
    raise SystemExit("ARTIFACT_IDENTITY_FAIL_CLOSED: " + "; ".join(_fail))

ARTIFACT_IDENTITY_RESULT = f"{sum(v['match'] for v in ARTIFACT_IDENTITY.values())} / {len(REQUIRED)}"
print(f"\nARTIFACT_IDENTITY = {ARTIFACT_IDENTITY_RESULT}")
print(f"D-05 independent copy: link_count={D05['link_count']} inode={D05['inode']} — not hardlinked")

D-02 REP_FEATURES_v002.parquet          MATCH  dfae8e01cd3f…  symlink  inode=1698415  nlink=1


D-03 MORPH_FEATURES_KIWI_v001.parquet   MATCH  0fe5bd74e399…  symlink  inode=1699677  nlink=1


D-04 TOKEN_O200K_BASE_v001.parquet      MATCH  1c30e3276222…  symlink  inode=1699671  nlink=1


D-05 CHUNK_O200K_BASE_v001.parquet      MATCH  bfa98bd6cf7e…  regular file  inode=1703860  nlink=1


D-01 PAIR_REGISTRY_v002.parquet         MATCH  95f523d11b0e…  symlink  inode=1697629  nlink=1

ARTIFACT_IDENTITY = 5 / 5
D-05 independent copy: link_count=1 inode=1703860 — not hardlinked


5개 canonical artifact 전부가 G5에서 동결된 SHA-256과 **일치**한다 (`5 / 5`).
D-05는 link count 1의 독립 복사본이며 hardlink가 아니다 — G5 tree와 물리적으로 다른 파일을
해싱해서 같은 값을 얻었으므로, 이 노트북의 chunk 기술통계는 NB06/G5 원본과 bit 단위로 동일한
입력에서 나온다. 하나라도 어긋났다면 위 셀은 `SystemExit` 으로 중단되며, 그 이후의 어떤 수치도
생성되지 않는다.

## 03 — Analysis cohort 조립과 동결 확인

왜: NB07의 모든 수치는 `ANALYSIS_COHORT_v001` (G5에서 동결) 위에서 계산되어야 하며,
이 노트북은 그 cohort를 **다시 조립**해서 `N` 과 pair-set hash를 독립적으로 재확인한다.
cohort는 D-04 pair-id spine이고, D-01/D-02/D-03/D-05가 그 위에 join된다.
어떤 행도 삭제하지 않는다 — 결과에 따른 사후 행 삭제는 금지된다.

기대값: `N = 3,835,988`, pair-set hash `d9660d654ee449e4d0c23a0070225274`.
어긋나면 fail closed.

30초를 넘길 수 있는 작업이므로 `RuntimeTelemetry` (ENG-OBS-001 · R1) 10초 주기 heartbeat 아래에서
실행한다: stage · elapsed · processed/total · throughput · RSS · MemAvailable · status.
percent/ETA는 신뢰할 수 있는 분모가 없으면 지어내지 않는다.

In [3]:
P = f"read_parquet('{(REG / 'PAIR_REGISTRY_v002.parquet').as_posix()}')"
R = f"read_parquet('{(REG / 'REP_FEATURES_v002.parquet').as_posix()}')"
M = f"read_parquet('{(REG / 'MORPH_FEATURES_KIWI_v001.parquet').as_posix()}')"
T = f"read_parquet('{(REG / 'TOKEN_O200K_BASE_v001.parquet').as_posix()}')"
K = f"read_parquet('{(REG / 'CHUNK_O200K_BASE_v001.parquet').as_posix()}')"

COHORT_SQL = f"""
SELECT
  t.pair_id,
  split_part(p.source_id, '-', 1)                          AS source,
  p.domain,
  p.translation_direction,
  p.length_stratum,
  split_part(p.source_id, '-', 1) || '-' || p.domain       AS source_domain_cell,
  -- B. token counts / C. TP · logTP · ΔT
  t.ko_token_count, t.en_token_count,
  t.token_premium, t.log_token_premium, t.token_difference,
  -- D. exact decomposition
  t.code_point_ratio, t.byte_density_ratio, t.compression_penalty,
  t.log_code_point_ratio, t.log_byte_density_ratio, t.log_compression_penalty,
  t.ko_tokens_per_byte, t.en_tokens_per_byte,
  t.ko_tokens_per_codepoint, t.en_tokens_per_codepoint,
  -- E. representation structure
  r.ko_codepoint_count, r.en_codepoint_count, r.ko_utf8_bytes, r.en_utf8_bytes,
  r.ko_bytes_per_codepoint, r.en_bytes_per_codepoint,
  r.ko_whitespace_density, r.en_whitespace_density,
  r.ko_hangul_share, r.ko_latin_share, r.ko_digit_share,
  r.ko_punctuation_share, r.ko_symbol_other_share,
  r.en_hangul_share, r.en_latin_share, r.en_digit_share,
  r.en_punctuation_share, r.en_symbol_other_share,
  CAST(r.ko_script_type_count   AS SMALLINT) AS ko_script_type_count,
  CAST(r.ko_script_switch_count AS INTEGER)  AS ko_script_switch_count,
  CAST(r.en_script_type_count   AS SMALLINT) AS en_script_type_count,
  CAST(r.en_script_switch_count AS INTEGER)  AS en_script_switch_count,
  r.ko_eojeol_count, r.en_word_count,
  0.5 * (ln(r.ko_codepoint_count) + ln(r.en_codepoint_count)) AS pair_log_size,
  -- F. morphology (D-03)
  m.eojeol_count AS morph_eojeol_count, m.morpheme_count,
  m.morpheme_density, m.particle_ratio, m.ending_ratio, m.deriv_affix_ratio,
  m.function_morpheme_ratio,
  -- G. D-05 regex-chunk mechanism descriptors
  k.ko_chunk_count, k.en_chunk_count,
  ln(k.ko_chunk_count) AS ko_chunk_count_log,
  ln(k.en_chunk_count) AS en_chunk_count_log,
  k.ko_mean_chunk_bytes, k.ko_p50_chunk_bytes, k.ko_p90_chunk_bytes,
  CAST(k.ko_max_chunk_bytes AS INTEGER) AS ko_max_chunk_bytes,
  k.en_mean_chunk_bytes, k.en_p50_chunk_bytes, k.en_p90_chunk_bytes,
  CAST(k.en_max_chunk_bytes AS INTEGER) AS en_max_chunk_bytes,
  k.ko_tokens_per_chunk, k.en_tokens_per_chunk,
  CAST(k.ko_max_tokens_per_chunk AS INTEGER) AS ko_max_tokens_per_chunk,
  CAST(k.en_max_tokens_per_chunk AS INTEGER) AS en_max_tokens_per_chunk,
  k.ko_chunk_type_share_letter, k.ko_chunk_type_share_number,
  k.ko_chunk_type_share_punctuation, k.ko_chunk_type_share_whitespace,
  k.en_chunk_type_share_letter, k.en_chunk_type_share_number,
  k.en_chunk_type_share_punctuation, k.en_chunk_type_share_whitespace,
  k.pair_chunk_ratio
FROM {T} t
JOIN {P} p ON p.pair_id = t.pair_id
JOIN {R} r ON r.pair_id = t.pair_id
JOIN {M} m ON m.pair_id = t.pair_id
JOIN {K} k ON k.pair_id = t.pair_id
"""

MATRIX = (RUNTIME / "nb07_matrix.parquet")
con = duckdb.connect()
con.execute("PRAGMA memory_limit='3GB'")
con.execute("PRAGMA threads=4")
con.execute("SET preserve_insertion_order=false")
con.execute(f"PRAGMA temp_directory='{(RUNTIME / 'spill').as_posix()}'")

COHORT: dict = {"cohort_id": "ANALYSIS_COHORT_v001", "expected_N": EXPECTED_N,
                "expected_pair_set_hash": EXPECTED_PAIR_SET}

with RuntimeTelemetry(run_id=f"{RUN_ID}_COHORT", stage="SOURCE_COUNTS", total=EXPECTED_N,
                      abort_on_red=True) as tel:
    COHORT["source_row_counts"] = {
        key: dict(zip(("rows", "distinct_pair_id"),
                      con.execute(f"SELECT count(*), count(DISTINCT pair_id) FROM {rel}").fetchone(),
                      strict=True))
        for key, rel in (("D-01", P), ("D-02", R), ("D-03", M), ("D-04", T), ("D-05", K))
    }
    tel.set_stage("MATERIALIZE")
    con.execute(f"COPY ({COHORT_SQL}) TO '{MATRIX.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)")

    tel.set_stage("COHORT_IDENTITY")
    A = f"read_parquet('{MATRIX.as_posix()}')"
    _n, _nd = con.execute(f"SELECT count(*), count(DISTINCT pair_id) FROM {A}").fetchone()
    _hash = con.execute(f"SELECT md5(string_agg(pair_id, '' ORDER BY pair_id)) FROM {A}").fetchone()[0]
    tel.update(_n)
    COHORT.update({"N": int(_n), "distinct_pair_id": int(_nd), "pair_set_hash": _hash,
                   "N_matches_expected": _n == EXPECTED_N,
                   "distinct_pair_id_equals_N": _nd == _n,
                   "pair_set_hash_matches_expected": _hash == EXPECTED_PAIR_SET,
                   "join_preserves_d04_spine": _n == COHORT["source_row_counts"]["D-04"]["rows"]})

    tel.set_stage("COMPLETENESS")
    _cols = [c[0] for c in con.execute(f"DESCRIBE SELECT * FROM {A}").fetchall() if c[0] != "pair_id"]
    _numeric = [c[0] for c in con.execute(f"DESCRIBE SELECT * FROM {A}").fetchall()
                if c[1] not in ("VARCHAR",) and c[0] != "pair_id"]
    _nulls = con.execute(
        "SELECT " + " + ".join(f"sum(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END)" for c in _cols)
        + f" FROM {A}").fetchone()[0]
    _nonfin = con.execute(
        "SELECT " + " + ".join(
            f"sum(CASE WHEN {c} IS NOT NULL AND NOT isfinite(CAST({c} AS DOUBLE)) THEN 1 ELSE 0 END)"
            for c in _numeric) + f" FROM {A}").fetchone()[0]
    COHORT.update({"columns": len(_cols) + 1, "null_values": int(_nulls),
                   "nonfinite_values": int(_nonfin),
                   "total_null_or_nonfinite": int(_nulls) + int(_nonfin)})
    COHORT_TELEMETRY = tel.summary()

_bad = [k for k in ("N_matches_expected", "distinct_pair_id_equals_N",
                    "pair_set_hash_matches_expected", "join_preserves_d04_spine")
        if not COHORT[k]]
if _bad or COHORT["total_null_or_nonfinite"] != 0:
    raise SystemExit(f"COHORT_FAIL_CLOSED: {_bad} nulls/nonfinite={COHORT['total_null_or_nonfinite']}")

print(f"COHORT_N        = {COHORT['N']:,}   (expected {EXPECTED_N:,})")
print(f"distinct pair_id= {COHORT['distinct_pair_id']:,}")
print(f"PAIR_SET_HASH   = {COHORT['pair_set_hash']}   (expected {EXPECTED_PAIR_SET})")
print(f"columns         = {COHORT['columns']}   null={COHORT['null_values']}  nonfinite={COHORT['nonfinite_values']}")
print(f"matrix size     = {MATRIX.stat().st_size / 1e6:.1f} MB  (runtime-only, .gitignore'd)")
print(f"telemetry       = {COHORT_TELEMETRY['sample_count']} samples @ "
      f"{COHORT_TELEMETRY['interval_sec']}s · worst status {COHORT_TELEMETRY['worst_memory_status']} · "
      f"peak RSS {COHORT_TELEMETRY['peak_rss_gib']} GiB · min MemAvailable "
      f"{COHORT_TELEMETRY['min_mem_available_gib']} GiB")
print("\nsource row counts:")
for _k, _v in COHORT["source_row_counts"].items():
    print(f"  {_k}: rows={_v['rows']:,} distinct_pair_id={_v['distinct_pair_id']:,}")
print("\nANALYSIS_COHORT_FROZEN — 재조립 결과가 G5 동결값과 일치")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

COHORT_N        = 3,835,988   (expected 3,835,988)
distinct pair_id= 3,835,988
PAIR_SET_HASH   = d9660d654ee449e4d0c23a0070225274   (expected d9660d654ee449e4d0c23a0070225274)
columns         = 78   null=0  nonfinite=0
matrix size     = 429.8 MB  (runtime-only, .gitignore'd)
telemetry       = 6 samples @ 10.0s · worst status RED · peak RSS 3.312 GiB · min MemAvailable 6.586 GiB

source row counts:
  D-01: rows=5,652,925 distinct_pair_id=5,652,925
  D-02: rows=3,835,988 distinct_pair_id=3,835,988
  D-03: rows=3,835,988 distinct_pair_id=3,835,988
  D-04: rows=3,835,988 distinct_pair_id=3,835,988
  D-05: rows=3,835,988 distinct_pair_id=3,835,988

ANALYSIS_COHORT_FROZEN — 재조립 결과가 G5 동결값과 일치


cohort가 독립적으로 재조립되어 G5 동결값과 정확히 일치했다: `N = 3,835,988`,
distinct `pair_id` 도 동일, pair-set hash `d9660d654ee449e4d0c23a0070225274`.
D-01은 5,652,925행을 갖지만 D-04 spine으로의 join이 `N` 을 정확히 보존했으므로
조용히 얻거나 잃은 행이 없다. null·비유한값은 0이다.

배제(exclusion)는 **없다**: `translation_direction = UNKNOWN`, `eojeol_count = 1`,
짧은 문장, TP 극단, 형태소 극단 모두 primary cohort에 **포함된 채로** 남는다.
이상치는 §17에서 삭제 없이 등록·시각화된다.

## 04 — SSOT / RQ 대응표와 figure 계약

왜: 이후 모든 절은 자기 절의 RQ를 SSOT 원문 표현으로 선언하고 시작한다.
이 셀은 그 대응을 한 곳에 모으고, `KOEN-TP-RS-001` §16.2 · §35 figure 계약과 §34 RQ traceability를
직접 재독한 결과를 기록한다. RQ 번호를 이전 노트북이나 프롬프트에서 추론하지 않는다.

| RQ | SSOT 원문 (§6) | canonical input | NB07 역할 |
|---|---|---|---|
| RQ1 | 의미 대응 KO‑EN 문장쌍에서 o200k_base 기준 한국어 Tokenization Premium은 1보다 큰가? | D‑04 | annotate only (NB08 종결) |
| RQ2 | 전체 premium 가운데 CodePointRatio · ByteDensityRatio · CompressionPenalty 세 요소가 각각 어느 정도의 분포를 보이는가? | D‑02 + D‑04 | **canonical NB07 결과** |
| RQ3 | 공백 밀도, grapheme 구조, Hangul/Latin/digit/punctuation 비율, script mixing, 문장 길이, 숫자·특수기호가 log(TP)와 log(CompressionPenalty)에 어떤 조건부 연관을 갖는가? | D‑02 + D‑04 | 표면형 기술 구조만 |
| RQ4 | 형태소 밀도, 조사 비율, 어미 비율, 접사 비율이 길이·byte·공백·문자군·도메인·출처를 통제한 후에도 log(TP) 또는 log(CompressionPenalty)의 추가 설명력을 제공하는가? | D‑03 + D‑04 | 형태소 분포만 |
| RQ5 | 고정된 o200k_base 구현에서 관측되는 regex chunk count, chunk 길이, token‑per‑chunk, token‑per‑byte 등의 mechanism feature가 최종 premium과 어떻게 연결되는가? | D‑05 + D‑04 | regex-chunk 구조만 |
| RQ6 | 도메인, 문장 유형, 번역 방향, 출처 및 길이 strata에 따라 Tokenization Premium과 설명요인의 크기가 달라지는가? | D‑01..D‑05 | 기술적 이질성만 |
| RQ7 | Serving‑level 차이 (Track B) | D‑06 + D‑07 | 범위 밖 — `DEFERRED_NOT_EXECUTED` |

**§16.2 필수 시각화** (원문, §35 최종 표·그림 설계가 같은 목록을 재기술한다): F01 KO vs EN paired token scatter, identity line · F02 TP histogram + ECDF ·
F03 domain별 TP violin/box · F04 exact decomposition component distribution ·
F05 ByteDensityRatio vs CompressionPenalty · F06 morphology density vs logTP partial relationship ·
F07 extreme TP case audit panel · F08 source/domain forest plot ·
F09 Track B request/output/latency comparison.

NB07이 dependency-ready로 생성하는 것은 **F01 · F02 · F03 · F04 · F05 · F07** 이다.
F06(형태소 partial effect) · F08(설명 forest) · F09(Track B)는 각각 NB09 · NB09 · NB12에
의존하므로 `DEFERRED_BY_DEPENDENCY` 로 표기하고 여기서 만들지 않는다.
`NB08-RQ1-Vxx`, `NB06_D05_Vxx` 는 reference일 뿐이며 절대 Fxx로 개명하지 않는다.

## 05 — 시각화 · 집계 helper (노트북 내 정의, 은닉 없음)

왜: notebook-first 규칙에 따라 이 노트북이 쓰는 helper는 전부 여기서 투명하게 정의된다.
외부 `src`/`scripts` 에서 가져오는 것은 `RuntimeTelemetry` (heartbeat 계약 구현) 하나뿐이며,
그 호출은 각 heavy 셀에 그대로 보인다.

`density_panel` 은 3,835,988점을 개별 marker로 그리지 않고 2D histogram + log color scale로
그린다. 이는 over-plotting으로 밀도 구조가 사라지는 것을 막기 위한 것이며, 표본추출이 아니라
**전수 집계**다.

In [4]:
FIGURES: list[dict] = []
FIG_SEQ: dict[str, int] = {}


def save_fig(fig, fig_id: str, title_ko: str, *, rq: str, contract: str,
             note: str = "", tag: str = "CANONICAL") -> dict:
    """figure를 PNG+SVG로 저장하고 SHA-256과 함께 manifest 항목을 등록한다."""
    if fig_id in FIG_SEQ:
        raise ValueError(f"figure id 중복: {fig_id}")
    FIG_SEQ[fig_id] = len(FIG_SEQ)
    png, svg = FIGDIR / f"{fig_id}.png", FIGDIR / f"{fig_id}.svg"
    with warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter("always")
        fig.savefig(png, format="png", metadata={"Software": "KOEN NB07"})
        fig.savefig(svg, format="svg", metadata={"Date": "2026-08-18", "Creator": "KOEN NB07"})
    missing = [str(w.message) for w in captured if "Glyph" in str(w.message) and "missing" in str(w.message)]
    if missing:
        raise AssertionError(f"{fig_id}: 한글 glyph 누락 — {missing}")
    svg_text = svg.read_text(encoding="utf-8")
    has_hangul = any("가" <= c <= "힣" for c in svg_text)
    if not has_hangul:
        raise AssertionError(f"{fig_id}: SVG에 한글 text가 없다 — 한국어 시각화 계약 위반")
    entry = {
        "figure_id": fig_id, "title_ko": title_ko, "rq": rq, "contract": contract,
        "tag": tag, "note": note, "status": "GENERATED",
        "png": png.relative_to(ROOT).as_posix(), "svg": svg.relative_to(ROOT).as_posix(),
        "png_sha256": sha256_file(png), "svg_sha256": sha256_file(svg),
        "png_size_bytes": png.stat().st_size, "svg_size_bytes": svg.stat().st_size,
        "korean_text_in_svg": has_hangul, "font": KOREAN_PLOT_FONT,
    }
    FIGURES.append(entry)
    plt.close(fig)
    print(f"  saved {fig_id}  png {entry['png_size_bytes'] / 1024:.0f} KiB  "
          f"sha {entry['png_sha256'][:12]}…  한글={has_hangul}")
    return entry


def density_panel(ax, x, y, *, bins=200, xlabel="", ylabel="", title="",
                  xlim=None, ylim=None, cmap="magma_r"):
    """전수 2D histogram 밀도 패널 (표본추출 없음, log color scale)."""
    rng = [list(xlim) if xlim else [float(np.min(x)), float(np.max(x))],
           list(ylim) if ylim else [float(np.min(y)), float(np.max(y))]]
    H, xe, ye = np.histogram2d(x, y, bins=bins, range=rng)
    Hm = np.ma.masked_where(H.T == 0, H.T)
    pcm = ax.pcolormesh(xe, ye, Hm, norm=LogNorm(vmin=1, vmax=max(H.max(), 2)),
                        cmap=cmap, shading="auto", rasterized=True)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    cb = ax.figure.colorbar(pcm, ax=ax, pad=0.02)
    cb.set_label("문장쌍 수 (로그 눈금)", fontsize=8)
    return pcm


def binned_median(ax, x, y, *, bins=40, color="#1b6ca8", label="구간별 중앙값"):
    """구간별 robust 중앙값 곡선을 밀도 패널 위에 덧그린다."""
    edges = np.quantile(x, np.linspace(0, 1, bins + 1))
    edges = np.unique(edges)
    if edges.size < 3:
        return
    med, _, _ = stats.binned_statistic(x, y, statistic="median", bins=edges)
    centres = 0.5 * (edges[:-1] + edges[1:])
    ok = np.isfinite(med)
    ax.plot(centres[ok], med[ok], color=color, lw=1.8, label=label)
    ax.legend(loc="best", framealpha=0.85)


def fetch(cols: str, where: str = "") -> pd.DataFrame:
    """cohort matrix에서 필요한 열만 가져온다 (메모리 상한 유지)."""
    return con.execute(f"SELECT {cols} FROM {A}{(' WHERE ' + where) if where else ''}").fetchdf()


def col(name: str, where: str = "") -> np.ndarray:
    return fetch(name, where)[name].to_numpy(dtype=np.float64, copy=False)


def qsummary(name: str, expr: str, qs=(0.0, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.0)) -> dict:
    """단일 표현식의 전수 분위·평균·표준편차 요약."""
    row = con.execute(
        f"SELECT count(*), avg({expr}), stddev_samp({expr}), quantile_cont({expr}, {list(qs)}) FROM {A}"
    ).fetchone()
    return {"name": name, "n": int(row[0]), "mean": float(row[1]), "sd": float(row[2]),
            "quantiles": {str(q): float(v) for q, v in zip(qs, row[3], strict=True)}}


def show(df: pd.DataFrame, floatfmt: str = "{:,.6g}") -> None:
    with pd.option_context("display.width", 200, "display.max_columns", 60,
                           "display.float_format", floatfmt.format):
        print(df.to_string(index=False))


SUMMARY: dict = {
    "artifact_id": "NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001",
    "run_id": RUN_ID,
    "execution_base_type": EXECUTION_BASE_TYPE,
    "execution_base_sha": EXECUTION_BASE_SHA,
    "g5_research_head": G5_RESEARCH_HEAD,
    "started_kst": NB07_STARTED_KST,
    "environment": ENV,
    "korean_plot_font": KOREAN_PLOT_FONT,
    "korean_plot_font_path": KOREAN_PLOT_FONT_PATH,
    "korean_font_resolution_trace": FONT_TRACE,
    "artifact_identity": ARTIFACT_IDENTITY,
    "artifact_identity_result": ARTIFACT_IDENTITY_RESULT,
    "cohort": COHORT,
    "legacy_casebook_numerical_source": False,
    "telemetry": {"artifact_sha": ARTIFACT_SHA_TELEMETRY, "cohort": COHORT_TELEMETRY},
}
ANOMALIES: list[dict] = []


def anomaly(ref: str, observed: str, n: int, rule: str, status: str, ssot: str,
            interpretation: str, not_established: str, downstream: str,
            share: float | None = None, cohort_disposition: str = "INCLUDED") -> dict:
    entry = {"ref": ref, "observed": observed, "n": int(n),
             "share": float(n / COHORT["N"]) if share is None else float(share),
             "rule": rule, "status": status, "primary_cohort": cohort_disposition,
             "ssot_relevance": ssot, "interpretation": interpretation,
             "not_established": not_established, "downstream": downstream}
    ANOMALIES.append(entry)
    print(f"[{ref}] {status:18s} n={entry['n']:>9,}  share={entry['share'] * 100:8.4f}%  {observed}")
    return entry


print("helper 정의 완료 — figure/anomaly/summary registry 초기화됨")

helper 정의 완료 — figure/anomaly/summary registry 초기화됨


## 06 — A. Cohort 구성: source · domain · translation_direction · length

### SSOT / RQ 대응

**Research Question**: RQ6 — 도메인, 문장 유형, 번역 방향, 출처 및 길이 strata에 따라
Tokenization Premium과 설명요인의 크기가 달라지는가?

**SSOT section**: `KOEN-TP-RS-001` §6.6, §9.2 (층화 변수), §20.2 (identifiability)

**Canonical input**: D-01 `PAIR_REGISTRY_v002` (층화 변수), D-04 `TOKEN_O200K_BASE_v001` (spine)

**Physical variables**: `source`, `domain`, `translation_direction`, `length_stratum`,
`source_domain_cell`, `ko_codepoint_count`, `en_codepoint_count`

**Research purpose**: 이후 모든 기술통계·이질성 해석이 어떤 관측 support 위에서 이루어지는지
먼저 고정한다. 어떤 비교가 자료 안에서 실제로 관측되는지 밝히는 것이 목적이다.

**Allowed claim**: 실현된 cohort의 구성비와 층 크기에 대한 기술적 진술.

**Prohibited claim**: 어떤 층이 premium을 "일으킨다"는 진술, 층 간 차이의 통계적 유의성,
모집단 대표성 주장.

In [5]:
comp: dict[str, pd.DataFrame] = {}
for _v in ("source", "domain", "translation_direction", "length_stratum", "source_domain_cell"):
    _d = con.execute(f"SELECT {_v} AS level, count(*) AS n FROM {A} GROUP BY 1 ORDER BY 2 DESC").fetchdf()
    _d["share"] = _d["n"] / COHORT["N"]
    comp[_v] = _d
    print(f"— {_v} —")
    show(_d.assign(share=lambda d: (d["share"] * 100).round(4)))
    print()

_len = con.execute(f"""
  SELECT length_stratum,
         count(*) AS n,
         quantile_cont(ko_codepoint_count, 0.5) AS ko_cp_p50,
         quantile_cont(en_codepoint_count, 0.5) AS en_cp_p50,
         quantile_cont(ko_token_count, 0.5)     AS ko_tok_p50,
         quantile_cont(en_token_count, 0.5)     AS en_tok_p50
  FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
print("— length_stratum별 규모 (중앙값) —")
show(_len)
SUMMARY["cohort_composition"] = {k: v.to_dict(orient="records") for k, v in comp.items()}
SUMMARY["length_stratum_scale"] = _len.to_dict(orient="records")

— source —
level       n   share
  025 2485963 64.8063
  026 1350025 35.1937

— domain —
     level       n   share
     other 2155630 56.1949
   general  804291  20.967
  dialogue  516162 13.4558
technology  359905  9.3823

— translation_direction —
   level       n   share
KO_TO_EN 2512152  65.489
EN_TO_KO 1273289 33.1932
 UNKNOWN   50547  1.3177

— length_stratum —
level      n   share
   Q2 858560 22.3817
   Q1 840667 21.9153
   Q3 802735 20.9264
   Q4 725825 18.9215
   Q5 608201 15.8551

— source_domain_cell —
         level       n   share
     025-other 1165510 30.3836
     026-other  990120 25.8113
   025-general  804291  20.967
  025-dialogue  516162 13.4558
026-technology  359905  9.3823



— length_stratum별 규모 (중앙값) —
length_stratum      n  ko_cp_p50  en_cp_p50  ko_tok_p50  en_tok_p50
            Q1 840667         16         33          10           8
            Q2 858560         27         54          15          12
            Q3 802735         39         83          23          17
            Q4 725825         66        145          39          28
            Q5 608201         94        219          55          42


In [6]:
fig, axes = plt.subplots(2, 3, figsize=(15.5, 8.2))
_pal = ["#2f6690", "#3a7ca5", "#81c3d7", "#d9dcd6", "#16425b"]

for _ax, _v, _t in zip(
        axes.flat[:4],
        ("source", "domain", "translation_direction", "length_stratum"),
        ("출처(source) 구성", "도메인(domain) 구성", "번역 방향(translation_direction) 구성",
         "길이 층(length_stratum) 구성"), strict=True):
    _d = comp[_v].sort_values("n")
    _ax.barh(_d["level"].astype(str), _d["n"], color=_pal[:len(_d)][::-1])
    for _i, (_n, _s) in enumerate(zip(_d["n"], _d["share"], strict=True)):
        _ax.text(_n, _i, f" {_n:,} ({_s * 100:.2f}%)", va="center", fontsize=8)
    _ax.set_xlabel("문장쌍 수")
    _ax.set_title(_t)
    _ax.set_xlim(0, _d["n"].max() * 1.42)

_cp = fetch("ko_codepoint_count, en_codepoint_count, ko_token_count, en_token_count")
_ax = axes.flat[4]
_bins = np.logspace(0, np.log10(max(_cp["ko_codepoint_count"].max(), _cp["en_codepoint_count"].max())), 90)
_ax.hist(_cp["ko_codepoint_count"], bins=_bins, histtype="step", lw=1.6, color="#c1272d", label="한국어 code point")
_ax.hist(_cp["en_codepoint_count"], bins=_bins, histtype="step", lw=1.6, color="#0f4c81", label="영어 code point")
_ax.set_xscale("log"); _ax.set_yscale("log")
_ax.set_xlabel("code point 수 (로그 눈금)"); _ax.set_ylabel("문장쌍 수 (로그 눈금)")
_ax.set_title("문장 길이 분포 — code point"); _ax.legend()

_ax = axes.flat[5]
_bins = np.logspace(0, np.log10(max(_cp["ko_token_count"].max(), _cp["en_token_count"].max())), 90)
_ax.hist(_cp["ko_token_count"], bins=_bins, histtype="step", lw=1.6, color="#c1272d", label="한국어 token")
_ax.hist(_cp["en_token_count"], bins=_bins, histtype="step", lw=1.6, color="#0f4c81", label="영어 token")
_ax.set_xscale("log"); _ax.set_yscale("log")
_ax.set_xlabel("o200k_base token 수 (로그 눈금)"); _ax.set_ylabel("문장쌍 수 (로그 눈금)")
_ax.set_title("문장 길이 분포 — token"); _ax.legend()

fig.suptitle(f"NB07-S01 · 분석 cohort 구성 (N = {COHORT['N']:,}, ANALYSIS_COHORT_v001)", fontsize=13)
fig.tight_layout()
save_fig(fig, "NB07-S01_cohort_composition_v001", "분석 cohort 구성",
         rq="RQ6", contract="SUPPORTING_DESCRIPTIVE",
         note="source/domain/direction/length 구성과 길이 분포")
del _cp

  saved NB07-S01_cohort_composition_v001  png 209 KiB  sha e9878958cbac…  한글=True


### 수치 해석

cohort는 3,835,988 문장쌍이다. 출처는 025가 2,485,963쌍(64.81%), 026이 1,350,025쌍(35.19%)이다.
도메인은 `other` 2,155,630(56.20%), `general` 804,291(20.97%), `dialogue` 516,162(13.46%),
`technology` 359,905(9.38%)로, 절반 이상이 `other` 에 몰려 있다.
번역 방향은 `KO_TO_EN` 2,512,152(65.49%), `EN_TO_KO` 1,273,289(33.19%),
`UNKNOWN` 50,547(1.32%)이다. 길이 층은 Q1~Q5가 608,201~858,560 범위로 비교적 고르다.

### 분포 해석

길이 분포는 code point·token 모두 오른쪽 꼬리가 긴 로그 정규형에 가깝고, 두 언어 곡선이
서로 다른 위치에 있다. 영어 code point 분포가 한국어보다 뚜렷하게 오른쪽에 있다(영어 문장이
문자 수로 더 길다). 그런데 token 분포에서는 순서가 뒤집혀 한국어 곡선이 오른쪽으로 이동한다.
이 한 장의 비교가 이 연구의 출발 관찰이며, §11에서 정량화된다.

### 연구적 의미

RQ6의 층화 해석은 이 support 위에서만 가능하다. `other` 도메인이 과반이고 `UNKNOWN` 방향이
1.32% 남아 있다는 사실은 이후 층별 비교의 해상도를 직접 제한한다.

### 해석 한계

이 구성비는 실현된 cohort의 속성이지 한국어–영어 번역 말뭉치 일반의 속성이 아니다.
층 크기 차이 자체는 어떤 효과의 증거도 아니다. `sentence_type` 은 실현 수준이 하나뿐이어서
(G5 §4.1) 층화 변수로 쓸 수 없고, 여기서도 제시하지 않는다.

## 07 — Identifiability support: source × domain, cell × direction

### SSOT / RQ 대응

**Research Question**: RQ6 — 도메인, 문장 유형, 번역 방향, 출처 및 길이 strata에 따라
Tokenization Premium과 설명요인의 크기가 달라지는가?

**SSOT section**: `KOEN-TP-RS-001` §20.1 마지막 절, §20.2, §20.1 마지막 항목; G5 protocol T-03 · T-04;
G5 adjudication §5.1 · §5.2 → NOTE `ID-03`, `ID-04`

**Canonical input**: D-01 `PAIR_REGISTRY_v002`

**Physical variables**: `source`, `domain`, `source_domain_cell`, `translation_direction`

**Research purpose**: G5가 발견한 **불완전 중첩**을 눈에 보이게 만든다. 어떤 비교가 자료
안에서 식별되고 어떤 비교가 비어 있는지는 표로만 남기면 이후 단계에서 잊힌다.

**Allowed claim**: 관측 support 구조 / 식별 가능한 비교 범위에 대한 진술.

**Prohibited claim**: 순수 source 효과, 순수 domain 효과. `source_domain_cell` 계수를
둘 중 하나로 읽는 것. 비어 있는 셀에 대한 외삽.

**참조**: G5 `ID-03`, `ID-04`

In [7]:
_sd = con.execute(f"SELECT source, domain, count(*) n FROM {A} GROUP BY 1,2").fetchdf()
SD = _sd.pivot(index="source", columns="domain", values="n").reindex(
    columns=["dialogue", "general", "other", "technology"]).fillna(0).astype(np.int64)
print("— ID-03 · source × domain support —")
show(SD.reset_index())
print(f"\n실현 셀 {int((SD > 0).to_numpy().sum())} / {SD.size}   "
      f"빈 셀 {int((SD == 0).to_numpy().sum())}")
print("두 출처 모두에서 관측되는 도메인: " + ", ".join(SD.columns[(SD > 0).all(axis=0)]))

_cd = con.execute(
    f"SELECT source_domain_cell, translation_direction, count(*) n FROM {A} GROUP BY 1,2").fetchdf()
CD = _cd.pivot(index="source_domain_cell", columns="translation_direction", values="n").reindex(
    columns=["KO_TO_EN", "EN_TO_KO", "UNKNOWN"]).fillna(0).astype(np.int64)
print("\n— ID-04 · source_domain_cell × translation_direction support —")
show(CD.reset_index())
_empty = [(i, c) for i in CD.index for c in CD.columns if CD.loc[i, c] == 0]
_single = [(i, c, int(CD.loc[i, c])) for i in CD.index for c in CD.columns
           if 0 < CD.loc[i, c] <= 100]
print(f"\n빈 셀 {len(_empty)} / {CD.size}: {_empty}")
print(f"near-singleton 셀 (n ≤ 100): {_single}")
print(f"UNKNOWN 총계 {int(CD['UNKNOWN'].sum()):,} — 제거하지 않고 보존한다")
print(f"026 계열의 EN_TO_KO 총계 = {int(CD.loc[[i for i in CD.index if i.startswith('026')], 'EN_TO_KO'].sum())}")

SUMMARY["identifiability_support"] = {
    "ID-03_source_x_domain": json.loads(SD.to_json(orient="index")),
    "ID-03_realized_cells": int((SD > 0).to_numpy().sum()),
    "ID-03_empty_cells": int((SD == 0).to_numpy().sum()),
    "ID-03_shared_domains": list(SD.columns[(SD > 0).all(axis=0)]),
    "ID-04_cell_x_direction": json.loads(CD.to_json(orient="index")),
    "ID-04_empty_cells": [list(e) for e in _empty],
    "ID-04_near_singleton_cells": [list(e) for e in _single],
    "ID-04_unknown_total": int(CD["UNKNOWN"].sum()),
    "ID-04_026_en_to_ko": 0,
}

— ID-03 · source × domain support —
source  dialogue  general   other  technology
   025    516162   804291 1165510           0
   026         0        0  990120      359905

실현 셀 5 / 8   빈 셀 3
두 출처 모두에서 관측되는 도메인: other

— ID-04 · source_domain_cell × translation_direction support —
source_domain_cell  KO_TO_EN  EN_TO_KO  UNKNOWN
      025-dialogue    247947    254955    13260
       025-general    354654    449606       31
         025-other    559527    568728    37255
         026-other    990119         0        1
    026-technology    359905         0        0

빈 셀 3 / 15: [('026-other', 'EN_TO_KO'), ('026-technology', 'EN_TO_KO'), ('026-technology', 'UNKNOWN')]
near-singleton 셀 (n ≤ 100): [('025-general', 'UNKNOWN', 31), ('026-other', 'UNKNOWN', 1)]
UNKNOWN 총계 50,547 — 제거하지 않고 보존한다
026 계열의 EN_TO_KO 총계 = 0


In [8]:
def support_heatmap(ax, mat: pd.DataFrame, title: str, xlabel: str, ylabel: str):
    arr = mat.to_numpy(dtype=float)
    masked = np.ma.masked_where(arr == 0, arr)
    cmap = plt.get_cmap("YlGnBu").copy()
    cmap.set_bad("#f2c6c6")                       # 빈 셀은 색으로 즉시 구분된다
    pcm = ax.imshow(masked, cmap=cmap, norm=LogNorm(vmin=max(arr[arr > 0].min(), 1), vmax=arr.max()),
                    aspect="auto")
    ax.set_xticks(range(mat.shape[1]), mat.columns, rotation=20, ha="right")
    ax.set_yticks(range(mat.shape[0]), mat.index)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = int(arr[i, j])
            ax.text(j, i, "비어 있음\n(0)" if v == 0 else f"{v:,}", ha="center", va="center",
                    fontsize=8.5, color="#7a1f1f" if v == 0 else ("white" if v > arr.max() / 6 else "#14213d"),
                    fontweight="bold" if v == 0 or 0 < v <= 100 else "normal")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(False)
    cb = ax.figure.colorbar(pcm, ax=ax, pad=0.02)
    cb.set_label("문장쌍 수 (로그 눈금)", fontsize=8)


fig, ax = plt.subplots(figsize=(9.0, 3.6))
support_heatmap(ax, SD, "NB07-REF-ID-03 · 출처 × 도메인 관측 support\n"
                        "(025-other 대 026-other = 동일 도메인 내 출처 층 대비)",
                "도메인 (domain)", "출처 (source)")
fig.tight_layout()
save_fig(fig, "NB07-REF-ID-03_source_domain_support_v001", "출처 × 도메인 관측 support",
         rq="RQ6", contract="G5_REVIEW_REGISTER",
         note="G5 ID-03 — source와 domain은 분리 식별되지 않는다", tag="G5_REFERENCE")

fig, ax = plt.subplots(figsize=(9.4, 4.2))
support_heatmap(ax, CD, "NB07-REF-ID-04 · source_domain_cell × 번역 방향 관측 support\n"
                        "(빈 셀·near-singleton·UNKNOWN 모두 표시, 제거하지 않음)",
                "번역 방향 (translation_direction)", "출처-도메인 셀 (source_domain_cell)")
fig.tight_layout()
save_fig(fig, "NB07-REF-ID-04_cell_direction_support_v001", "출처-도메인 셀 × 번역 방향 support",
         rq="RQ6", contract="G5_REVIEW_REGISTER",
         note="G5 ID-04 — 방향 대비는 025 내부 변동에서만 식별된다", tag="G5_REFERENCE")

  saved NB07-REF-ID-03_source_domain_support_v001  png 62 KiB  sha 926fd9dd07d2…  한글=True


  saved NB07-REF-ID-04_cell_direction_support_v001  png 95 KiB  sha 1e4b7596e66b…  한글=True


{'figure_id': 'NB07-REF-ID-04_cell_direction_support_v001',
 'title_ko': '출처-도메인 셀 × 번역 방향 support',
 'rq': 'RQ6',
 'contract': 'G5_REVIEW_REGISTER',
 'tag': 'G5_REFERENCE',
 'note': 'G5 ID-04 — 방향 대비는 025 내부 변동에서만 식별된다',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/NB07-REF-ID-04_cell_direction_support_v001.png',
 'svg': 'outputs/figures/nb07/NB07-REF-ID-04_cell_direction_support_v001.svg',
 'png_sha256': '1e4b7596e66bc2910cdeed7e16b1a1a3c9c2e73da61e06bedc7a0abb78d866aa',
 'svg_sha256': 'd7e200548e74586c8978d98e757cb83a4c990c7d7b9551ea17083e93db9b45d4',
 'png_size_bytes': 97631,
 'svg_size_bytes': 29927,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

### 수치 해석 · 분포 해석

**ID-03.** 출처 2개 × 도메인 4개 중 실현된 셀은 **5개**뿐이고 **3개가 비어 있다**.
두 출처 모두에서 관측되는 도메인은 `other` 하나다: 025-other 1,165,510쌍,
026-other 990,120쌍. `dialogue`(516,162)와 `general`(804,291)은 025 전용,
`technology`(359,905)는 026 전용이다.

따라서 **025-other 대 026-other 비교만이 "동일 도메인 내 출처 층 대비
(same-domain source-stratum contrast)"** 로 성립한다. 이것조차 순수 source 효과가 아니다 —
같은 도메인 label 안에서도 두 출처의 문장 구성·번역 관행·표기 규약이 함께 다르기 때문이다.

**ID-04.** 5개 셀 × 3개 방향 15조합 중 **3개가 비어 있고**(026-other × EN_TO_KO,
026-technology × EN_TO_KO, 026-technology × UNKNOWN) **2개가 near-singleton**이다
(025-general × UNKNOWN = 31, 026-other × UNKNOWN = 1).
**026 계열에는 `EN_TO_KO` 관측이 하나도 없다.**
`UNKNOWN` 50,547쌍은 정책상 제거하지 않고 그대로 둔다.

### 연구적 의미

이 두 그림은 이후 모든 층별 진술의 유효 범위를 규정한다. 방향 대비를 해석할 때마다
"025 내부 변동에서 식별됨" 이라는 단서가 함께 붙어야 한다.
`cell × direction` 상호작용은 실현 설계에서 추정 불가능하며 도입하지 않는다.

### 해석 한계 · Not established

순수 source 효과와 순수 domain 효과는 이 자료에서 **분리되지 않는다**.
`source_domain_cell` 은 관측 층 조건부 통제일 뿐이다.
빈 셀에 대한 어떤 값도 추정되거나 외삽되지 않았다.

**Reference**: G5 `ID-03`, G5 `ID-04`

## 08 — B. 한국어 대 영어 token 수  ·  **F01**

### SSOT / RQ 대응

**Research Question**: RQ1 — 의미 대응 KO‑EN 문장쌍에서 o200k_base 기준 한국어
Tokenization Premium은 1보다 큰가?

**SSOT section**: `KOEN-TP-RS-001` §6.1, §7.2, §16.2 (F01 — KO vs EN paired token scatter, identity line) · §35

**Canonical input**: D-04 `TOKEN_O200K_BASE_v001`

**Physical variables**: `ko_token_count`, `en_token_count`

**Research purpose**: primary outcome의 원재료를 있는 그대로 보인다. TP는 이 두 수의 비이므로,
비를 보기 전에 두 수의 결합분포와 identity line 대비 위치를 먼저 확인한다.

**Allowed claim**: 두 token 수의 결합분포, identity line 위/아래 비율에 대한 기술적 진술.

**Prohibited claim**: 이 그림에 근거한 검정·유의성·효과 크기 확정 진술. RQ1의 결론은
NB08에서 이미 산출되었고 여기서 재산출하지 않는다.

In [9]:
_tok = fetch("ko_token_count, en_token_count, token_difference")
_ko = _tok["ko_token_count"].to_numpy(np.float64)
_en = _tok["en_token_count"].to_numpy(np.float64)
_dt = _tok["token_difference"].to_numpy(np.float64)

TOKEN_STATS = {
    "ko_token_count": qsummary("ko_token_count", "ko_token_count"),
    "en_token_count": qsummary("en_token_count", "en_token_count"),
    "token_difference": qsummary("token_difference", "token_difference"),
    "n_ko_gt_en": int((_ko > _en).sum()),
    "n_ko_eq_en": int((_ko == _en).sum()),
    "n_ko_lt_en": int((_ko < _en).sum()),
    "spearman_ko_en": float(stats.spearmanr(_ko, _en).statistic),
    "pearson_log_ko_en": float(np.corrcoef(np.log(_ko), np.log(_en))[0, 1]),
}
SUMMARY["token_counts"] = TOKEN_STATS
print(f"한국어 token 중앙값 {TOKEN_STATS['ko_token_count']['quantiles']['0.5']:,.0f} · "
      f"영어 token 중앙값 {TOKEN_STATS['en_token_count']['quantiles']['0.5']:,.0f}")
print(f"identity line 위 (KO > EN) {TOKEN_STATS['n_ko_gt_en']:,} "
      f"({TOKEN_STATS['n_ko_gt_en'] / COHORT['N'] * 100:.4f}%)")
print(f"identity line 상 (KO = EN) {TOKEN_STATS['n_ko_eq_en']:,} "
      f"({TOKEN_STATS['n_ko_eq_en'] / COHORT['N'] * 100:.4f}%)")
print(f"identity line 아래 (KO < EN) {TOKEN_STATS['n_ko_lt_en']:,} "
      f"({TOKEN_STATS['n_ko_lt_en'] / COHORT['N'] * 100:.4f}%)")
print(f"Spearman ρ(KO, EN) = {TOKEN_STATS['spearman_ko_en']:.6f}   "
      f"Pearson r(log KO, log EN) = {TOKEN_STATS['pearson_log_ko_en']:.6f}")

한국어 token 중앙값 21 · 영어 token 중앙값 16
identity line 위 (KO > EN) 3,375,095 (87.9850%)
identity line 상 (KO = EN) 196,718 (5.1282%)
identity line 아래 (KO < EN) 264,175 (6.8868%)
Spearman ρ(KO, EN) = 0.940656   Pearson r(log KO, log EN) = 0.941403


In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14.2, 5.6))

_lk, _le = np.log10(_ko), np.log10(_en)
_hi = float(max(_lk.max(), _le.max()))
density_panel(axes[0], _le, _lk, bins=220, xlim=(0, _hi), ylim=(0, _hi),
              xlabel="영어 token 수 (상용로그 눈금)", ylabel="한국어 token 수 (상용로그 눈금)",
              title="(a) 문장쌍별 token 수 결합분포 — 전수 밀도")
axes[0].plot([0, _hi], [0, _hi], color="#0b0b0b", lw=1.4, ls="--", label="동일선 (KO = EN)")
_ticks = np.array([0, 1, 2, 3])
for _ax in (axes[0],):
    _ax.set_xticks(_ticks, [f"{10 ** t:,.0f}" for t in _ticks])
    _ax.set_yticks(_ticks, [f"{10 ** t:,.0f}" for t in _ticks])
axes[0].legend(loc="upper left", framealpha=0.9)

_edges = np.unique(np.quantile(_en, np.linspace(0, 1, 61)))
_med, _, _ = stats.binned_statistic(_en, _ko, statistic="median", bins=_edges)
_q25, _, _ = stats.binned_statistic(_en, _ko, statistic=lambda v: np.quantile(v, 0.25), bins=_edges)
_q75, _, _ = stats.binned_statistic(_en, _ko, statistic=lambda v: np.quantile(v, 0.75), bins=_edges)
_ctr = 0.5 * (_edges[:-1] + _edges[1:])
_ok = np.isfinite(_med)
axes[1].fill_between(_ctr[_ok], _q25[_ok], _q75[_ok], color="#81c3d7", alpha=0.55,
                     label="사분위 구간 (Q1–Q3)")
axes[1].plot(_ctr[_ok], _med[_ok], color="#16425b", lw=2.0, label="구간별 한국어 token 중앙값")
axes[1].plot([0, _ctr[_ok].max()], [0, _ctr[_ok].max()], color="#c1272d", lw=1.4, ls="--",
             label="동일선 (KO = EN)")
axes[1].set_xlabel("영어 token 수 (60분위 구간)"); axes[1].set_ylabel("한국어 token 수")
axes[1].set_title("(b) 영어 token 수 조건부 한국어 token 수 — robust 요약")
axes[1].set_xlim(0, float(np.quantile(_en, 0.999))); axes[1].set_ylim(0, float(np.quantile(_ko, 0.999)))
axes[1].legend(loc="upper left")

fig.suptitle(f"F01 · 한국어–영어 token 수 관계 (o200k_base, N = {COHORT['N']:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "F01_token_count_relationship_v001", "한국어–영어 token 수 관계",
         rq="RQ1", contract="SSOT_§16.2_F01 — KO vs EN paired token scatter, identity line",
         note="전수 2D 밀도 + identity line + 구간별 robust 요약")

  saved F01_token_count_relationship_v001  png 178 KiB  sha 07cd8e33503f…  한글=True


{'figure_id': 'F01_token_count_relationship_v001',
 'title_ko': '한국어–영어 token 수 관계',
 'rq': 'RQ1',
 'contract': 'SSOT_§16.2_F01 — KO vs EN paired token scatter, identity line',
 'tag': 'CANONICAL',
 'note': '전수 2D 밀도 + identity line + 구간별 robust 요약',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/F01_token_count_relationship_v001.png',
 'svg': 'outputs/figures/nb07/F01_token_count_relationship_v001.svg',
 'png_sha256': '07cd8e33503febae766bf7ceef0e02a7e6508cf89757971634117dd61a37bced',
 'svg_sha256': 'e335392e64f617526f2cd091f7cb0e7d41e1c81c843a2c30f3a96ece51d14551',
 'png_size_bytes': 182669,
 'svg_size_bytes': 64737,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

### 수치 해석

한국어 token 중앙값은 21, 영어 token 중앙값은 16이다.
동일선 **위**(한국어가 더 많음) 3,375,095쌍(87.99%), 동일선 **상**(정확히 같음)
196,718쌍(5.13%), 동일선 **아래** 264,175쌍(6.89%)이다.
두 token 수의 Spearman ρ는 0.9407로 높다 — 문장 길이가 공통 요인으로 작용한다.

### 분포 해석

(a)의 밀도 덩어리 전체가 점선(동일선) **위쪽**으로 치우쳐 있다. (b)의 구간별 중앙값 곡선은
관측 범위 전체에서 동일선 위에 있고, 두 곡선의 간격이 영어 token 수가 커질수록 함께 넓어진다.
즉 격차는 짧은 문장에만 국한된 국소 현상이 아니라 길이 축 전체에 걸쳐 지속된다.
사분위 구간(Q1–Q3)의 하단조차 대부분 구간에서 동일선 위에 있다.

### 연구적 의미

이 그림은 RQ1의 방향성 관찰을 시각적으로 뒷받침한다. 동시에, 절대 차이 ΔT의 폭이
길이에 비례해 커진다는 점은 `D‑01` 결정(주 검정을 ΔT가 아닌 log TP에 대해 수행)의
실질적 근거를 그림으로 보여준다.

### 해석 한계

이 그림은 어떤 검정도 수행하지 않는다. 상관계수는 두 길이 척도의 공변을 요약할 뿐이며
어떤 인과 구조도 나타내지 않는다. 동일선 위 비율 87.99%는 표본 비율이지
`P(TP > 1)` 의 추정 절차 결과가 아니다(그 값은 NB08에 있다).

## 09 — C. TP · log TP · ΔT  ·  **F02**

### SSOT / RQ 대응

**Research Question**: RQ1 — 의미 대응 KO‑EN 문장쌍에서 o200k_base 기준 한국어
Tokenization Premium은 1보다 큰가?

**SSOT section**: `KOEN-TP-RS-001` §6.1 (주 estimand `θ_TP = Median(log TP_i)`),
§7.2 (Primary `log_token_premium`, Secondary `token_premium` · `token_difference`),
§8.1, §16.2 (F02 — TP histogram + ECDF) · §35

**Canonical input**: D-04 `TOKEN_O200K_BASE_v001`

**Physical variables**: `token_premium`, `log_token_premium`, `token_difference`

**Research purpose**: primary outcome의 전수 분포·ECDF·격자 구조를 기술한다.

**Allowed claim**: 분포 형태, 분위수, 이산 격자(lattice) 구조, 경계값 비율에 대한 기술.

**Prohibited claim**: median의 신뢰구간·검정·p-value. 이들은 NB08의 결과이며 재산출하지 않는다.

In [11]:
_tp = fetch("token_premium, log_token_premium")
_TP = _tp["token_premium"].to_numpy(np.float64)
_LTP = _tp["log_token_premium"].to_numpy(np.float64)

TP_STATS = {
    "token_premium": qsummary("token_premium", "token_premium",
                              qs=(0.0, 0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1.0)),
    "log_token_premium": qsummary("log_token_premium", "log_token_premium",
                                  qs=(0.0, 0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999, 1.0)),
    "token_difference": TOKEN_STATS["token_difference"],
    "share_TP_gt_1": float((_TP > 1).mean()),
    "share_TP_eq_1": float((_TP == 1).mean()),
    "share_TP_lt_1": float((_TP < 1).mean()),
    "distinct_TP_values": int(con.execute(f"SELECT count(DISTINCT token_premium) FROM {A}").fetchone()[0]),
    "iqr_log_token_premium": float(np.quantile(_LTP, 0.75) - np.quantile(_LTP, 0.25)),
    "mad_log_token_premium": float(np.median(np.abs(_LTP - np.median(_LTP)))),
}
SUMMARY["token_premium"] = TP_STATS
print(f"TP    중앙값 {TP_STATS['token_premium']['quantiles']['0.5']:.6f}  "
      f"평균 {TP_STATS['token_premium']['mean']:.6f}  "
      f"범위 [{TP_STATS['token_premium']['quantiles']['0.0']:.6f}, "
      f"{TP_STATS['token_premium']['quantiles']['1.0']:.6f}]")
print(f"logTP 중앙값 {TP_STATS['log_token_premium']['quantiles']['0.5']:.10f}  "
      f"평균 {TP_STATS['log_token_premium']['mean']:.10f}  "
      f"IQR {TP_STATS['iqr_log_token_premium']:.6f}  MAD {TP_STATS['mad_log_token_premium']:.6f}")
print(f"TP > 1 {TP_STATS['share_TP_gt_1'] * 100:.4f}% · TP = 1 {TP_STATS['share_TP_eq_1'] * 100:.4f}% · "
      f"TP < 1 {TP_STATS['share_TP_lt_1'] * 100:.4f}%")
print(f"서로 다른 TP 값 개수 = {TP_STATS['distinct_TP_values']:,} — TP는 두 정수의 비이므로 이산 격자다")

TP    중앙값 1.333333  평균 1.362988  범위 [0.064018, 38.000000]
logTP 중앙값 0.2876820725  평균 0.2851767862  IQR 0.273293  MAD 0.137300
TP > 1 87.9850% · TP = 1 5.1282% · TP < 1 6.8868%
서로 다른 TP 값 개수 = 3,725 — TP는 두 정수의 비이므로 이산 격자다


In [12]:
fig, axes = plt.subplots(1, 2, figsize=(14.2, 5.4))

_lo, _hi = float(np.quantile(_LTP, 0.0005)), float(np.quantile(_LTP, 0.9995))
axes[0].hist(_LTP, bins=400, range=(_lo, _hi), color="#3a7ca5", edgecolor="none")
_med = float(np.median(_LTP))
axes[0].axvline(0.0, color="#0b0b0b", lw=1.4, ls="--", label="TP = 1 (log TP = 0)")
axes[0].axvline(_med, color="#c1272d", lw=1.8, label=f"중앙값 log TP = {_med:.6f}  (TP = 4/3)")
axes[0].set_xlabel("log Tokenization Premium  =  ln(한국어 token 수 / 영어 token 수)")
axes[0].set_ylabel("문장쌍 수")
axes[0].set_title("(a) log TP 전수 분포 (0.05–99.95 분위 구간)")
axes[0].legend(loc="upper left")
_sec = axes[0].secondary_xaxis("top", functions=(np.exp, np.log))
_sec.set_xlabel("Tokenization Premium (비율 눈금)", fontsize=9)

_srt = np.sort(_LTP)
_ecdf = np.arange(1, _srt.size + 1) / _srt.size
_step = max(_srt.size // 200_000, 1)
axes[1].plot(_srt[::_step], _ecdf[::_step], color="#16425b", lw=1.8)
axes[1].axvline(0.0, color="#0b0b0b", lw=1.2, ls="--")
axes[1].axhline(0.5, color="#8d99ae", lw=1.0, ls=":")
axes[1].plot([_med], [0.5], marker="o", ms=7, color="#c1272d",
             label=f"중앙값 {_med:.4f}")
_at0 = float((_LTP < 0).mean())
axes[1].annotate(f"log TP < 0 인 비율 = {_at0 * 100:.2f}%\n(TP = 1 정확히 일치 {TP_STATS['share_TP_eq_1'] * 100:.2f}%)",
                 xy=(0.0, _at0), xytext=(0.05, 0.22), fontsize=9,
                 arrowprops={"arrowstyle": "->", "color": "#0b0b0b", "lw": 1.0})
axes[1].set_xlim(_lo, _hi); axes[1].set_ylim(0, 1)
axes[1].set_xlabel("log Tokenization Premium"); axes[1].set_ylabel("누적 비율 (ECDF)")
axes[1].set_title("(b) log TP 경험적 누적분포")
axes[1].legend(loc="lower right")

fig.suptitle(f"F02 · Tokenization Premium 분포와 ECDF (N = {COHORT['N']:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "F02_tp_distribution_ecdf_v001", "Tokenization Premium 분포와 ECDF",
         rq="RQ1", contract="SSOT_§16.2_F02 — TP histogram + ECDF",
         note="log TP 전수 히스토그램 + ECDF, TP=1 경계와 중앙값 표시")

/home/sieg/projects-wsl/Tokenization_KOEN/.venv/lib/python3.12/site-packages/matplotlib/scale.py:263: RuntimeWarning: divide by zero encountered in log
  return self._forward(values)


  saved F02_tp_distribution_ecdf_v001  png 126 KiB  sha 36d5fe0f9c70…  한글=True


{'figure_id': 'F02_tp_distribution_ecdf_v001',
 'title_ko': 'Tokenization Premium 분포와 ECDF',
 'rq': 'RQ1',
 'contract': 'SSOT_§16.2_F02 — TP histogram + ECDF',
 'tag': 'CANONICAL',
 'note': 'log TP 전수 히스토그램 + ECDF, TP=1 경계와 중앙값 표시',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/F02_tp_distribution_ecdf_v001.png',
 'svg': 'outputs/figures/nb07/F02_tp_distribution_ecdf_v001.svg',
 'png_sha256': '36d5fe0f9c70cd563b80a20e3708da75da42c5dbc91dc4a178aa0f0b1e0d1c1d',
 'svg_sha256': '3f6dee4d34d7f87fe511a402ef8d43a8d9cdd7750ef8c2b6d9067eb15c934fc8',
 'png_size_bytes': 129099,
 'svg_size_bytes': 137049,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(14.2, 4.6))
_lo_d, _hi_d = float(np.quantile(_dt, 0.001)), float(np.quantile(_dt, 0.999))
axes[0].hist(_dt, bins=int(_hi_d - _lo_d) + 1, range=(_lo_d, _hi_d), color="#7f9c96", edgecolor="none")
axes[0].axvline(0.0, color="#c1272d", lw=1.6, ls="--", label="ΔT = 0")
axes[0].set_xlabel("ΔT = 한국어 token 수 − 영어 token 수"); axes[0].set_ylabel("문장쌍 수")
axes[0].set_title("(a) 절대 token 차이 ΔT 분포 (0.1–99.9 분위)")
axes[0].legend()

_bins = np.unique(np.quantile(_en, np.linspace(0, 1, 41)))
_mdt, _, _ = stats.binned_statistic(_en, _dt, statistic="median", bins=_bins)
_ctr = 0.5 * (_bins[:-1] + _bins[1:])
_ok = np.isfinite(_mdt)
axes[1].plot(_ctr[_ok], _mdt[_ok], color="#16425b", lw=2.0, marker="o", ms=3,
             label="구간별 ΔT 중앙값")
axes[1].axhline(0.0, color="#c1272d", lw=1.2, ls="--")
axes[1].set_xlabel("영어 token 수 (40분위 구간)"); axes[1].set_ylabel("ΔT 중앙값")
axes[1].set_title("(b) ΔT는 문장 길이에 비례해 커진다 — 단위 의존성 (SSOT D‑01의 근거)")
axes[1].set_xlim(0, float(np.quantile(_en, 0.999)))
axes[1].legend()
fig.suptitle("NB07-S02 · 절대 token 차이 ΔT의 보조 기술 (Secondary outcome)", fontsize=13)
fig.tight_layout()
save_fig(fig, "NB07-S02_token_difference_v001", "절대 token 차이 ΔT 기술",
         rq="RQ1", contract="SUPPORTING_DESCRIPTIVE",
         note="ΔT 분포와 길이 의존성 — SSOT D-01 근거의 시각화")
del _tok, _tp

  saved NB07-S02_token_difference_v001  png 93 KiB  sha 8a65189785e5…  한글=True


### 수치 해석

TP 중앙값은 정확히 1.333333…(= 4/3), 평균은 1.362988이다.
log TP 중앙값은 0.2876820724517808, IQR은 0.273293, MAD는 0.137300이다.
서로 다른 TP 값은 3,725개뿐이다.
TP > 1 비율 87.985%, TP = 1 비율 5.128%, TP < 1 비율 6.887%.
관측 범위는 TP ∈ [0.0640, 38.0]으로, 양쪽 꼬리가 모두 존재한다.

### 분포 해석

log TP 분포는 0보다 오른쪽에 중심을 두고 있으며, 눈에 띄는 **이산 격자(lattice) 구조**를 갖는다.
TP는 두 정수의 비이므로 짧은 문장에서 소수의 값(1/1, 4/3, 3/2, 2/1 …)에 질량이 집중된다.
ECDF가 log TP = 0 부근에서 계단처럼 수직으로 뛰는 것이 그 직접적 결과다 —
정확히 TP = 1인 문장쌍이 196,718개(5.13%) 있기 때문이다.
이 격자 구조는 NB08이 보고한 median CI의 퇴화(degenerate interval)와 같은 원인에서 나온다.

ΔT는 중앙값 5 token이지만, (b)에서 보이듯 영어 token 수가 커질수록 ΔT 중앙값이 함께 커진다.
즉 ΔT의 단위는 문장 길이에 의존한다.

### 연구적 의미

분포가 이산 격자라는 사실은 기술통계 단계에서 반드시 드러나야 한다.
ΔT의 길이 의존성은 SSOT `Decision D‑01`(주 검정을 ΔT가 아니라 log TP에 대해 수행)이
형식적 선택이 아니라 자료 구조에서 나온 요구임을 보여준다.

### 해석 한계

여기 어떤 값도 검정 결과나 구간 추정치가 아니다. 표본 비율 87.985%는 `P(TP > 1)` 의
기술 통계이며, NB08이 보고한 값과 같은 수라도 그 지위는 다르다.
격자 구조에 대한 관찰은 분포의 형태에 관한 것이지, 어떤 추론 절차의 타당성을
확정하거나 부정하지 않는다.

**Reference**: `[EDA-REF-A02]` (TP = 1 정확 일치), `[EDA-REF-A03]` · `[EDA-REF-A04]` (양측 꼬리) — §21

## 10 — RQ1 주석 (annotation only, 재검정 없음)

### SSOT / RQ 대응

**Research Question**: RQ1 — 의미 대응 KO‑EN 문장쌍에서 o200k_base 기준 한국어
Tokenization Premium은 1보다 큰가?

**SSOT section**: `KOEN-TP-RS-001` §6.1, §17; `RD-RQ1-FIRST-RESULT-01`;
NB08 `NB08_RQ1_PROTOCOL_v001` closeout

**Research purpose**: RQ1은 NB08에서 **이미 추론되어 종결**되었다. NB07은 그 결과를 인용해
위 기술 그림에 좌표를 부여할 뿐이며, 어떤 검정도 다시 실행하지 않는다.

**Allowed claim**: NB08 결과의 인용과, 그 결과가 §08–§09의 기술 분포와 정합적이라는 진술.

**Prohibited claim**: NB07이 RQ1을 검정했다는 진술. 새로운 p-value·CI·효과 크기.

In [14]:
RQ1 = json.loads((ROOT / "ssot_nb01" / "06_NB08_RQ1_SSOT_CLOSEOUT_v001.json").read_text(encoding="utf-8"))
_interp = RQ1["INTERPRETATION"]
_cross = {
    "nb08_median_logTP": _interp["median_logTP"],
    "nb07_descriptive_median_logTP": float(np.median(_LTP)),
    "agree_to": abs(_interp["median_logTP"] - float(np.median(_LTP))),
    "nb08_P_TP_gt_1": _interp["P_TP_gt_1"],
    "nb07_descriptive_share_TP_gt_1": TP_STATS["share_TP_gt_1"],
    "nb08_ties_excluded": RQ1["CONDITIONAL_NONZERO_SIGN_TEST"]["primary"]["ties_excluded"],
    "nb07_descriptive_n_TP_eq_1": int((_TP == 1).sum()),
}
SUMMARY["rq1_annotation"] = {
    "status": "CLOSED_IN_NB08 — NB07 annotates only, does not re-test",
    "source": "ssot_nb01/06_NB08_RQ1_SSOT_CLOSEOUT_v001.json",
    "median_logTP": _interp["median_logTP"],
    "median_TP_scale": _interp["median_TP_scale"],
    "median_TP_as_ratio": _interp["median_TP_as_ratio"],
    "median_premium_percent": _interp["median_premium_percent"],
    "P_TP_gt_1": _interp["P_TP_gt_1"],
    "ci_degenerate": RQ1["CI_DEGENERACY"]["order_statistic_interval"]["degenerate"],
    "publication_p_form": RQ1["REPORTING"]["publication_form"],
    "cross_check_against_nb07_descriptive": _cross,
}
print("NB08 RQ1 (인용, 재검정 없음)")
print(f"  median log TP           = {_interp['median_logTP']:.16f}")
print(f"  median TP               = {_interp['median_TP_scale']:.16f}  ({_interp['median_TP_as_ratio']})")
print(f"  median premium          = {_interp['median_premium_percent']:.4f}%")
print(f"  P(TP > 1)               = {_interp['P_TP_gt_1']:.10f}")
print(f"  median CI               = degenerate (점질량 안에 두 order statistic이 모두 들어감)")
print(f"  보고 형식               = {RQ1['REPORTING']['publication_form']}")
print("\nNB07 기술통계와의 정합 확인 (독립 재계산):")
print(f"  median log TP 차이      = {_cross['agree_to']:.3e}")
print(f"  TP = 1 개수: NB08 tie {_cross['nb08_ties_excluded']:,} vs NB07 {_cross['nb07_descriptive_n_TP_eq_1']:,}")
print(f"  P(TP>1): NB08 {_cross['nb08_P_TP_gt_1']:.8f} vs NB07 표본비율 {_cross['nb07_descriptive_share_TP_gt_1']:.8f}")

NB08 RQ1 (인용, 재검정 없음)
  median log TP           = 0.2876820724517808
  median TP               = 1.3333333333333333  (4 / 3)
  median premium          = 33.3333%
  P(TP > 1)               = 0.8798502498
  median CI               = degenerate (점질량 안에 두 order statistic이 모두 들어감)
  보고 형식               = p < 1e-300

NB07 기술통계와의 정합 확인 (독립 재계산):
  median log TP 차이      = 0.000e+00
  TP = 1 개수: NB08 tie 196,718 vs NB07 196,718
  P(TP>1): NB08 0.87985025 vs NB07 표본비율 0.87985025


### 수치 해석

NB08이 종결한 RQ1 결과: median log TP = 0.2876820724517808, 즉 median TP = 4/3 = 1.3333…,
중앙값 기준 premium 33.33%. `P(TP > 1) = 0.87985`. median의 order-statistic 구간은
**퇴화**했다 — 하한과 상한 rank가 모두 같은 점질량(123,040개 관측) 안에 들어가기 때문이다.
p-value는 `p < 1e-300` 형식으로만 보고된다.

### 정합성

NB07이 독립적으로 다시 계산한 기술통계는 NB08 값과 정확히 일치한다:
median log TP 차이 0, TP = 1 개수 196,718 대 196,718, `P(TP > 1)` 표본비율 0.87985.
두 노트북이 같은 동결 cohort에서 같은 outcome 열을 읽었으므로 이는 재현성 확인이지
독립적 증거의 추가가 아니다.

### 연구적 의미

§08–§09의 기술 그림들은 NB08 결과와 같은 방향을 가리키며, 그 결과에 분포적 맥락을 준다.
특히 CI 퇴화의 원인(격자 + 거대 점질량)이 F02의 ECDF 계단으로 눈에 보인다.

### 해석 한계

NB07은 RQ1을 **검정하지 않았다**. 위 표의 추론값은 전부 NB08에서 인용된 것이다.
NB07의 기술통계는 NB08 결과를 확증하거나 반증하지 않는다 — 같은 자료의 다른 표현이다.

## 11 — D. Exact decomposition  ·  **F04**  ·  RQ2 canonical 결과

### SSOT / RQ 대응

**Research Question**: RQ2 — 전체 premium 가운데 CodePointRatio · ByteDensityRatio ·
CompressionPenalty 세 요소가 각각 어느 정도의 분포를 보이는가?

**SSOT section**: `KOEN-TP-RS-001` §6.2, §8 (§8.1–§8.4 정의와 정확 분해식),
Interpretation Rule IR‑01, §16.2 (F04 — exact decomposition component distribution) · §35

**Canonical input**: D-02 `REP_FEATURES_v002` + D-04 `TOKEN_O200K_BASE_v001`

**Physical variables**: `log_token_premium`, `log_code_point_ratio`,
`log_byte_density_ratio`, `log_compression_penalty`

**Research purpose**: SSOT §8의 항등식
`log TP_i = log CR_i + log BDR_i + log CP_i` 를 **전수 N에서 재검증**하고,
세 성분의 분포와 기여를 기술한다. 이것이 NB07의 canonical RQ2 결과다.

**Allowed claim**: 항등식의 수치적 성립, 세 성분의 분포·중앙값·기여 비중에 대한 기술.

**Prohibited claim**: 세 성분 중 하나가 premium을 "설명한다"·"일으킨다"는 진술.
이 분해는 대수적 항등식이지 인과 분해가 아니다.

In [15]:
_dec = fetch("log_token_premium, log_code_point_ratio, log_byte_density_ratio, log_compression_penalty")
_ltp = _dec["log_token_premium"].to_numpy(np.float64)
_lcr = _dec["log_code_point_ratio"].to_numpy(np.float64)
_lbdr = _dec["log_byte_density_ratio"].to_numpy(np.float64)
_lcp = _dec["log_compression_penalty"].to_numpy(np.float64)

_resid = _ltp - (_lcr + _lbdr + _lcp)
_max_abs = float(np.max(np.abs(_resid)))
_max_rel = float(np.max(np.abs(_resid) / np.maximum(np.abs(_ltp), 1e-12)))
_tol = 1e-12
DECOMP = {
    "identity": "log TP = log CodePointRatio + log ByteDensityRatio + log CompressionPenalty",
    "rows_checked": int(_ltp.size),
    "max_abs_error": _max_abs,
    "mean_abs_error": float(np.mean(np.abs(_resid))),
    "max_relative_error": _max_rel,
    "tolerance": _tol,
    "within_tolerance": bool(_max_abs <= _tol),
    "rows_outside_tolerance": int((np.abs(_resid) > _tol).sum()),
    "double_eps": float(np.finfo(np.float64).eps),
}
if not DECOMP["within_tolerance"]:
    raise SystemExit(f"DECOMPOSITION_FAIL_CLOSED: max abs error {_max_abs} > {_tol}")

for _n, _v in (("log TP", _ltp), ("log CR", _lcr), ("log BDR", _lbdr), ("log CP", _lcp)):
    DECOMP[_n] = {"mean": float(_v.mean()), "median": float(np.median(_v)), "sd": float(_v.std(ddof=1)),
                  "q05": float(np.quantile(_v, 0.05)), "q95": float(np.quantile(_v, 0.95)),
                  "share_positive": float((_v > 0).mean())}
DECOMP["mean_contribution_share"] = {
    k: float(DECOMP[k]["mean"] / DECOMP["log TP"]["mean"]) for k in ("log CR", "log BDR", "log CP")}
DECOMP["median_contribution_at_median_pair"] = {
    k: DECOMP[k]["median"] for k in ("log CR", "log BDR", "log CP")}
SUMMARY["exact_decomposition"] = DECOMP

print(f"항등식 전수 재검증 — {DECOMP['rows_checked']:,}행")
print(f"  max |오차|      = {_max_abs:.3e}   (허용 {_tol:.0e}, float64 eps {DECOMP['double_eps']:.3e})")
print(f"  mean |오차|     = {DECOMP['mean_abs_error']:.3e}")
print(f"  허용 초과 행 수 = {DECOMP['rows_outside_tolerance']}")
print(f"  RQ2_DECOMPOSITION_STATUS = EXACT_IDENTITY_VERIFIED_FULL_COHORT\n")
show(pd.DataFrame([{"성분": k, "평균": DECOMP[k]["mean"], "중앙값": DECOMP[k]["median"],
                    "표준편차": DECOMP[k]["sd"], "5%": DECOMP[k]["q05"], "95%": DECOMP[k]["q95"],
                    "양수 비율": DECOMP[k]["share_positive"]}
                   for k in ("log TP", "log CR", "log BDR", "log CP")]))
print("\n평균 기여 비중 (mean 성분 / mean log TP):")
for _k, _v in DECOMP["mean_contribution_share"].items():
    print(f"  {_k:8s} {_v * 100:8.3f}%")

항등식 전수 재검증 — 3,835,988행
  max |오차|      = 8.882e-16   (허용 1e-12, float64 eps 2.220e-16)
  mean |오차|     = 9.704e-17
  허용 초과 행 수 = 0
  RQ2_DECOMPOSITION_STATUS = EXACT_IDENTITY_VERIFIED_FULL_COHORT

     성분        평균       중앙값      표준편차         5%       95%      양수 비율
 log TP  0.285177  0.287682  0.222096 -0.0870114  0.635989    0.87985
 log CR -0.753588 -0.760451     0.222   -1.09861 -0.386773 0.00193275
log BDR  0.878476  0.896088 0.0662249   0.757096  0.944462   0.999653
 log CP  0.160289  0.175154  0.187665  -0.167054  0.440663    0.80536

평균 기여 비중 (mean 성분 / mean log TP):
  log CR   -264.253%
  log BDR   308.046%
  log CP     56.207%


In [16]:
fig, axes = plt.subplots(1, 3, figsize=(16.0, 5.0))
_cols = {"log CR": "#c1272d", "log BDR": "#e8a33d", "log CP": "#2f6690", "log TP": "#3f3f3f"}
_names = {"log CR": "log CodePointRatio (문자 수 비)",
          "log BDR": "log ByteDensityRatio (UTF-8 byte 밀도 비)",
          "log CP": "log CompressionPenalty (tokenizer 압축 비)",
          "log TP": "log TokenizationPremium (합계)"}
_arr = {"log CR": _lcr, "log BDR": _lbdr, "log CP": _lcp, "log TP": _ltp}

_lo = min(float(np.quantile(v, 0.001)) for v in _arr.values())
_hi = max(float(np.quantile(v, 0.999)) for v in _arr.values())
for _k in ("log TP", "log CR", "log BDR", "log CP"):
    axes[0].hist(_arr[_k], bins=300, range=(_lo, _hi), histtype="step", lw=1.8,
                 color=_cols[_k], label=_names[_k])
axes[0].axvline(0.0, color="#0b0b0b", lw=1.0, ls=":")
axes[0].set_xlabel("로그 비율 값"); axes[0].set_ylabel("문장쌍 수")
axes[0].set_title("(a) 세 성분과 합계의 전수 분포")
axes[0].legend(fontsize=8, loc="upper left")

_mean = [DECOMP[k]["mean"] for k in ("log CR", "log BDR", "log CP")]
_median = [DECOMP[k]["median"] for k in ("log CR", "log BDR", "log CP")]
_x = np.arange(3)
axes[1].bar(_x - 0.2, _mean, width=0.38, color=["#c1272d", "#e8a33d", "#2f6690"], label="평균")
axes[1].bar(_x + 0.2, _median, width=0.38, color=["#c1272d", "#e8a33d", "#2f6690"],
            alpha=0.5, hatch="//", label="중앙값")
for _i, (_m, _md) in enumerate(zip(_mean, _median, strict=True)):
    axes[1].text(_i - 0.2, _m, f"{_m:+.4f}", ha="center",
                 va="bottom" if _m >= 0 else "top", fontsize=8.5)
    axes[1].text(_i + 0.2, _md, f"{_md:+.4f}", ha="center",
                 va="bottom" if _md >= 0 else "top", fontsize=8.5)
axes[1].axhline(0.0, color="#0b0b0b", lw=1.0)
axes[1].axhline(DECOMP["log TP"]["mean"], color="#3f3f3f", lw=1.6, ls="--",
                label=f"합계 평균 log TP = {DECOMP['log TP']['mean']:+.4f}")
axes[1].set_xticks(_x, ["log CR\n문자 수 비", "log BDR\nbyte 밀도 비", "log CP\n압축 penalty"])
axes[1].set_ylabel("로그 비율 값")
axes[1].set_title("(b) 성분별 평균·중앙값 기여 — 세 값의 합이 log TP")
axes[1].legend(fontsize=8.5)

_rb = np.max(np.abs(_resid)) * 1.15 or 1e-16
axes[2].hist(_resid, bins=121, range=(-_rb, _rb), color="#7f9c96", edgecolor="none")
axes[2].axvline(0.0, color="#c1272d", lw=1.2, ls="--")
axes[2].set_xlabel("잔차  log TP − (log CR + log BDR + log CP)")
axes[2].set_ylabel("문장쌍 수 (로그 눈금)"); axes[2].set_yscale("log")
axes[2].set_title(f"(c) 항등식 잔차 — 최대 |오차| {_max_abs:.2e}\n"
                  f"float64 반올림 한계 수준, 허용 초과 0행")
axes[2].ticklabel_format(axis="x", style="sci", scilimits=(0, 0))

fig.suptitle(f"F04 · 정확 분해 log TP = log CR + log BDR + log CP (전수 N = {COHORT['N']:,})",
             fontsize=13)
fig.tight_layout()
save_fig(fig, "F04_exact_decomposition_v001", "정확 분해 성분 분포",
         rq="RQ2", contract="SSOT_§16.2_F04 — exact decomposition component distribution",
         note="전수 항등식 재검증 + 성분 분포 + 잔차")

  saved F04_exact_decomposition_v001  png 166 KiB  sha da415362a7fb…  한글=True


{'figure_id': 'F04_exact_decomposition_v001',
 'title_ko': '정확 분해 성분 분포',
 'rq': 'RQ2',
 'contract': 'SSOT_§16.2_F04 — exact decomposition component distribution',
 'tag': 'CANONICAL',
 'note': '전수 항등식 재검증 + 성분 분포 + 잔차',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/F04_exact_decomposition_v001.png',
 'svg': 'outputs/figures/nb07/F04_exact_decomposition_v001.svg',
 'png_sha256': 'da415362a7fb2b3ed8406223068225c95cdaf30eb992ada4f8ae74e4f450d0bc',
 'svg_sha256': '90860b377c20364b6d1a01e622c1d19c7a5aa5ac991f811b246459e056558c28',
 'png_size_bytes': 169799,
 'svg_size_bytes': 105180,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

### 수치 해석

항등식은 **전수 3,835,988행에서 성립**한다. 최대 절대오차 8.88e-16으로 float64 반올림
한계 수준이며, 허용치 1e-12를 넘는 행은 0개다.
`RQ2_DECOMPOSITION_STATUS = EXACT_IDENTITY_VERIFIED_FULL_COHORT`.

성분별 평균: log CR = −0.753588, log BDR = +0.878476, log CP = +0.160289.
성분별 중앙값: log CR = −0.760451, log BDR = +0.896088, log CP = +0.175154.
합계 평균 log TP = +0.285177 (= −0.753588 + 0.878476 + 0.160289).

### 분포 해석

세 성분의 역할이 서로 완전히 다르다.

- **log CR (문자 수 비)** 은 거의 전부 **음수**다 — 양수 비율이 0.193%에 불과하고 99.67%가
  0 미만이다. 한국어 문장은 같은 내용을 영어보다 **적은 code point**로 표현한다.
  이 성분만 보면 한국어가 유리하며, 세 성분 중 산포(표준편차 0.2220)도 가장 크다.
- **log BDR (byte 밀도 비)** 은 사실상 전부 **양수**다(99.965%). 세 성분 중 평균 절대값이
  가장 크고 산포는 가장 작다(표준편차 0.0662). 한글 음절은 UTF-8에서 3 byte, ASCII 라틴
  문자는 1 byte이므로 구조적으로 거의 상수처럼 발생한다.
- **log CP (압축 penalty)** 는 중앙값이 양수(+0.1752)이고 80.54%가 양수, 18.84%가 음수다.
  세 성분 중 유일하게 부호가 실질적으로 갈린다.

즉 관측된 premium은 "한국어 글자가 많아서"라는 형태로는 기술되지 않는다.
항등식 위에서 보면 문자 수 우위(log CR = −0.75)가 byte 밀도 열위(log BDR = +0.88)에
의해 상쇄되고도 남고, 거기에 압축 열위(log CP = +0.16)가 더해져 순 premium(+0.29)이
남는다. 이것은 세 항의 **산술적 구성**에 대한 진술이며, 어느 항이 premium을 일으킨다는
인과 진술이 아니다.

(b)의 "평균 기여 비중"은 세 성분 평균을 합계 평균으로 나눈 산술 비다: log CR −264.3%,
log BDR +308.0%, log CP +56.2%. 합계가 100%지만 개별 값이 100%를 넘거나 음수인 이유는
분모인 순 합계(0.285)가 개별 성분 절대값보다 훨씬 작기 때문이다. 이 수치는 **크기 비교용
산술 분해**이며 설명력 분해가 아니다.

### 연구적 의미

SSOT `Interpretation Rule IR‑01` 이 요구하는 분리 보고가 여기서 실현된다:
ByteDensityRatio가 크다는 사실(UTF‑8 표현 구조)과 CompressionPenalty가 크다는 사실
(고정 tokenizer의 압축 성능)은 서로 다른 이야기이며, F04는 둘을 같은 축 위에 나란히 놓되
합치지 않는다.

### 해석 한계

이 분해는 **대수적 항등식**이다. "성분 X가 premium의 Y%를 설명한다"는 표현은
회귀적 설명력과 혼동될 수 있으므로 쓰지 않는다. 위 (b)의 기여 비중은 평균의 산술적 분해일 뿐,
조건부 연관이나 인과 기여가 아니다. 어떤 성분도 다른 성분을 통제한 상태에서 측정되지 않았다.

## 12 — E. 표현 구조: 문자·byte·공백·문자군·표현 역전  ·  **F05**

### SSOT / RQ 대응

**Research Question**: RQ3 — 공백 밀도, grapheme 구조, Hangul/Latin/digit/punctuation 비율,
script mixing, 문장 길이, 숫자·특수기호가 log(TP)와 log(CompressionPenalty)에
어떤 조건부 연관을 갖는가?

**SSOT section**: `KOEN-TP-RS-001` §6.3, §8.2–§8.4, IR‑01, §16.2 (F05 — ByteDensityRatio vs
CompressionPenalty) · §35

**Canonical input**: D-02 `REP_FEATURES_v002` + D-04 `TOKEN_O200K_BASE_v001`

**Physical variables**: `ko/en_bytes_per_codepoint`, `ko/en_whitespace_density`,
`ko/en_*_share`, `ko/en_script_type_count`, `ko/en_script_switch_count`,
`log_byte_density_ratio`, `log_compression_penalty`, `log_code_point_ratio`

**Research purpose**: RQ3의 **표면형 기술 구조만** 제시한다. 조건부 연관의 추정은
NB09 M1 vs M0의 몫이다.

**Allowed claim**: 주변분포·결합밀도·구간별 중앙값 등 기술적 진술.

**Prohibited claim**: "X가 log TP를 높인다", "X를 통제하면", 부분효과, 계수, 조건부 연관의 크기.

In [17]:
_rep = fetch("""log_byte_density_ratio, log_compression_penalty, log_code_point_ratio,
                log_token_premium, ko_bytes_per_codepoint, en_bytes_per_codepoint,
                ko_whitespace_density, en_whitespace_density,
                ko_tokens_per_byte, en_tokens_per_byte""")

REP_STATS = {c: qsummary(c, c) for c in (
    "ko_bytes_per_codepoint", "en_bytes_per_codepoint",
    "ko_whitespace_density", "en_whitespace_density",
    "ko_tokens_per_byte", "en_tokens_per_byte",
    "ko_tokens_per_codepoint", "en_tokens_per_codepoint")}
_shares = con.execute(f"""SELECT
  avg(ko_hangul_share) ko_hangul, avg(ko_latin_share) ko_latin, avg(ko_digit_share) ko_digit,
  avg(ko_punctuation_share) ko_punct, avg(ko_symbol_other_share) ko_symbol,
  avg(ko_whitespace_density) ko_ws,
  avg(en_hangul_share) en_hangul, avg(en_latin_share) en_latin, avg(en_digit_share) en_digit,
  avg(en_punctuation_share) en_punct, avg(en_symbol_other_share) en_symbol,
  avg(en_whitespace_density) en_ws FROM {A}""").fetchdf().iloc[0].to_dict()
REP_STATS["mean_script_composition"] = {k: float(v) for k, v in _shares.items()}
REP_STATS["composition_closure_ko"] = float(sum(
    _shares[k] for k in ("ko_hangul", "ko_latin", "ko_digit", "ko_punct", "ko_symbol", "ko_ws")))
REP_STATS["composition_closure_en"] = float(sum(
    _shares[k] for k in ("en_hangul", "en_latin", "en_digit", "en_punct", "en_symbol", "en_ws")))

_rev = con.execute(f"""SELECT
  sum(CASE WHEN log_code_point_ratio < 0 THEN 1 ELSE 0 END) AS cr_neg,
  sum(CASE WHEN log_code_point_ratio < 0 AND log_token_premium > 0 THEN 1 ELSE 0 END) AS reversal,
  sum(CASE WHEN log_code_point_ratio < 0 AND log_token_premium <= 0 THEN 1 ELSE 0 END) AS cr_neg_no_rev,
  sum(CASE WHEN log_code_point_ratio >= 0 AND log_token_premium > 0 THEN 1 ELSE 0 END) AS both_pos,
  sum(CASE WHEN log_byte_density_ratio > 0 THEN 1 ELSE 0 END) AS bdr_pos,
  sum(CASE WHEN log_compression_penalty > 0 THEN 1 ELSE 0 END) AS cp_pos
  FROM {A}""").fetchdf().iloc[0].astype(np.int64).to_dict()
REP_STATS["representation_reversal"] = {k: int(v) for k, v in _rev.items()}
REP_STATS["representation_reversal_share"] = float(_rev["reversal"] / COHORT["N"])
REP_STATS["spearman_lbdr_lcp"] = float(stats.spearmanr(
    _rep["log_byte_density_ratio"], _rep["log_compression_penalty"]).statistic)
SUMMARY["representation_structure"] = REP_STATS

print("평균 문자군 구성 (문자 비율 + 공백 밀도, 각 언어별 합 = 1):")
show(pd.DataFrame([
    {"언어": "한국어", "한글": _shares["ko_hangul"], "라틴": _shares["ko_latin"],
     "숫자": _shares["ko_digit"], "문장부호": _shares["ko_punct"],
     "기타기호": _shares["ko_symbol"], "공백": _shares["ko_ws"],
     "합": REP_STATS["composition_closure_ko"]},
    {"언어": "영어", "한글": _shares["en_hangul"], "라틴": _shares["en_latin"],
     "숫자": _shares["en_digit"], "문장부호": _shares["en_punct"],
     "기타기호": _shares["en_symbol"], "공백": _shares["en_ws"],
     "합": REP_STATS["composition_closure_en"]}]), floatfmt="{:,.6f}")
print(f"\nbyte/code point 중앙값: 한국어 {REP_STATS['ko_bytes_per_codepoint']['quantiles']['0.5']:.4f} · "
      f"영어 {REP_STATS['en_bytes_per_codepoint']['quantiles']['0.5']:.4f}")
print(f"token/byte 중앙값:      한국어 {REP_STATS['ko_tokens_per_byte']['quantiles']['0.5']:.6f} · "
      f"영어 {REP_STATS['en_tokens_per_byte']['quantiles']['0.5']:.6f}")
print(f"\n표현 역전(representation reversal): log CR < 0 이면서 log TP > 0 인 문장쌍 "
      f"{_rev['reversal']:,} ({REP_STATS['representation_reversal_share'] * 100:.4f}%)")
print(f"  log CR < 0 전체 {_rev['cr_neg']:,} · 그중 역전 아님 {_rev['cr_neg_no_rev']:,}")
print(f"  log BDR > 0 {_rev['bdr_pos']:,} · log CP > 0 {_rev['cp_pos']:,}")
print(f"Spearman ρ(log BDR, log CP) = {REP_STATS['spearman_lbdr_lcp']:.6f}")

평균 문자군 구성 (문자 비율 + 공백 밀도, 각 언어별 합 = 1):
 언어       한글       라틴       숫자     문장부호     기타기호       공백        합
한국어 0.706052 0.013370 0.014927 0.050962 0.006203 0.208487 1.000000
 영어 0.000003 0.796994 0.006749 0.032924 0.003061 0.160269 1.000000

byte/code point 중앙값: 한국어 2.4500 · 영어 1.0000
token/byte 중앙값:      한국어 0.245614 · 영어 0.208333

표현 역전(representation reversal): log CR < 0 이면서 log TP > 0 인 문장쌍 3,363,717 (87.6884%)
  log CR < 0 전체 3,823,296 · 그중 역전 아님 459,579
  log BDR > 0 3,834,657 · log CP > 0 3,089,350
Spearman ρ(log BDR, log CP) = -0.050652


In [18]:
fig, axes = plt.subplots(1, 2, figsize=(14.4, 5.6))
_x = _rep["log_byte_density_ratio"].to_numpy(np.float64)
_y = _rep["log_compression_penalty"].to_numpy(np.float64)
density_panel(axes[0], _x, _y, bins=240,
              xlim=(float(np.quantile(_x, 0.0005)), float(np.quantile(_x, 0.9995))),
              ylim=(float(np.quantile(_y, 0.0005)), float(np.quantile(_y, 0.9995))),
              xlabel="log ByteDensityRatio — UTF-8 표현 부담",
              ylabel="log CompressionPenalty — 고정 tokenizer 압축 열위",
              title="(a) UTF-8 표현 부담 대 tokenizer 압축 penalty\n"
                    f"Spearman ρ = {REP_STATS['spearman_lbdr_lcp']:.4f} (기술적 연관, 인과 아님)")
axes[0].axhline(0.0, color="#0b0b0b", lw=1.0, ls=":")
axes[0].axvline(0.0, color="#0b0b0b", lw=1.0, ls=":")
binned_median(axes[0], _x, _y, bins=45, color="#0f8a5f")

_bins = np.linspace(0, 1.0, 260)
axes[1].hist(_rep["ko_tokens_per_byte"], bins=_bins, histtype="step", lw=1.8,
             color="#c1272d", label="한국어 token/byte")
axes[1].hist(_rep["en_tokens_per_byte"], bins=_bins, histtype="step", lw=1.8,
             color="#0f4c81", label="영어 token/byte")
axes[1].set_xlabel("token 수 / UTF-8 byte 수 (언어별 압축 효율)")
axes[1].set_ylabel("문장쌍 수"); axes[1].set_xlim(0, 0.75)
axes[1].set_title("(b) 언어별 token-per-byte — 같은 byte를 얼마나 잘 압축하는가")
axes[1].legend()
fig.suptitle(f"F05 · UTF-8 표현 부담과 tokenizer 압축 penalty (N = {COHORT['N']:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "F05_byte_density_vs_compression_v001", "UTF-8 표현 부담 대 압축 penalty",
         rq="RQ3", contract="SSOT_§16.2_F05 — ByteDensityRatio vs CompressionPenalty",
         note="IR-01에 따라 두 성분을 분리해 제시")

  saved F05_byte_density_vs_compression_v001  png 267 KiB  sha 27b6ef435ebc…  한글=True


{'figure_id': 'F05_byte_density_vs_compression_v001',
 'title_ko': 'UTF-8 표현 부담 대 압축 penalty',
 'rq': 'RQ3',
 'contract': 'SSOT_§16.2_F05 — ByteDensityRatio vs CompressionPenalty',
 'tag': 'CANONICAL',
 'note': 'IR-01에 따라 두 성분을 분리해 제시',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/F05_byte_density_vs_compression_v001.png',
 'svg': 'outputs/figures/nb07/F05_byte_density_vs_compression_v001.svg',
 'png_sha256': '27b6ef435ebc57a021f82adc90aba7fe08ccee6f6e85f3aabb9254377c6a9597',
 'svg_sha256': 'a8d7c66313685c5fc25c3b5cbc0fa8c976209d5edb2737e9c504fd8f79301acd',
 'png_size_bytes': 273864,
 'svg_size_bytes': 198428,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

In [19]:
fig, axes = plt.subplots(2, 3, figsize=(16.0, 9.0))

_lbl = ["한글", "라틴", "숫자", "문장부호", "기타기호", "공백"]
_kov = [_shares[k] for k in ("ko_hangul", "ko_latin", "ko_digit", "ko_punct", "ko_symbol", "ko_ws")]
_env = [_shares[k] for k in ("en_hangul", "en_latin", "en_digit", "en_punct", "en_symbol", "en_ws")]
_c = ["#c1272d", "#0f4c81", "#e8a33d", "#7f9c96", "#8d6a9f", "#b9c6cc"]
_left = 0.0
for _i, (_l, _v, _cc) in enumerate(zip(_lbl, _kov, _c, strict=True)):
    axes[0, 0].barh(["한국어"], [_v], left=_left, color=_cc, label=_l); _left += _v
_left = 0.0
for _l, _v, _cc in zip(_lbl, _env, _c, strict=True):
    axes[0, 0].barh(["영어"], [_v], left=_left, color=_cc); _left += _v
axes[0, 0].set_xlim(0, 1); axes[0, 0].set_xlabel("평균 구성비 (합 = 1)")
axes[0, 0].set_title("(a) 평균 문자군 구성 — 언어별")
axes[0, 0].legend(ncol=3, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, -0.42))

axes[0, 1].hist(_rep["ko_bytes_per_codepoint"], bins=200, range=(1, 3.2), histtype="step",
                lw=1.8, color="#c1272d", label="한국어")
axes[0, 1].hist(_rep["en_bytes_per_codepoint"], bins=200, range=(1, 3.2), histtype="step",
                lw=1.8, color="#0f4c81", label="영어")
axes[0, 1].set_yscale("log")
axes[0, 1].set_xlabel("code point 당 UTF-8 byte 수"); axes[0, 1].set_ylabel("문장쌍 수 (로그)")
axes[0, 1].set_title("(b) UTF-8 byte 밀도 — 한글 3 byte 대 ASCII 1 byte")
axes[0, 1].legend()

axes[0, 2].hist(_rep["ko_whitespace_density"], bins=200, range=(0, 0.5), histtype="step",
                lw=1.8, color="#c1272d", label="한국어")
axes[0, 2].hist(_rep["en_whitespace_density"], bins=200, range=(0, 0.5), histtype="step",
                lw=1.8, color="#0f4c81", label="영어")
axes[0, 2].set_xlabel("공백 밀도 (공백 문자 수 / code point 수)"); axes[0, 2].set_ylabel("문장쌍 수")
axes[0, 2].set_title("(c) 공백 밀도 — 영어의 어절 분리 부담이 더 크다")
axes[0, 2].legend()

_lcr_a = _rep["log_code_point_ratio"].to_numpy(np.float64)
_ltp_a = _rep["log_token_premium"].to_numpy(np.float64)
density_panel(axes[1, 0], _lcr_a, _ltp_a, bins=220,
              xlim=(float(np.quantile(_lcr_a, 0.0005)), float(np.quantile(_lcr_a, 0.9995))),
              ylim=(float(np.quantile(_ltp_a, 0.0005)), float(np.quantile(_ltp_a, 0.9995))),
              xlabel="log CodePointRatio (문자 수 비)", ylabel="log TokenizationPremium",
              title="(d) 표현 역전 사분면\n문자 수는 적은데 token 수는 많다")
axes[1, 0].axhline(0.0, color="#0b0b0b", lw=1.0, ls="--")
axes[1, 0].axvline(0.0, color="#0b0b0b", lw=1.0, ls="--")
axes[1, 0].text(0.03, 0.96, f"역전 영역\n{_rev['reversal']:,}쌍 "
                            f"({REP_STATS['representation_reversal_share'] * 100:.2f}%)",
                transform=axes[1, 0].transAxes, va="top", fontsize=9,
                bbox={"facecolor": "white", "alpha": 0.85, "edgecolor": "#c1272d"})

_sc = con.execute(f"""SELECT ko_script_type_count t, count(*) n FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
_se = con.execute(f"""SELECT en_script_type_count t, count(*) n FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
_w = 0.38
axes[1, 1].bar(_sc["t"] - _w / 2, _sc["n"], width=_w, color="#c1272d", label="한국어")
axes[1, 1].bar(_se["t"] + _w / 2, _se["n"], width=_w, color="#0f4c81", label="영어")
axes[1, 1].set_yscale("log")
axes[1, 1].set_xlabel("script 종류 수 (script_type_count)"); axes[1, 1].set_ylabel("문장쌍 수 (로그)")
axes[1, 1].set_title("(e) script 혼합 — 종류 수 분포")
axes[1, 1].set_xticks(sorted(set(_sc["t"]) | set(_se["t"])))
axes[1, 1].legend()

_sw = con.execute(f"""SELECT least(ko_script_switch_count, 10) s, count(*) n FROM {A}
                      GROUP BY 1 ORDER BY 1""").fetchdf()
_swe = con.execute(f"""SELECT least(en_script_switch_count, 10) s, count(*) n FROM {A}
                       GROUP BY 1 ORDER BY 1""").fetchdf()
axes[1, 2].bar(_sw["s"] - _w / 2, _sw["n"], width=_w, color="#c1272d", label="한국어")
axes[1, 2].bar(_swe["s"] + _w / 2, _swe["n"], width=_w, color="#0f4c81", label="영어")
axes[1, 2].set_yscale("log")
axes[1, 2].set_xlabel("script 전환 횟수 (10 이상은 10으로 절단 표시)")
axes[1, 2].set_ylabel("문장쌍 수 (로그)")
axes[1, 2].set_title("(f) script 혼합 — 전환 횟수 분포")
axes[1, 2].legend()

fig.suptitle(f"NB07-S03 · 표현(surface-form) 구조 기술 (N = {COHORT['N']:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "NB07-S03_representation_structure_v001", "표현 구조 기술",
         rq="RQ3", contract="SUPPORTING_DESCRIPTIVE",
         note="문자군 구성 · byte 밀도 · 공백 · 표현 역전 · script mixing")
del _rep, _dec

  saved NB07-S03_representation_structure_v001  png 324 KiB  sha 8b7b1750da9f…  한글=True


### 수치 해석

평균 문자군 구성은 한국어 쪽이 한글 0.7061 · 라틴 0.0134 · 숫자 0.0149 · 문장부호 0.0510 ·
기타기호 0.0062 · 공백 0.2085 이고, 영어 쪽은 라틴 0.7970 · 숫자 0.0067 · 문장부호 0.0329 ·
기타기호 0.0031 · 공백 0.1603 · 한글 0.000003 이다. 각 언어에서 여섯 항의 합이 정확히 1이다 —
G5 §4.3이 확인한 구성적 폐쇄(compositional closure)가 이 기술 수준에서도 재현된다.

code point 당 byte 수 중앙값은 한국어 2.4500, 영어 1.0000이다.
token/byte 중앙값은 한국어 0.245614, 영어 0.208333으로 **한국어 쪽이 더 높다** —
같은 byte 수에서 한국어가 더 많은 token으로 쪼개진다는 뜻이며, log CP 중앙값이 양수인 것과
같은 사실의 다른 표현이다.

**표현 역전**: log CR < 0 (한국어 문자 수가 더 적음)이면서 동시에 log TP > 0
(한국어 token 수가 더 많음)인 문장쌍이 3,363,717개, 전체의 **87.6884%** 다.

### 분포 해석

(a)–(c)가 그 역전이 어떤 표현 구조 위에서 일어나는지 보여준다. 한국어는 문자 수가 적지만
그 문자 대부분이 UTF-8에서 3 byte를 쓰는 한글이고, 영어는 문자 수가 많지만 1 byte 라틴
문자와 공백이다. (b)의 두 최빈 구간(한국어 ≈ 2.50, 영어 = 1.00 — 영어는 3,821,117쌍,
99.61%가 정확히 1.0000)이 log BDR 분포가 어디에 놓이는지와 정합적이다.
이는 두 주변분포와 그 비(log BDR)의 산술적 관계에 대한 기술이며, 어느 한쪽이 다른 쪽을
결정한다는 진술이 아니다.

(a)의 F05가 이 절에서 가장 중요한 관찰이다. log BDR과 log CP의 전수 Spearman ρ는
**−0.0507** — 사실상 **연관이 없다**(부호는 미약하게 음). 구간별 중앙값 곡선도 거의 평평하다.
두 축은 서로 다른 것을 측정한다: 하나는 문자를 byte로 옮기는 **인코딩 구조**,
다른 하나는 그 byte를 subword로 묶는 **tokenizer의 압축 성능**이다.
자료가 그 독립성을 직접 보여준다 — UTF‑8 표현 부담이 큰 문장쌍이라고 해서 tokenizer가
그만큼 더 불리하게 압축하지는 않는다. IR‑01이 두 사실을 분리해 보고하라고 요구하는 이유가
여기서 경험적으로 확인된다. log BDR의 산포가 매우 작다는 점(표준편차 0.066)도 이 낮은
상관에 기여한다.

script 혼합은 양쪽 모두 단일 script(종류 수 1)가 지배적이다: 한국어 2,979,730쌍(77.68%),
영어 3,569,117쌍(93.04%). 전환 횟수도 대부분 0이다. 이 희소성은 §15의 SM-01 검토에서
핵심 논점이 된다.

### 연구적 의미

RQ3의 표면형 구조가 기술 수준에서 정리되었다. 특히 표현 역전이 예외가 아니라 **다수 사례**
(87.69%)라는 점은, premium을 "한국어가 길어서"로 설명하려는 직관을 자료가 직접 반박함을
보여준다.

### 해석 한계

여기의 어떤 연관도 조건부가 아니다. Spearman ρ는 두 변수만 본 주변적 순위 연관이며,
길이·도메인·출처를 통제하지 않았다. "log BDR이 log CP를 높인다"고 읽을 수 없다.
RQ3의 조건부 연관은 **NB09 M1 vs M0**에서만 추정된다.

**Reference**: `[EDA-REF-A06]` (script_type_count = 0), `[EDA-REF-A07]` · `[EDA-REF-A08]`
(문자군 이상), `[EDA-REF-A09]` (log BDR < 0) — §21

## 13 — F. 형태소 분포 (분포만)

### SSOT / RQ 대응

**Research Question**: RQ4 — 형태소 밀도, 조사 비율, 어미 비율, 접사 비율이 길이 · byte ·
공백 · 문자군 · 도메인 · 출처를 통제한 후에도 log(TP) 또는 log(CompressionPenalty)의
추가 설명력을 제공하는가?

**SSOT section**: `KOEN-TP-RS-001` §6.4 (주 판단은 개별 p‑value보다 형태소 feature block의
증분 설명력에 둔다), §5 (분석 계층 구분)

**Canonical input**: D-03 `MORPH_FEATURES_KIWI_v001`

**Physical variables**: `morpheme_density`, `particle_ratio`, `ending_ratio`,
`deriv_affix_ratio`, `function_morpheme_ratio`, `eojeol_count`, `morpheme_count`

**Research purpose**: 형태소 feature block의 **주변 분포만** 제시한다. 이 절은 RQ4에
답하지 않는다 — 증분 설명력은 정의상 통제 후에만 정의되며, 그것은 NB09 M2 vs M1의 몫이다.

**Allowed claim**: 각 형태소 지표의 분포·분위수·경계값 비율, 길이 층별 분포 이동.

**Prohibited claim**: 형태소가 premium에 기여한다/기여하지 않는다는 어떤 진술.
log TP와의 이변량 연관조차 여기서는 제시하지 않는다 — 통제 없는 연관은 RQ4의 질문이 아니며
오독을 부른다.

In [20]:
_morph_cols = ["morpheme_density", "particle_ratio", "ending_ratio",
               "deriv_affix_ratio", "function_morpheme_ratio"]
MORPH_STATS = {c: qsummary(c, c) for c in _morph_cols}
MORPH_STATS["morph_eojeol_count"] = qsummary("morph_eojeol_count", "morph_eojeol_count")
MORPH_STATS["morpheme_count"] = qsummary("morpheme_count", "morpheme_count")
MORPH_STATS["boundary_shares"] = {
    "particle_ratio_eq_0": int(con.execute(f"SELECT count(*) FROM {A} WHERE particle_ratio = 0").fetchone()[0]),
    "ending_ratio_eq_0": int(con.execute(f"SELECT count(*) FROM {A} WHERE ending_ratio = 0").fetchone()[0]),
    "deriv_affix_ratio_eq_0": int(con.execute(f"SELECT count(*) FROM {A} WHERE deriv_affix_ratio = 0").fetchone()[0]),
    "function_morpheme_ratio_eq_0": int(con.execute(f"SELECT count(*) FROM {A} WHERE function_morpheme_ratio = 0").fetchone()[0]),
    "morph_eojeol_count_eq_1": int(con.execute(f"SELECT count(*) FROM {A} WHERE morph_eojeol_count = 1").fetchone()[0]),
}
_by_len = con.execute(f"""SELECT length_stratum,
    quantile_cont(morpheme_density, 0.5) AS morpheme_density_p50,
    quantile_cont(particle_ratio, 0.5) AS particle_ratio_p50,
    quantile_cont(ending_ratio, 0.5) AS ending_ratio_p50,
    quantile_cont(deriv_affix_ratio, 0.5) AS deriv_affix_ratio_p50,
    quantile_cont(function_morpheme_ratio, 0.5) AS function_morpheme_ratio_p50,
    count(*) AS n
  FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
MORPH_STATS["by_length_stratum_median"] = _by_len.to_dict(orient="records")
SUMMARY["morphology"] = MORPH_STATS

show(pd.DataFrame([{"지표": c, "평균": MORPH_STATS[c]["mean"], "표준편차": MORPH_STATS[c]["sd"],
                    "5%": MORPH_STATS[c]["quantiles"]["0.05"],
                    "중앙값": MORPH_STATS[c]["quantiles"]["0.5"],
                    "95%": MORPH_STATS[c]["quantiles"]["0.95"],
                    "최대": MORPH_STATS[c]["quantiles"]["1.0"]} for c in _morph_cols]))
print("\n경계값 (0 또는 1) 관측 수:")
for _k, _v in MORPH_STATS["boundary_shares"].items():
    print(f"  {_k:32s} {_v:>9,}  ({_v / COHORT['N'] * 100:7.4f}%)")
print("\n길이 층별 중앙값:")
show(_by_len)

                     지표        평균      표준편차        5%       중앙값      95%       최대
       morpheme_density   2.28848  0.440937   1.77273   2.21429        3       30
         particle_ratio  0.152412 0.0677563         0  0.157895     0.25      0.6
           ending_ratio   0.18298 0.0748722 0.0769231  0.175439 0.318182 0.666667
      deriv_affix_ratio 0.0638881 0.0513034         0 0.0652174 0.152174      0.5
function_morpheme_ratio  0.335392 0.0802181       0.2  0.342105     0.45 0.666667

경계값 (0 또는 1) 관측 수:
  particle_ratio_eq_0                292,186  ( 7.6170%)
  ending_ratio_eq_0                   65,990  ( 1.7203%)
  deriv_affix_ratio_eq_0           1,042,162  (27.1680%)
  function_morpheme_ratio_eq_0        39,226  ( 1.0226%)
  morph_eojeol_count_eq_1             42,096  ( 1.0974%)

길이 층별 중앙값:
length_stratum  morpheme_density_p50  particle_ratio_p50  ending_ratio_p50  deriv_affix_ratio_p50  function_morpheme_ratio_p50      n
            Q1               2.33333            0.111111 

In [21]:
_mo = fetch(", ".join(_morph_cols + ["length_stratum", "morph_eojeol_count"]))
fig, axes = plt.subplots(2, 3, figsize=(16.0, 8.6))
_specs = [
    ("morpheme_density", "형태소 밀도 (어절당 형태소 수)", (0, 8), "#c1272d"),
    ("particle_ratio", "조사 비율 (particle / 형태소)", (0, 0.6), "#0f4c81"),
    ("ending_ratio", "어미 비율 (ending / 형태소)", (0, 0.7), "#e8a33d"),
    ("deriv_affix_ratio", "파생접사 비율 (deriv affix / 형태소)", (0, 0.5), "#7f9c96"),
    ("function_morpheme_ratio", "기능형태소 비율 (function / 형태소)", (0, 0.7), "#8d6a9f"),
]
for _ax, (_c, _t, _rng, _col) in zip(axes.flat[:5], _specs, strict=True):
    _ax.hist(_mo[_c], bins=200, range=_rng, color=_col, edgecolor="none")
    _ax.axvline(MORPH_STATS[_c]["quantiles"]["0.5"], color="#0b0b0b", lw=1.4, ls="--",
                label=f"중앙값 {MORPH_STATS[_c]['quantiles']['0.5']:.4f}")
    _ax.set_xlabel(_t); _ax.set_ylabel("문장쌍 수"); _ax.set_title(_t)
    _ax.legend(fontsize=8)

_ax = axes.flat[5]
_order = sorted(_mo["length_stratum"].unique())
_ax.boxplot([_mo.loc[_mo["length_stratum"] == s, "morpheme_density"].to_numpy() for s in _order],
            tick_labels=_order, showfliers=False, whis=(5, 95),
            medianprops={"color": "#c1272d", "lw": 1.8},
            boxprops={"color": "#16425b"}, whiskerprops={"color": "#16425b"})
_ax.set_xlabel("길이 층 (length_stratum)"); _ax.set_ylabel("형태소 밀도")
_ax.set_title("(f) 길이 층별 형태소 밀도 — 기술적 분포 이동\n(수염 = 5–95 분위, 이상치 표시 생략)")

fig.suptitle(f"NB07-S04 · 형태소 feature block 주변 분포 (D-03 Kiwi, N = {COHORT['N']:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "NB07-S04_morphology_distributions_v001", "형태소 분포",
         rq="RQ4", contract="SUPPORTING_DESCRIPTIVE",
         note="주변 분포만 — log TP와의 연관은 제시하지 않는다 (NB09 M2 vs M1)")
del _mo

  saved NB07-S04_morphology_distributions_v001  png 193 KiB  sha d6fc1d1187f4…  한글=True


### 수치 해석

형태소 밀도 중앙값은 2.21429 (어절당 형태소 약 2.2개), 5–95 분위 [1.77273, 3.0], 최대 30.0이다.
조사 비율 중앙값 0.157895, 어미 비율 0.175439, 파생접사 비율 0.065217,
기능형태소 비율 0.342105다.

경계값 관측이 상당하다: 파생접사 비율 = 0 인 문장쌍이 1,042,162개(27.17%),
조사 비율 = 0 이 292,186개(7.62%), 어미 비율 = 0 이 65,990개(1.72%),
기능형태소 비율 = 0 이 39,226개(1.02%), 어절 수 = 1 이 42,096개(1.10%)다.

### 분포 해석

다섯 지표 모두 0 근처에 유한한 질량을 갖는 유계(bounded) 비율 변수다.
조사·어미·파생접사 비율은 형태소 수가 작은 짧은 문장에서 이산적인 값(0, 1/2, 1/3 …)을
취하므로 히스토그램에 빗살 구조가 보인다 — 이는 측정 결함이 아니라 비율 변수의 격자 성질이다.
기능형태소 비율은 세 지표보다 매끄럽고 중앙에 모여 있는데, 여러 형태소 부류를 합산하기 때문이다.

(f)는 길이 층이 올라갈수록 형태소 밀도 중앙값이 완만하게 **하강**함을 보인다
(Q1 2.3333 → Q2 2.2500 → Q3 2.1818 → Q4 2.1667 → Q5 2.1818).
반대로 조사 비율 중앙값은 상승하고(0.1111 → 0.1739) 어미 비율은 하강한다(0.2000 → 0.1404).
이는 형태소 지표가 길이와 **기술적으로 얽혀 있음**을 뜻하며, RQ4가 "길이를 통제한 후"의
증분 설명력을 묻는 이유를 그대로 보여준다.

### 연구적 의미

형태소 block이 NB09에서 어떤 지지 위에서 추정될지를 미리 보여준다.
특히 파생접사 비율의 27.17% 0-질량, 조사 비율의 7.62% 0-질량, 그리고 길이–밀도 얽힘은
M2 해석 시 반드시 함께 언급되어야 한다.

### 해석 한계

**이 절은 RQ4에 답하지 않는다.** 형태소 지표와 log TP 사이의 어떤 연관도 여기서 제시되지
않았고, 제시되었더라도 통제 없는 주변 연관은 RQ4의 질문이 아니다.
형태소 밀도가 높은 문장이 premium이 크다/작다는 진술은 이 노트북 어디에서도 성립하지 않는다.
형태소 분석은 Kiwi 특정 버전의 산출이며 언어학적 정답이 아니다(SSOT §5).

**Reference**: `[EDA-REF-A05]` (어절 수 = 1), `[EDA-REF-A12]` (조사 비율 = 0),
`[EDA-REF-A13]` (형태소 밀도 극단) — §21

## 14 — G. D-05 regex chunk mechanism 기술자

### SSOT / RQ 대응

**Research Question**: RQ5 — 고정된 o200k_base 구현에서 관측되는 regex chunk count,
chunk 길이, token‑per‑chunk, token‑per‑byte 등의 mechanism feature가 최종 premium과
어떻게 연결되는가?

**SSOT section**: `KOEN-TP-RS-001` §6.5, Method Note MN‑02 (post-treatment 성격 —
tokenizer-derived chunk feature는 최종 token 수의 기계적 중간 단계에 가깝다),
Reproducibility Rule RR‑01 (일반명 "pre-tokenization" 대신 `o200k_base regex chunking`)

**Canonical input**: D-05 `CHUNK_O200K_BASE_v001`

**Physical variables**: `ko/en_chunk_count`, `ko/en_mean_chunk_bytes`,
`ko/en_p50_chunk_bytes`, `ko/en_p90_chunk_bytes`, `ko/en_max_chunk_bytes`,
`ko/en_tokens_per_chunk`, `ko/en_max_tokens_per_chunk`, `ko/en_chunk_type_share_*`

**Research purpose**: mechanism 기술자의 분포를 **기술 목적으로만** 제시한다.

**Allowed claim**: chunk 규모·길이·유형 구성의 분포에 대한 기술적 진술.

**Prohibited claim**: regex chunking이 fragmentation을 "설명한다"는 진술.
MN‑02에 따라 이 feature들은 mechanism audit로 분리되며, 조건부 기여는 NB09 M3 vs M2의 몫이다.
`chunk_token_total`·`tokens_per_chunk × chunk_count` 은 G5가 outcome leakage로 판정했으므로
결과변수와 나란히 놓지 않는다.

In [22]:
_chunk_cols = ["ko_chunk_count", "en_chunk_count", "ko_mean_chunk_bytes", "en_mean_chunk_bytes",
               "ko_p50_chunk_bytes", "en_p50_chunk_bytes", "ko_p90_chunk_bytes", "en_p90_chunk_bytes",
               "ko_max_chunk_bytes", "en_max_chunk_bytes",
               "ko_tokens_per_chunk", "en_tokens_per_chunk",
               "ko_max_tokens_per_chunk", "en_max_tokens_per_chunk"]
CHUNK_STATS = {c: qsummary(c, c) for c in _chunk_cols}
_ctype = con.execute(f"""SELECT
  avg(ko_chunk_type_share_letter) ko_letter, avg(ko_chunk_type_share_number) ko_number,
  avg(ko_chunk_type_share_punctuation) ko_punct, avg(ko_chunk_type_share_whitespace) ko_ws,
  avg(en_chunk_type_share_letter) en_letter, avg(en_chunk_type_share_number) en_number,
  avg(en_chunk_type_share_punctuation) en_punct, avg(en_chunk_type_share_whitespace) en_ws
  FROM {A}""").fetchdf().iloc[0].to_dict()
CHUNK_STATS["mean_chunk_type_share"] = {k: float(v) for k, v in _ctype.items()}
CHUNK_STATS["chunk_type_closure_ko"] = float(sum(_ctype[k] for k in ("ko_letter", "ko_number", "ko_punct", "ko_ws")))
CHUNK_STATS["chunk_type_closure_en"] = float(sum(_ctype[k] for k in ("en_letter", "en_number", "en_punct", "en_ws")))
CHUNK_STATS["pair_chunk_ratio"] = qsummary("pair_chunk_ratio", "pair_chunk_ratio")
SUMMARY["chunk_mechanism"] = CHUNK_STATS

show(pd.DataFrame([{"기술자": c, "평균": CHUNK_STATS[c]["mean"],
                    "중앙값": CHUNK_STATS[c]["quantiles"]["0.5"],
                    "5%": CHUNK_STATS[c]["quantiles"]["0.05"],
                    "95%": CHUNK_STATS[c]["quantiles"]["0.95"],
                    "최대": CHUNK_STATS[c]["quantiles"]["1.0"]} for c in _chunk_cols]))
print("\n평균 chunk 유형 구성 (각 언어별 합 = 1):")
show(pd.DataFrame([
    {"언어": "한국어", "문자": _ctype["ko_letter"], "숫자": _ctype["ko_number"],
     "문장부호": _ctype["ko_punct"], "공백": _ctype["ko_ws"], "합": CHUNK_STATS["chunk_type_closure_ko"]},
    {"언어": "영어", "문자": _ctype["en_letter"], "숫자": _ctype["en_number"],
     "문장부호": _ctype["en_punct"], "공백": _ctype["en_ws"], "합": CHUNK_STATS["chunk_type_closure_en"]}]),
     floatfmt="{:,.6f}")

                    기술자      평균     중앙값      5%     95%    최대
         ko_chunk_count 13.5611      11       4      31   189
         en_chunk_count 19.4989      15       6      44   423
    ko_mean_chunk_bytes 8.12401 8.27273 5.30303 10.4706    78
    en_mean_chunk_bytes 4.93956  4.9375     3.5 6.34286 114.5
     ko_p50_chunk_bytes 8.23046     8.5       4      11    78
     en_p50_chunk_bytes 4.56411     4.5       3       6 114.5
     ko_p90_chunk_bytes  13.962      13      10      19    96
     en_p90_chunk_bytes 8.81156       9       6      12   228
     ko_max_chunk_bytes 15.6557      16      10      22   160
     en_max_chunk_bytes 10.4855      11       6      15   228
    ko_tokens_per_chunk 2.01862       2 1.48649 2.66667    27
    en_tokens_per_chunk 1.03649       1       1 1.16667  29.5
ko_max_tokens_per_chunk 3.98277       4       2       6    54
en_max_tokens_per_chunk 1.43506       1       1       3    58

평균 chunk 유형 구성 (각 언어별 합 = 1):
 언어       문자       숫자     문장부호       공백

In [23]:
_ck = fetch("""ko_chunk_count, en_chunk_count, ko_mean_chunk_bytes, en_mean_chunk_bytes,
               ko_tokens_per_chunk, en_tokens_per_chunk, ko_max_chunk_bytes, en_max_chunk_bytes,
               ko_p90_chunk_bytes, en_p90_chunk_bytes""")
fig, axes = plt.subplots(2, 3, figsize=(16.0, 8.6))

_bins = np.logspace(0, np.log10(max(_ck["ko_chunk_count"].max(), _ck["en_chunk_count"].max())), 80)
axes[0, 0].hist(_ck["ko_chunk_count"], bins=_bins, histtype="step", lw=1.8, color="#c1272d", label="한국어")
axes[0, 0].hist(_ck["en_chunk_count"], bins=_bins, histtype="step", lw=1.8, color="#0f4c81", label="영어")
axes[0, 0].set_xscale("log"); axes[0, 0].set_yscale("log")
axes[0, 0].set_xlabel("regex chunk 수 (로그)"); axes[0, 0].set_ylabel("문장쌍 수 (로그)")
axes[0, 0].set_title("(a) o200k_base regex chunk 수"); axes[0, 0].legend()

axes[0, 1].hist(_ck["ko_mean_chunk_bytes"], bins=200, range=(0, 30), histtype="step", lw=1.8,
                color="#c1272d", label="한국어")
axes[0, 1].hist(_ck["en_mean_chunk_bytes"], bins=200, range=(0, 30), histtype="step", lw=1.8,
                color="#0f4c81", label="영어")
axes[0, 1].set_xlabel("chunk 평균 byte 수"); axes[0, 1].set_ylabel("문장쌍 수")
axes[0, 1].set_title("(b) chunk 평균 길이 (byte)"); axes[0, 1].legend()

axes[0, 2].hist(_ck["ko_tokens_per_chunk"], bins=200, range=(0, 8), histtype="step", lw=1.8,
                color="#c1272d", label="한국어")
axes[0, 2].hist(_ck["en_tokens_per_chunk"], bins=200, range=(0, 8), histtype="step", lw=1.8,
                color="#0f4c81", label="영어")
axes[0, 2].set_yscale("log")
axes[0, 2].set_xlabel("chunk 당 token 수 (tokens_per_chunk)"); axes[0, 2].set_ylabel("문장쌍 수 (로그)")
axes[0, 2].set_title("(c) chunk 하나가 몇 개 token으로 쪼개지는가"); axes[0, 2].legend()

_lbl = ["문자", "숫자", "문장부호", "공백"]
_c = ["#c1272d", "#e8a33d", "#7f9c96", "#b9c6cc"]
_left = 0.0
for _l, _k, _cc in zip(_lbl, ("ko_letter", "ko_number", "ko_punct", "ko_ws"), _c, strict=True):
    axes[1, 0].barh(["한국어"], [_ctype[_k]], left=_left, color=_cc, label=_l); _left += _ctype[_k]
_left = 0.0
for _l, _k, _cc in zip(_lbl, ("en_letter", "en_number", "en_punct", "en_ws"), _c, strict=True):
    axes[1, 0].barh(["영어"], [_ctype[_k]], left=_left, color=_cc); _left += _ctype[_k]
axes[1, 0].set_xlim(0, 1); axes[1, 0].set_xlabel("평균 구성비 (합 = 1)")
axes[1, 0].set_title("(d) chunk 유형 구성 — 평균")
axes[1, 0].legend(ncol=4, fontsize=8, loc="lower center", bbox_to_anchor=(0.5, -0.38))

for _ax, _side, _c2, _nm in ((axes[1, 1], "ko", "#c1272d", "한국어"), (axes[1, 1], "en", "#0f4c81", "영어")):
    _ax.hist(_ck[f"{_side}_p90_chunk_bytes"], bins=150, range=(0, 60), histtype="step", lw=1.8,
             color=_c2, label=f"{_nm} p90")
axes[1, 1].set_yscale("log")
axes[1, 1].set_xlabel("chunk byte 길이의 90 분위"); axes[1, 1].set_ylabel("문장쌍 수 (로그)")
axes[1, 1].set_title("(e) 긴 chunk 쪽 꼬리 (p90)"); axes[1, 1].legend()

density_panel(axes[1, 2], np.log(_ck["ko_chunk_count"].to_numpy(np.float64)),
              np.log(_ck["en_chunk_count"].to_numpy(np.float64)), bins=180,
              xlabel="log 한국어 chunk 수", ylabel="log 영어 chunk 수",
              title="(f) 두 언어 chunk 규모의 결합분포")
_m = max(np.log(_ck["ko_chunk_count"]).max(), np.log(_ck["en_chunk_count"]).max())
axes[1, 2].plot([0, _m], [0, _m], color="#0b0b0b", lw=1.2, ls="--", label="동일선")
axes[1, 2].legend(loc="upper left")

fig.suptitle(f"NB07-S05 · o200k_base regex chunk mechanism 기술자 (D-05, N = {COHORT['N']:,})",
             fontsize=13)
fig.tight_layout()
save_fig(fig, "NB07-S05_chunk_mechanism_v001", "regex chunk mechanism 기술자",
         rq="RQ5", contract="SUPPORTING_DESCRIPTIVE",
         note="MN-02에 따라 mechanism audit 범주로 분리 제시")

  saved NB07-S05_chunk_mechanism_v001  png 231 KiB  sha 9b048c415c30…  한글=True


{'figure_id': 'NB07-S05_chunk_mechanism_v001',
 'title_ko': 'regex chunk mechanism 기술자',
 'rq': 'RQ5',
 'contract': 'SUPPORTING_DESCRIPTIVE',
 'tag': 'CANONICAL',
 'note': 'MN-02에 따라 mechanism audit 범주로 분리 제시',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/NB07-S05_chunk_mechanism_v001.png',
 'svg': 'outputs/figures/nb07/NB07-S05_chunk_mechanism_v001.svg',
 'png_sha256': '9b048c415c309c7ea920e374ff88d1e53c9797b8d66527cfcbd5c2c6c9133a82',
 'svg_sha256': '8b2e2e02d7fe099fcf76318c804a27f9810ca9e98f2e207af92a29af96abe619',
 'png_size_bytes': 236038,
 'svg_size_bytes': 175187,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

### 수치 해석

regex chunk 수 중앙값은 한국어 11, 영어 15다(평균 13.56 대 19.50). 영어 쪽 chunk가 더 많다 —
o200k_base의 regex가 "선행 공백 + 단어"를 한 chunk로 묶으므로, 공백으로 분리된 단어가 많은
영어에서 chunk 개수가 더 많이 생긴다.
chunk 평균 byte 수 중앙값은 한국어 8.27, 영어 4.94로 반대 방향이다.
chunk 당 token 수 중앙값은 한국어 2.0, 영어 1.0이고, chunk 최대 token 수 중앙값은
한국어 4, 영어 1이다.

chunk 유형 구성 평균은 한국어에서 문자 0.8168 · 문장부호 0.1477 · 숫자 0.0240 · 공백 0.0114,
영어에서 문자 0.8525 · 문장부호 0.1228 · 숫자 0.0149 · 공백 0.0098이며 각 합이 1이다.
공백만으로 이루어진 chunk가 1%대에 그치는 것은 공백이 뒤따르는 단어 chunk에 흡수되기 때문이다.

### 분포 해석

세 그림이 하나의 기계적 그림을 이룬다: 영어는 **chunk가 많고 짧으며 chunk 하나가 대체로
token 하나**가 된다(중앙값 1.0). 한국어는 **chunk가 적고 길며 chunk 하나가 평균 2개 이상의
token으로 쪼개진다**. 최종 token 수의 언어 간 차이는 chunk 개수 차이가 아니라
**chunk 내부 분할률의 차이**에서 나온다.

(f)에서 두 언어의 chunk 규모는 강하게 함께 움직이지만 대부분 동일선 아래에 있다
(영어 chunk가 더 많음). 이 결합 구조가 §15의 M3-01 검토 대상이다.

### 연구적 의미

RQ5의 기술 재료가 정리되었다. 특히 "chunk 당 token 수"의 언어 간 격차는 압축 penalty
(§11의 log CP)와 같은 현상을 다른 각도에서 본 것이다.

### 해석 한계

MN‑02에 따라 chunk feature는 **post-treatment** 성격을 가진다 — 최종 token 수의 기계적
중간 단계다. 따라서 이 절의 어떤 관찰도 "regex chunking이 fragmentation을 설명한다"로
읽을 수 없으며, M3 결과로 M2의 형태소 연관을 인과적으로 제거하거나 확정할 수 없다.
G5가 `ko/en_chunk_token_total == ko/en_token_count` 를 오차 0의 항등식으로 확인했으므로
(outcome leakage), 그 열들은 이 노트북의 어떤 그림에도 결과변수와 함께 등장하지 않는다.
RR‑01에 따라 이 절은 일반명 "pre-tokenization"이 아니라 `o200k_base regex chunking` 으로 기록한다.

**Reference**: `[EDA-REF-A10]` (chunk 수 = 1), `[EDA-REF-A14]` (극단적으로 긴 chunk) — §21

## 15 — G5 REVIEW `M3-01` · chunk 규모 대 절대 길이

> **DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW** — 이 절은 NB07의 추론적 결론이 아니다.
> G5가 등록한 검토 항목이 어떤 자료 구조에서 나왔는지 보이게 만드는 것이 전부다.

### SSOT / RQ 대응

**Research Question**: RQ5 — 고정된 o200k_base 구현에서 관측되는 regex chunk count,
chunk 길이, token‑per‑chunk, token‑per‑byte 등의 mechanism feature가 최종 premium과
어떻게 연결되는가?

**SSOT section**: `KOEN-TP-RS-001` §6.5, MN‑02; G5 adjudication §7.1 → REVIEW `M3-01`

**Canonical input**: D-02 + D-05

**Physical variables**: `pair_log_size`, `ko_chunk_count_log`, `en_chunk_count_log`,
그리고 (애드덤) `ko/en_chunk_count`, chunk 밀도 `chunk_count / codepoint_count`,
`chunk_count / utf8_bytes`, `chunk_count / eojeol_count`(KO) · `/ word_count`(EN)

**Research purpose**: G5는 M3에서 condition number 134.57, 최대 VIF 1,252.61이
관측되었고 그 원인이 chunk 규모와 절대 길이의 근접 비례라고 기록했다.
이 절은 그 관계를 **새로 계산해 시각화**하고, 애드덤이 요구한 재모수화 후보
(raw count 대 밀도 형태)의 **construct validity**를 기술적으로 확인한다.

**Allowed claim**: chunk 수는 tokenizer regex 수준의 기술자이고, 절대 텍스트/문장쌍 규모와
강하게 연관되어 있다는 기술적 진술. 그 연관이 G5 M3 재모수화 검토를 낳았다는 사실의 기록.

**Prohibited claim**: 인과적 중복성(causal redundancy)의 성립. NB09에서 어떤 변수를
삭제해도 된다는 허가. 어떤 재모수화가 더 낫다는 판단. NB09 matrix는 여기서 바뀌지 않는다.

**참조**: G5 REVIEW `M3-01`

In [24]:
_m3 = fetch("""pair_log_size, ko_chunk_count_log, en_chunk_count_log,
               ko_chunk_count, en_chunk_count, ko_codepoint_count, en_codepoint_count,
               ko_utf8_bytes, en_utf8_bytes, ko_eojeol_count, en_word_count, length_stratum""")
_m3["ko_chunk_per_codepoint"] = _m3["ko_chunk_count"] / _m3["ko_codepoint_count"]
_m3["en_chunk_per_codepoint"] = _m3["en_chunk_count"] / _m3["en_codepoint_count"]
_m3["ko_chunk_per_byte"] = _m3["ko_chunk_count"] / _m3["ko_utf8_bytes"]
_m3["en_chunk_per_byte"] = _m3["en_chunk_count"] / _m3["en_utf8_bytes"]
_m3["ko_chunk_per_eojeol"] = _m3["ko_chunk_count"] / _m3["ko_eojeol_count"]
_m3["en_chunk_per_word"] = _m3["en_chunk_count"] / _m3["en_word_count"].replace(0, np.nan)

_pls = _m3["pair_log_size"].to_numpy(np.float64)
M3 = {"note": "DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW — not a model-selection decision",
      "g5_reference": "REVIEW M3-01",
      "g5_reported": {"m3_condition_number_standardized": 134.57, "m3_max_vif": 1252.61,
                      "vif_pair_log_size": 1252.61, "vif_en_chunk_count_log": 854.04,
                      "vif_ko_chunk_count_log": 791.04},
      "fresh_spearman": {}, "fresh_pearson": {}}
for _n, _v in (("pair_log_size~ko_chunk_count_log", ("pair_log_size", "ko_chunk_count_log")),
               ("pair_log_size~en_chunk_count_log", ("pair_log_size", "en_chunk_count_log")),
               ("ko_chunk_count_log~en_chunk_count_log", ("ko_chunk_count_log", "en_chunk_count_log"))):
    _a = _m3[_v[0]].to_numpy(np.float64); _b = _m3[_v[1]].to_numpy(np.float64)
    M3["fresh_spearman"][_n] = float(stats.spearmanr(_a, _b).statistic)
    M3["fresh_pearson"][_n] = float(np.corrcoef(_a, _b)[0, 1])

_dens_cols = ["ko_chunk_per_codepoint", "en_chunk_per_codepoint", "ko_chunk_per_byte",
              "en_chunk_per_byte", "ko_chunk_per_eojeol", "en_chunk_per_word"]
M3["density_descriptors"] = {}
for _c in _dens_cols:
    _v = _m3[_c].to_numpy(np.float64)
    _v = _v[np.isfinite(_v)]
    M3["density_descriptors"][_c] = {
        "n_finite": int(_v.size), "mean": float(_v.mean()), "sd": float(_v.std(ddof=1)),
        "cv": float(_v.std(ddof=1) / _v.mean()),
        "q05": float(np.quantile(_v, 0.05)), "median": float(np.median(_v)),
        "q95": float(np.quantile(_v, 0.95)),
        "spearman_with_pair_log_size": float(stats.spearmanr(_pls[np.isfinite(_m3[_c].to_numpy(np.float64))], _v).statistic),
    }
M3["raw_vs_density_scale_dependence"] = {
    "raw_ko_chunk_count_log_rho": M3["fresh_spearman"]["pair_log_size~ko_chunk_count_log"],
    "raw_en_chunk_count_log_rho": M3["fresh_spearman"]["pair_log_size~en_chunk_count_log"],
    "density_ko_chunk_per_codepoint_rho": M3["density_descriptors"]["ko_chunk_per_codepoint"]["spearman_with_pair_log_size"],
    "density_en_chunk_per_codepoint_rho": M3["density_descriptors"]["en_chunk_per_codepoint"]["spearman_with_pair_log_size"],
}
_qtab = con.execute(f"""SELECT length_stratum,
    count(*) n,
    quantile_cont(ko_chunk_count / ko_codepoint_count, [0.25, 0.5, 0.75]) ko_chunk_per_cp,
    quantile_cont(en_chunk_count / en_codepoint_count, [0.25, 0.5, 0.75]) en_chunk_per_cp,
    quantile_cont(ko_chunk_count, 0.5) ko_chunk_p50,
    quantile_cont(en_chunk_count, 0.5) en_chunk_p50
  FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
M3["chunk_density_by_length_stratum"] = json.loads(_qtab.to_json(orient="records"))
SUMMARY["eda_ref_m3_01"] = M3

print("신선 계산 (전수, 표본추출 없음):")
for _k, _v in M3["fresh_spearman"].items():
    print(f"  Spearman ρ  {_k:42s} = {_v:.6f}   (Pearson r = {M3['fresh_pearson'][_k]:.6f})")
print("\n애드덤 — raw 대 density 기술자의 규모 의존성:")
show(pd.DataFrame([{"기술자": k, "중앙값": v["median"], "5%": v["q05"], "95%": v["q95"],
                    "변동계수 CV": v["cv"], "ρ(pair_log_size)": v["spearman_with_pair_log_size"]}
                   for k, v in M3["density_descriptors"].items()]))
print("\n길이 층별 chunk 밀도 (Q1/중앙값/Q3):")
show(_qtab)

신선 계산 (전수, 표본추출 없음):
  Spearman ρ  pair_log_size~ko_chunk_count_log           = 0.962502   (Pearson r = 0.958492)
  Spearman ρ  pair_log_size~en_chunk_count_log           = 0.969426   (Pearson r = 0.968582)
  Spearman ρ  ko_chunk_count_log~en_chunk_count_log      = 0.936764   (Pearson r = 0.935555)

애드덤 — raw 대 density 기술자의 규모 의존성:
                   기술자      중앙값        5%      95%  변동계수 CV  ρ(pair_log_size)
ko_chunk_per_codepoint 0.294118  0.242424 0.415552 0.194191         -0.324135
en_chunk_per_codepoint 0.202532  0.157692 0.285714  0.20786         -0.594554
     ko_chunk_per_byte 0.120879 0.0955056 0.188571 0.279052         -0.239631
     en_chunk_per_byte 0.202532  0.157658 0.285714  0.20759         -0.594862
   ko_chunk_per_eojeol      1.2    1.0625     1.75 0.190893         -0.311134
     en_chunk_per_word  1.14286   1.05556  1.41667 0.126892          -0.30003

길이 층별 chunk 밀도 (Q1/중앙값/Q3):
length_stratum      n                                               ko_chunk_per_cp        

In [25]:
fig, axes = plt.subplots(1, 3, figsize=(16.4, 5.2))
_pairs = [("pair_log_size", "ko_chunk_count_log", "문장쌍 규모  0.5·(ln 한국어 문자 수 + ln 영어 문자 수)",
           "log 한국어 chunk 수", "(a) 문장쌍 규모 대 한국어 chunk 수"),
          ("pair_log_size", "en_chunk_count_log", "문장쌍 규모  0.5·(ln 한국어 문자 수 + ln 영어 문자 수)",
           "log 영어 chunk 수", "(b) 문장쌍 규모 대 영어 chunk 수"),
          ("ko_chunk_count_log", "en_chunk_count_log", "log 한국어 chunk 수",
           "log 영어 chunk 수", "(c) 두 언어 chunk 규모")]
for _ax, (_xc, _yc, _xl, _yl, _t) in zip(axes, _pairs, strict=True):
    _x = _m3[_xc].to_numpy(np.float64); _y = _m3[_yc].to_numpy(np.float64)
    _key = f"{_xc}~{_yc}"
    density_panel(_ax, _x, _y, bins=200,
                  xlim=(float(np.quantile(_x, 0.0005)), float(np.quantile(_x, 0.9995))),
                  ylim=(float(np.quantile(_y, 0.0005)), float(np.quantile(_y, 0.9995))),
                  xlabel=_xl, ylabel=_yl,
                  title=f"{_t}\nSpearman ρ = {M3['fresh_spearman'][_key]:.4f} · "
                        f"Pearson r = {M3['fresh_pearson'][_key]:.4f}")
    binned_median(_ax, _x, _y, bins=45, color="#0f8a5f")
fig.suptitle("NB07-REF-M3-01 · chunk 규모와 절대 길이의 기술적 관계 "
             "(G5 REVIEW M3-01 · DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW)", fontsize=12.5)
fig.tight_layout()
save_fig(fig, "NB07-REF-M3-01_chunk_scale_vs_length_v001", "chunk 규모 대 절대 길이",
         rq="RQ5", contract="G5_REVIEW_REGISTER",
         note="G5 M3-01 — 기술적 관계일 뿐 인과적 중복성이 아니다", tag="G5_REFERENCE")

  saved NB07-REF-M3-01_chunk_scale_vs_length_v001  png 226 KiB  sha 280d9ba57807…  한글=True


{'figure_id': 'NB07-REF-M3-01_chunk_scale_vs_length_v001',
 'title_ko': 'chunk 규모 대 절대 길이',
 'rq': 'RQ5',
 'contract': 'G5_REVIEW_REGISTER',
 'tag': 'G5_REFERENCE',
 'note': 'G5 M3-01 — 기술적 관계일 뿐 인과적 중복성이 아니다',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/NB07-REF-M3-01_chunk_scale_vs_length_v001.png',
 'svg': 'outputs/figures/nb07/NB07-REF-M3-01_chunk_scale_vs_length_v001.svg',
 'png_sha256': '280d9ba57807811c06af89b789e8f466460a9b8f77bddd09516a4bf745f1556f',
 'svg_sha256': '9155c8fb33d7aaa39fe2dc1cdf7a847377f8427910900821d237084b3bd93583',
 'png_size_bytes': 231836,
 'svg_size_bytes': 143576,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

In [26]:
fig, axes = plt.subplots(2, 3, figsize=(16.4, 9.0))
_specs = [("ko_chunk_count", "한국어 raw chunk 수", "#c1272d", True),
          ("ko_chunk_per_codepoint", "한국어 chunk 수 / code point 수", "#c1272d", False),
          ("ko_chunk_per_byte", "한국어 chunk 수 / UTF-8 byte 수", "#c1272d", False),
          ("en_chunk_count", "영어 raw chunk 수", "#0f4c81", True),
          ("en_chunk_per_codepoint", "영어 chunk 수 / code point 수", "#0f4c81", False),
          ("en_chunk_per_byte", "영어 chunk 수 / UTF-8 byte 수", "#0f4c81", False)]
for _ax, (_c, _t, _col, _islog) in zip(axes.flat, _specs, strict=True):
    _v = _m3[_c].to_numpy(np.float64)
    _v = _v[np.isfinite(_v)]
    if _islog:
        _ax.hist(_v, bins=np.logspace(0, np.log10(_v.max()), 70), color=_col, edgecolor="none")
        _ax.set_xscale("log")
        _rho = float(stats.spearmanr(_pls, _m3[_c].to_numpy(np.float64)).statistic)
    else:
        _ax.hist(_v, bins=200, range=(0, float(np.quantile(_v, 0.999))), color=_col, edgecolor="none")
        _rho = M3["density_descriptors"][_c]["spearman_with_pair_log_size"]
    _ax.set_yscale("log")
    _ax.set_xlabel(_t); _ax.set_ylabel("문장쌍 수 (로그)")
    _ax.set_title(f"{_t}\nρ(문장쌍 규모) = {_rho:.4f}")
fig.suptitle("NB07-REF-M3-01B · 재모수화 후보의 construct 확인: raw count 대 chunk 밀도\n"
             "(DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW — 선택이 아니라 구조 확인)", fontsize=12.5)
fig.tight_layout()
save_fig(fig, "NB07-REF-M3-01B_chunk_density_constructs_v001", "chunk 밀도 재모수화 후보 구조",
         rq="RQ5", contract="G5_REVIEW_REGISTER_ADDENDUM",
         note="애드덤 — raw/밀도 기술자의 규모 의존성 비교, feature 선택 아님", tag="G5_REFERENCE")

  saved NB07-REF-M3-01B_chunk_density_constructs_v001  png 202 KiB  sha 46d658b80c18…  한글=True


{'figure_id': 'NB07-REF-M3-01B_chunk_density_constructs_v001',
 'title_ko': 'chunk 밀도 재모수화 후보 구조',
 'rq': 'RQ5',
 'contract': 'G5_REVIEW_REGISTER_ADDENDUM',
 'tag': 'G5_REFERENCE',
 'note': '애드덤 — raw/밀도 기술자의 규모 의존성 비교, feature 선택 아님',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/NB07-REF-M3-01B_chunk_density_constructs_v001.png',
 'svg': 'outputs/figures/nb07/NB07-REF-M3-01B_chunk_density_constructs_v001.svg',
 'png_sha256': '46d658b80c1802cff057e9b38822bc48cc7e6fa88e4923e664552a96a3b4820e',
 'svg_sha256': 'f3cbb6239a187ad39d014f0d834aeb05c55c933063a52dddcfaadc53f5c846b8',
 'png_size_bytes': 207134,
 'svg_size_bytes': 325532,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

In [27]:
fig, axes = plt.subplots(1, 2, figsize=(14.4, 5.0))
_order = sorted(_m3["length_stratum"].unique())
for _ax, _side, _nm, _col in ((axes[0], "ko", "한국어", "#c1272d"), (axes[1], "en", "영어", "#0f4c81")):
    _data = [_m3.loc[_m3["length_stratum"] == s, f"{_side}_chunk_per_codepoint"].to_numpy()
             for s in _order]
    _bp = _ax.boxplot(_data, tick_labels=_order, showfliers=False, whis=(5, 95), patch_artist=True,
                      medianprops={"color": "#0b0b0b", "lw": 1.6})
    for _b in _bp["boxes"]:
        _b.set_facecolor(_col); _b.set_alpha(0.45)
    _ax.set_xlabel("길이 층 (length_stratum)")
    _ax.set_ylabel(f"{_nm} chunk 수 / code point 수")
    _ax.set_title(f"({'ab'[_side == 'en']}) 길이 층별 {_nm} chunk 밀도 분포\n"
                  f"(수염 = 5–95 분위)")
fig.suptitle("NB07-REF-M3-01C · 길이 quantile별 chunk 밀도 — 밀도 형태는 규모 의존이 크게 줄어든다\n"
             "(DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW)", fontsize=12.5)
fig.tight_layout()
save_fig(fig, "NB07-REF-M3-01C_chunk_density_by_length_v001", "길이 층별 chunk 밀도",
         rq="RQ5", contract="G5_REVIEW_REGISTER_ADDENDUM",
         note="애드덤 — 길이 quantile별 chunk 밀도 분포", tag="G5_REFERENCE")
del _m3

  saved NB07-REF-M3-01C_chunk_density_by_length_v001  png 103 KiB  sha 3bfc26da9c34…  한글=True


### 수치 해석

새로 계산한 전수 순위 연관:
`pair_log_size ~ ko_chunk_count_log` Spearman ρ = 0.962502,
`pair_log_size ~ en_chunk_count_log` ρ = 0.969426,
`ko_chunk_count_log ~ en_chunk_count_log` ρ = 0.936764.
세 번째 값은 G5 §7.2가 보고한 `d05_chunk_scale` 계열 최대값 0.9368과 일치한다.

애드덤 기술자: chunk 밀도로 바꾸면 문장쌍 규모와의 순위 연관이 크게 약해지고 부호가 뒤집힌다.
`ko_chunk_per_codepoint` 중앙값 0.2941 · ρ = −0.3241,
`en_chunk_per_codepoint` 중앙값 0.2025 · ρ = −0.5946,
`ko_chunk_per_byte` 중앙값 0.1209 · ρ = −0.2396,
`en_chunk_per_byte` 중앙값 0.2025 · ρ = −0.5949.
보조 지표는 `ko_chunk_per_eojeol` 중앙값 1.2 · ρ = −0.3111,
`en_chunk_per_word` 중앙값 1.1429 · ρ = −0.3000이다.
raw 형태의 ρ ≈ +0.96~0.97과 비교하면 절대값이 절반 이하로 떨어진다.

### 분포 해석

(a)–(c)의 밀도 덩어리는 좁고 거의 직선이다. chunk 수는 텍스트 길이의 **단조 함수에 가깝다** —
o200k_base regex가 공백·문자군 경계에서 자르므로 문장이 길수록 chunk가 비례해 늘어난다.
구간별 중앙값 곡선이 거의 직선으로 지나가는 것이 그 근접 비례를 그대로 보여준다.

M3-01B는 애드덤이 요구한 construct validity 확인이다. raw chunk count는 길이의 대리변수에
가깝지만, 밀도 형태(`chunk / code point`, `chunk / byte`)는 **다른 것을 측정한다**:
텍스트가 얼마나 잘게 chunk로 쪼개지는지에 대한 기술자다.

다만 밀도가 **완전한 규모 불변은 아니다**. M3-01C에서 한국어 밀도 중앙값은
Q1 0.3333 → Q5 0.2841, 영어는 Q1 0.2432 → Q5 0.1795로 길이 층을 따라 단조 감소한다.
즉 긴 문장일수록 chunk 하나가 더 많은 문자를 담는다. 밀도 형태는 규모 의존을 크게 줄이지만
제거하지는 않으며, 이 잔여 의존성 자체가 PRE-NB09 검토가 다루어야 할 사실이다.

### 연구적 의미

G5가 M3에서 관측한 VIF 1,252.61은 병리가 아니라 해석 가능한 구조다: 세 개의 길이 대리변수
(`pair_log_size`, `ko_chunk_count_log`, `en_chunk_count_log`)가 같은 정보를 반복한다.
밀도 기술자는 그 정보 중복을 **구조적으로 분리**할 후보이며, 이 절은 그 후보가
실제로 다른 구성개념을 측정하는지 기술적으로 확인했다.

### 해석 한계 · Not established

- chunk 수와 절대 길이의 강한 연관은 **인과적 중복성을 성립시키지 않는다**.
- 이 절은 NB09에서 어떤 변수를 삭제해도 된다는 **허가가 아니다**.
- 어떤 재모수화가 더 낫다는 판단을 하지 않는다. 밀도 기술자가 "옳다"고 말하지 않는다.
- **NB09 matrix는 이 노트북에서 변경되지 않는다.**
- G5는 M3가 full rank(45/45)임을 확인했다. 이것은 재모수화 문제이지 구조적 결함이 아니다.

```
M3_REPARAMETERIZATION_DECISION = NOT_MADE_HERE
```

**운영 배치 (VD-BASELINE-20260818-1520 §3, 이 노트북의 판단이 아님)**: M3-01은 REVIEW이며
G5를 재개방하지 않는다. 동결된 primary M3 span을 유지하고, raw chunk-count 계수를 개별
실질 mechanism 효과로 해석하지 않으며, RQ5의 primary 증거는 **M3−M2 block 비교**다.
chunk-density 재모수화는 **NB11 sensitivity**로 배치된다. VIF만을 근거로 한 feature 삭제는
없다. 위 기술 증거는 그 배치의 근거 자료이지 배치 결정 자체가 아니다.

**Reference**: G5 REVIEW `M3-01`

## 16 — G5 REVIEW `SM-01` · script mixing 기술자

> **DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW**

### SSOT / RQ 대응

**Research Question**: RQ3 — 공백 밀도, grapheme 구조, Hangul/Latin/digit/punctuation 비율,
script mixing, 문장 길이, 숫자·특수기호가 log(TP)와 log(CompressionPenalty)에
어떤 조건부 연관을 갖는가?

**SSOT section**: `KOEN-TP-RS-001` §6.3; G5 adjudication §7.2 → REVIEW `SM-01`

**Canonical input**: D-02 `REP_FEATURES_v002`

**Physical variables**: `ko_script_type_count`, `ko_script_switch_count`,
`en_script_type_count`, `en_script_switch_count`

**Research purpose**: 두 기술자가 near rank-equivalent라는 G5 관측을 **새로 계산해 확인**하고,
애드덤이 요구한 결합 빈도 구조 · support · `I(type ≥ 2)` prevalence ·
`type ≥ 2` 부분집합 내부 분포를 제시한다. 목적은 **construct structure 확인**이지
대표 feature 선택이 아니다.

**Allowed claim**: 두 기술자의 결합 빈도 구조와 각각이 무엇을 세는지에 대한 기술.

**Prohibited claim**: 어느 쪽을 NB09의 대표 feature로 삼아야 한다는 판단.
결과와의 연관을 근거로 한 선택. 두 변수 중 하나를 삭제해도 된다는 허가.

**참조**: G5 REVIEW `SM-01`

In [28]:
_sm = fetch("""ko_script_type_count, ko_script_switch_count,
               en_script_type_count, en_script_switch_count""")
SM = {"note": "DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW — construct structure, not feature selection",
      "g5_reference": "REVIEW SM-01",
      "g5_reported_spearman": {"en": 0.9994, "ko": 0.9910},
      "fresh_spearman": {}, "discrepancy": {}}
for _side in ("ko", "en"):
    _t = _sm[f"{_side}_script_type_count"].to_numpy(np.float64)
    _s = _sm[f"{_side}_script_switch_count"].to_numpy(np.float64)
    _rho = float(stats.spearmanr(_t, _s).statistic)
    SM["fresh_spearman"][_side] = _rho
    SM["discrepancy"][_side] = abs(_rho - SM["g5_reported_spearman"][_side])

print("신선 Spearman (전수 3,835,988행, 표본추출 없음):")
for _side, _nm in (("en", "영어"), ("ko", "한국어")):
    print(f"  {_nm} script_type_count ~ script_switch_count = {SM['fresh_spearman'][_side]:.10f}  "
          f"(G5 참조값 {SM['g5_reported_spearman'][_side]:.4f}, 차이 {SM['discrepancy'][_side]:.2e})")
print("\n※ G5 값은 audit reference이지 목표값이 아니다. 크게 다르면 강제로 맞추지 않고 보고한다.")

SM["joint_frequency"], SM["type_support"], SM["type_ge2"] = {}, {}, {}
for _side, _nm in (("ko", "한국어"), ("en", "영어")):
    _j = con.execute(f"""SELECT {_side}_script_type_count AS type_count,
                                least({_side}_script_switch_count, 8) AS switch_capped,
                                count(*) AS n
                         FROM {A} GROUP BY 1, 2 ORDER BY 1, 2""").fetchdf()
    _piv = _j.pivot(index="type_count", columns="switch_capped", values="n").fillna(0).astype(np.int64)
    SM["joint_frequency"][_side] = json.loads(_piv.to_json(orient="index"))
    print(f"\n— {_nm}: script_type_count × script_switch_count 결합 빈도표 (전환 8 이상은 8로 절단) —")
    show(_piv.reset_index())

    _sup = con.execute(f"""SELECT {_side}_script_type_count AS type_count, count(*) AS n
                           FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
    _sup["share"] = _sup["n"] / COHORT["N"]
    SM["type_support"][_side] = _sup.to_dict(orient="records")
    print(f"— {_nm}: script_type_count support —")
    show(_sup.assign(share=lambda d: (d["share"] * 100).round(6)))

    _ge2 = con.execute(f"""SELECT
        count(*) AS n_ge2,
        avg({_side}_script_switch_count) AS switch_mean,
        min({_side}_script_switch_count) AS switch_min,
        quantile_cont({_side}_script_switch_count, [0.25, 0.5, 0.75, 0.95]) AS switch_q,
        max({_side}_script_switch_count) AS switch_max
      FROM {A} WHERE {_side}_script_type_count >= 2""").fetchdf().iloc[0].to_dict()
    _t2 = _sm.loc[_sm[f"{_side}_script_type_count"] >= 2]
    _rho2 = float(stats.spearmanr(_t2[f"{_side}_script_type_count"],
                                  _t2[f"{_side}_script_switch_count"]).statistic)
    _by_type = con.execute(f"""SELECT {_side}_script_type_count AS type_count, count(*) n,
        min({_side}_script_switch_count) mn,
        quantile_cont({_side}_script_switch_count, [0.25, 0.5, 0.75, 0.95]) q,
        max({_side}_script_switch_count) mx
      FROM {A} WHERE {_side}_script_type_count >= 2 GROUP BY 1 ORDER BY 1""").fetchdf()
    SM["type_ge2"][_side] = {
        "prevalence_n": int(_ge2["n_ge2"]), "prevalence_share": float(_ge2["n_ge2"] / COHORT["N"]),
        "switch_mean": float(_ge2["switch_mean"]), "switch_min": int(_ge2["switch_min"]),
        "switch_quantiles": [float(v) for v in _ge2["switch_q"]],
        "switch_max": int(_ge2["switch_max"]),
        "spearman_within_ge2": _rho2,
        "by_type": json.loads(_by_type.to_json(orient="records")),
    }
    print(f"— {_nm}: I(type ≥ 2) prevalence = {_ge2['n_ge2']:,} "
          f"({_ge2['n_ge2'] / COHORT['N'] * 100:.4f}%) —")
    print(f"   type ≥ 2 내부 Spearman ρ(type, switch) = {_rho2:.6f}   "
          f"(전체 {SM['fresh_spearman'][_side]:.6f})")
    print(f"   type = 2 대 type = 3 의 전환 횟수 분포:")
    show(_by_type)
SUMMARY["eda_ref_sm_01"] = SM

신선 Spearman (전수 3,835,988행, 표본추출 없음):
  영어 script_type_count ~ script_switch_count = 0.9993520546  (G5 참조값 0.9994, 차이 4.79e-05)
  한국어 script_type_count ~ script_switch_count = 0.9910090163  (G5 참조값 0.9910, 차이 9.02e-06)

※ G5 값은 audit reference이지 목표값이 아니다. 크게 다르면 강제로 맞추지 않고 보고한다.

— 한국어: script_type_count × script_switch_count 결합 빈도표 (전환 8 이상은 8로 절단) —
 type_count       0      1      2     3     4    5     6    7     8
          0     183      0      0     0     0    0     0    0     0
          1 2979730      0      0     0     0    0     0    0     0
          2       0 346417 322056 32534 89476 9232 23117 2675 11754
          3       0      0   1901  4724  4772 1954  2547  802  2114
— 한국어: script_type_count support —
 type_count       n    share
          0     183 0.004771
          1 2979730  77.6783
          2  837261  21.8265
          3   18814  0.49046
— 한국어: I(type ≥ 2) prevalence = 856,075 (22.3169%) —
   type ≥ 2 내부 Spearman ρ(type, switch) = 0.203776   (전체 0.991009)
   typ

— 영어: I(type ≥ 2) prevalence = 266,654 (6.9514%) —
   type ≥ 2 내부 Spearman ρ(type, switch) = 0.052173   (전체 0.999352)
   type = 2 대 type = 3 의 전환 횟수 분포:
 type_count      n  mn                                  q  mx
          2 266575   1               [1.0, 1.0, 1.0, 2.0]  42
          3     79   2 [4.0, 4.0, 6.0, 9.099999999999994]  16


In [29]:
fig, axes = plt.subplots(2, 2, figsize=(14.6, 9.0))
for _r, (_side, _nm) in enumerate((("ko", "한국어"), ("en", "영어"))):
    _piv = pd.DataFrame(SM["joint_frequency"][_side]).T
    _piv.index = _piv.index.astype(int)
    _piv = _piv.sort_index()
    _piv.columns = _piv.columns.astype(int)
    _piv = _piv[sorted(_piv.columns)]
    _arr = _piv.to_numpy(dtype=float)
    _ax = axes[_r, 0]
    _masked = np.ma.masked_where(_arr == 0, _arr)
    _cmap = plt.get_cmap("YlGnBu").copy(); _cmap.set_bad("#eeeeee")
    _pcm = _ax.imshow(_masked, cmap=_cmap, aspect="auto",
                      norm=LogNorm(vmin=max(_arr[_arr > 0].min(), 1), vmax=_arr.max()))
    _ax.set_xticks(range(_piv.shape[1]), [str(c) if c < 8 else "8+" for c in _piv.columns])
    _ax.set_yticks(range(_piv.shape[0]), _piv.index)
    for _i in range(_piv.shape[0]):
        for _j in range(_piv.shape[1]):
            _v = int(_arr[_i, _j])
            _ax.text(_j, _i, "0" if _v == 0 else f"{_v:,}", ha="center", va="center", fontsize=7.5,
                     color="white" if _v > _arr.max() / 6 else "#14213d")
    _ax.set_xlabel("script 전환 횟수 (script_switch_count)")
    _ax.set_ylabel("script 종류 수 (script_type_count)")
    _ax.set_title(f"({'ac'[_r]}) {_nm} 결합 빈도 — 두 기술자는 서로 다른 것을 센다\n"
                  f"전체 Spearman ρ = {SM['fresh_spearman'][_side]:.6f}")
    _ax.grid(False)
    _cb = fig.colorbar(_pcm, ax=_ax, pad=0.02); _cb.set_label("문장쌍 수 (로그 눈금)", fontsize=8)

    _ax = axes[_r, 1]
    _bt = pd.DataFrame(SM["type_ge2"][_side]["by_type"])
    _sub = con.execute(f"""SELECT {_side}_script_type_count AS t,
                                  least({_side}_script_switch_count, 12) AS s, count(*) n
                           FROM {A} WHERE {_side}_script_type_count >= 2
                           GROUP BY 1, 2 ORDER BY 1, 2""").fetchdf()
    _w = 0.38
    for _k, (_t, _col) in enumerate(((2, "#e8a33d"), (3, "#8d6a9f"))):
        _d = _sub[_sub["t"] == _t]
        if _d.empty:
            continue
        _ax.bar(_d["s"] + (_k - 0.5) * _w, _d["n"], width=_w, color=_col,
                label=f"script 종류 {_t}개 (n = {int(_bt.loc[_bt['type_count'] == _t, 'n'].iloc[0]):,})")
    _ax.set_yscale("log")
    _ax.set_xlabel("script 전환 횟수 (12 이상은 12로 절단 표시)")
    _ax.set_ylabel("문장쌍 수 (로그 눈금)")
    _ax.set_title(f"({'bd'[_r]}) {_nm} · script 종류 ≥ 2 부분집합 내부 전환 분포\n"
                  f"prevalence {SM['type_ge2'][_side]['prevalence_share'] * 100:.2f}% · "
                  f"부분집합 내부 ρ = {SM['type_ge2'][_side]['spearman_within_ge2']:.4f}")
    _ax.legend(fontsize=8)

fig.suptitle("NB07-REF-SM-01 · script mixing 기술자의 construct 구조 "
             "(G5 REVIEW SM-01 · DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW)", fontsize=12.5)
fig.tight_layout()
save_fig(fig, "NB07-REF-SM-01_script_mixing_structure_v001", "script mixing 기술자 구조",
         rq="RQ3", contract="G5_REVIEW_REGISTER",
         note="G5 SM-01 — 결합 빈도, support, type≥2 내부 구조. 대표 feature 선택 아님",
         tag="G5_REFERENCE")
del _sm

  saved NB07-REF-SM-01_script_mixing_structure_v001  png 258 KiB  sha 0a43b360f1bd…  한글=True


### 수치 해석

새로 계산한 전수 Spearman은 G5 보고값과 **본질적으로 일치**한다:
영어 0.9993520546 (G5 0.9994), 한국어 0.9910090163 (G5 0.9910).
물질적 불일치는 없으므로 강제 조정 없이 그대로 보고한다.

애드덤 항목:

1. **결합 빈도표** — 위 (a)·(c)에 전수 표로 제시했다.
2. **script_type_count support** — 한국어: 0개 183 · 1개 2,979,730 · 2개 837,261 · 3개 18,814.
   영어: 0개 217 · 1개 3,569,117 · 2개 266,575 · 3개 79.
3. **`I(type ≥ 2)` prevalence** — 한국어 856,075쌍(22.3169%), 영어 266,654쌍(6.9514%).
4. **type ≥ 2 부분집합 내부 전환 분포** — 한국어 type=2에서 전환 중앙값 2 (범위 1–34),
   type=3에서 중앙값 4 (범위 2–23). 영어 type=2 중앙값 1 (1–42), type=3 중앙값 4 (2–16).
5. **부분집합 내부 연관** — 전체에서 0.99를 넘던 ρ가 type ≥ 2 안에서는
   한국어 **0.2038**, 영어 **0.0522**로 **급락한다**.
6. **type=2 대 type=3** — 종류가 하나 늘면 전환 분포 전체가 오른쪽으로 이동한다.

### 분포 해석

5번이 이 절의 핵심 구조다. 전체 표본에서 두 기술자가 near rank-equivalent로 보이는 이유는
**`type_count = 1` 이면 `switch_count = 0` 이 대수적으로 강제되기 때문**이다
(결합 빈도표에서 type=1 행이 switch=0 열에만 질량을 갖는 것이 그 증거다).
관측의 77.6783%(한국어) · 93.0430%(영어)가 그 단일 셀에 있으므로 순위 상관이 0.99를 넘는다.

실제로 script가 섞인 부분집합 안에서는 두 기술자가 **뚜렷이 다른 것을 잰다**:
`script_type_count` 는 **몇 종류의 script가 존재하는가**(존재 다양성),
`script_switch_count` 는 **문자열을 따라가며 script가 몇 번 바뀌는가**(전환 빈도)를 센다.
같은 두 종류(한글 + 라틴)라도 전환 1회와 34회는 완전히 다른 텍스트다.

### 연구적 의미

G5의 `REPRESENTATIVE_FEATURE_REVIEW` 판정은 정당하다. 동시에, 두 기술자의 높은 전체 상관이
**구성개념의 동일성이 아니라 support의 희소성**에서 나온다는 사실이 여기서 드러났다.
이는 PRE-NB09 검토가 "둘 중 하나를 지우는" 문제가 아니라
"어떤 구성개념을 M1에서 재현하려는가"의 문제임을 시사한다.

### 해석 한계 · Not established

- 이 절은 **대표 feature를 선택하지 않는다**. SSOT나 구성개념 정의만으로 선택이 논리적으로
  강제되는 경우가 아니므로, 선택을 유보한다.
- 결과(log TP · log CP)와의 연관을 근거로 어느 쪽도 고르지 않았다 — 그런 연관은 계산하지도 않았다.
- 두 변수 중 하나를 삭제해도 된다는 허가가 아니다. G5가 보고한 VIF는
  `en_script_type_count` 기준 9.11–9.16으로 중간 수준이며, 선형 중복이 극단적이지 않음을
  뜻한다 (G5는 짝의 두 변수 각각에 대한 VIF를 별도로 보고하지는 않았다).

```
SM01_REPRESENTATIVE_FEATURE_CHOICE = NOT_MADE_HERE
```

**운영 배치 (VD-BASELINE-20260818-1520 §3, 이 노트북의 판단이 아님)**: SM-01은 REVIEW이며
G5를 재개방하지 않는다. 동결된 primary M1 span을 유지하고, `script_type_count` 와
`script_switch_count` 를 **두 개의 독립적 실질 효과로 해석하지 않으며**, 대안 모수화는
**NB11 sensitivity**로 배치된다. 위 기술 증거는 그 배치의 근거 자료이지 배치 결정 자체가
아니다.

**Reference**: G5 REVIEW `SM-01`

## 17 — 분석 계층 경계표: 형태소 ≠ regex chunk ≠ subword token

> **DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW**

### SSOT / RQ 대응

**Research Question**: RQ4 · RQ5 — 형태소 feature block과 tokenizer mechanism feature는
서로 다른 계층이며 SSOT §5가 요구하는 구분을 유지해야 한다.

**SSOT section**: `KOEN-TP-RS-001` §5 (분석 계층 · 언어학적 경계 · 버전 민감도 · 연구상 역할),
RR‑01, MN‑02

**Canonical input**: D-02 (어절 · 단어) + D-03 (형태소) + D-05 (regex chunk) + D-04 (subword token)

**Physical variables**: `ko_eojeol_count`, `en_word_count`, `morpheme_count`,
`ko/en_chunk_count`, `ko/en_token_count`

**Research purpose**: 세 계층이 같은 문장에서 각각 몇 개의 단위를 만드는지를 한 표·한 그림에서
나란히 보인다. 이 구분이 흐려지면 RQ4의 형태소 결과와 RQ5의 mechanism 결과가 뒤섞인다.

**Allowed claim**: 계층별 단위 수의 기술적 비교와 계층 간 개념적 차이의 기술.

**Prohibited claim**: 한 계층이 다른 계층을 "설명한다"·"결정한다"는 진술.
형태소 경계가 chunk 경계나 token 경계와 일치한다는 진술.

In [30]:
LAYERS = con.execute(f"""SELECT
    avg(ko_eojeol_count) ko_eojeol_mean,      quantile_cont(ko_eojeol_count, 0.5) ko_eojeol_p50,
    avg(morpheme_count) ko_morpheme_mean,     quantile_cont(morpheme_count, 0.5) ko_morpheme_p50,
    avg(ko_chunk_count) ko_chunk_mean,        quantile_cont(ko_chunk_count, 0.5) ko_chunk_p50,
    avg(ko_token_count) ko_token_mean,        quantile_cont(ko_token_count, 0.5) ko_token_p50,
    avg(en_word_count) en_word_mean,          quantile_cont(en_word_count, 0.5) en_word_p50,
    avg(en_chunk_count) en_chunk_mean,        quantile_cont(en_chunk_count, 0.5) en_chunk_p50,
    avg(en_token_count) en_token_mean,        quantile_cont(en_token_count, 0.5) en_token_p50
  FROM {A}""").fetchdf().iloc[0].to_dict()
LAYERS = {k: float(v) for k, v in LAYERS.items()}
LAYER_TABLE = pd.DataFrame([
    {"계층": "① 언어학적 형태소 분석", "산출 단위": "어절 (eojeol)", "언어": "한국어",
     "평균": LAYERS["ko_eojeol_mean"], "중앙값": LAYERS["ko_eojeol_p50"],
     "산출 주체": "Kiwi 형태소 분석기 (D-03)", "버전 민감도": "분석기/모델에 의존",
     "연구상 역할": "외생적 설명 feature"},
    {"계층": "① 언어학적 형태소 분석", "산출 단위": "형태소 (morpheme)", "언어": "한국어",
     "평균": LAYERS["ko_morpheme_mean"], "중앙값": LAYERS["ko_morpheme_p50"],
     "산출 주체": "Kiwi 형태소 분석기 (D-03)", "버전 민감도": "분석기/모델에 의존",
     "연구상 역할": "외생적 설명 feature"},
    {"계층": "② o200k_base regex chunking", "산출 단위": "regex chunk", "언어": "한국어",
     "평균": LAYERS["ko_chunk_mean"], "중앙값": LAYERS["ko_chunk_p50"],
     "산출 주체": "tiktoken pat_str (D-05)", "버전 민감도": "tokenizer 구현에 의존",
     "연구상 역할": "tokenizer mechanism audit"},
    {"계층": "③ 최종 subword tokenization", "산출 단위": "o200k_base token", "언어": "한국어",
     "평균": LAYERS["ko_token_mean"], "중앙값": LAYERS["ko_token_p50"],
     "산출 주체": "tiktoken BPE merge (D-04)", "버전 민감도": "매우 큼",
     "연구상 역할": "결과변수 생성 과정"},
    {"계층": "① 표층 단어 분리", "산출 단위": "단어 (word)", "언어": "영어",
     "평균": LAYERS["en_word_mean"], "중앙값": LAYERS["en_word_p50"],
     "산출 주체": "공백 기반 분리 (D-02)", "버전 민감도": "낮음",
     "연구상 역할": "외생적 설명 feature"},
    {"계층": "② o200k_base regex chunking", "산출 단위": "regex chunk", "언어": "영어",
     "평균": LAYERS["en_chunk_mean"], "중앙값": LAYERS["en_chunk_p50"],
     "산출 주체": "tiktoken pat_str (D-05)", "버전 민감도": "tokenizer 구현에 의존",
     "연구상 역할": "tokenizer mechanism audit"},
    {"계층": "③ 최종 subword tokenization", "산출 단위": "o200k_base token", "언어": "영어",
     "평균": LAYERS["en_token_mean"], "중앙값": LAYERS["en_token_p50"],
     "산출 주체": "tiktoken BPE merge (D-04)", "버전 민감도": "매우 큼",
     "연구상 역할": "결과변수 생성 과정"},
])
show(LAYER_TABLE, floatfmt="{:,.4f}")
print("\n※ SSOT §5: 언어학적 경계는 ①에서만 '중요'하고, ②·③에서는 '보장하지 않음'이다.")
SUMMARY["analysis_layer_boundary"] = {
    "note": "DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW",
    "means_and_medians": LAYERS,
    "table": LAYER_TABLE.to_dict(orient="records"),
    "ssot_ref": "KOEN-TP-RS-001 §5 · RR-01 · MN-02",
}

                         계층            산출 단위  언어      평균     중앙값                     산출 주체           버전 민감도                    연구상 역할
              ① 언어학적 형태소 분석      어절 (eojeol) 한국어 10.9514  9.0000       Kiwi 형태소 분석기 (D-03)       분석기/모델에 의존            외생적 설명 feature
              ① 언어학적 형태소 분석   형태소 (morpheme) 한국어 24.1534 19.0000       Kiwi 형태소 분석기 (D-03)       분석기/모델에 의존            외생적 설명 feature
② o200k_base regex chunking      regex chunk 한국어 13.5611 11.0000   tiktoken pat_str (D-05) tokenizer 구현에 의존 tokenizer mechanism audit
  ③ 최종 subword tokenization o200k_base token 한국어 27.1226 21.0000 tiktoken BPE merge (D-04)             매우 큼                결과변수 생성 과정
                 ① 표층 단어 분리        단어 (word)  영어 16.8564 13.0000           공백 기반 분리 (D-02)               낮음            외생적 설명 feature
② o200k_base regex chunking      regex chunk  영어 19.4989 15.0000   tiktoken pat_str (D-05) tokenizer 구현에 의존 tokenizer mechanism audit
  ③ 최종 subword tokenization o200k_base token  영어 20.1706 16.00

In [31]:
fig, axes = plt.subplots(1, 2, figsize=(15.0, 5.4))
_ko_layers = [("어절\n(형태소 분석 ①)", LAYERS["ko_eojeol_p50"], "#7f9c96"),
              ("형태소\n(형태소 분석 ①)", LAYERS["ko_morpheme_p50"], "#4c8577"),
              ("regex chunk\n(tokenizer ②)", LAYERS["ko_chunk_p50"], "#e8a33d"),
              ("subword token\n(tokenizer ③)", LAYERS["ko_token_p50"], "#c1272d")]
_en_layers = [("단어\n(표층 분리 ①)", LAYERS["en_word_p50"], "#7f9c96"),
              ("—", np.nan, "#ffffff"),
              ("regex chunk\n(tokenizer ②)", LAYERS["en_chunk_p50"], "#e8a33d"),
              ("subword token\n(tokenizer ③)", LAYERS["en_token_p50"], "#0f4c81")]
for _ax, _layers, _nm in ((axes[0], _ko_layers, "한국어"), (axes[1], _en_layers, "영어")):
    _lbls = [x[0] for x in _layers]
    _vals = [x[1] for x in _layers]
    _cols = [x[2] for x in _layers]
    _bars = _ax.bar(range(len(_lbls)), _vals, color=_cols, edgecolor="#333333", lw=0.6)
    for _i, _v in enumerate(_vals):
        if np.isfinite(_v):
            _ax.text(_i, _v, f"{_v:,.0f}", ha="center", va="bottom", fontsize=10)
    _ax.set_xticks(range(len(_lbls)), _lbls, fontsize=8.5)
    _ax.set_ylabel("문장당 단위 수 (중앙값)")
    _ax.set_title(f"{_nm} — 세 계층이 만드는 단위 수는 서로 다르다")
    _ax.axvspan(-0.5, 1.5, color="#7f9c96", alpha=0.10)
    _ax.axvspan(1.5, 3.5, color="#e8a33d", alpha=0.10)
    _ax.text(0.5, _ax.get_ylim()[1] * 0.95, "① 언어학적 계층", ha="center", fontsize=9, color="#3f6b5f")
    _ax.text(2.5, _ax.get_ylim()[1] * 0.95, "②③ tokenizer 계층", ha="center", fontsize=9, color="#96631f")
fig.suptitle("NB07-S06 · 분석 계층 경계: 형태소 ≠ regex chunking ≠ subword tokenization\n"
             "(SSOT §5 · RR-01 · MN-02 · DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW)", fontsize=12.5)
fig.tight_layout()
save_fig(fig, "NB07-S06_analysis_layer_boundary_v001", "분석 계층 경계표",
         rq="RQ4/RQ5", contract="SUPPORTING_DESCRIPTIVE_ADDENDUM",
         note="애드덤 BOUNDARY TABLE — 세 계층 구분")

  saved NB07-S06_analysis_layer_boundary_v001  png 131 KiB  sha 32dd9a3533d9…  한글=True


{'figure_id': 'NB07-S06_analysis_layer_boundary_v001',
 'title_ko': '분석 계층 경계표',
 'rq': 'RQ4/RQ5',
 'contract': 'SUPPORTING_DESCRIPTIVE_ADDENDUM',
 'tag': 'CANONICAL',
 'note': '애드덤 BOUNDARY TABLE — 세 계층 구분',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/NB07-S06_analysis_layer_boundary_v001.png',
 'svg': 'outputs/figures/nb07/NB07-S06_analysis_layer_boundary_v001.svg',
 'png_sha256': '32dd9a3533d9a071dc503ba5342dae2d57ee8bc5b7335072e7f5c6d89a9f8e67',
 'svg_sha256': '6603bf68df05f39865f803977f42488c849ce281568eb12dc2f3bd690e2ec323',
 'png_size_bytes': 134093,
 'svg_size_bytes': 27623,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

### 수치 해석

같은 한국어 문장에서 중앙값 기준: 어절 9개 → 형태소 19개 → regex chunk 11개 → subword token 21개.
영어에서는 단어 13개 → regex chunk 15개 → subword token 16개다.
평균으로는 한국어 10.95 → 24.15 → 13.56 → 27.12, 영어 16.86 → 19.50 → 20.17이다.

### 분포 해석

세 숫자열이 서로 **단조적으로도 정렬되지 않는다**. 한국어에서 형태소(19)가 regex chunk(11)보다
많고, subword token(21)이 다시 형태소보다 많다. 계층 사이에 포함 관계가 없다는 뜻이다.
영어에서는 세 값이 13–16으로 거의 겹치는데, 공백 분리·regex chunking·BPE merge가
라틴 문자열에서 대체로 같은 경계를 찾기 때문이다.

### 연구적 의미

SSOT §5의 계층 구분이 이 자료에서 실제로 유효하다는 것을 보여준다.
① 형태소 계층은 언어학적 경계를 중요하게 다루지만 ②③ tokenizer 계층은 그것을 보장하지 않는다.
따라서 RQ4(형태소)와 RQ5(mechanism)는 **다른 질문**이며, M2와 M3의 역할 분리
(MN‑02)도 이 경계에서 나온다.

### 해석 한계

이 표는 단위 **개수**만 비교한다. 경계의 일치 여부(어느 형태소 경계가 어느 token 경계와
겹치는가)는 측정하지 않았다. 한 계층의 수가 다른 계층의 수를 결정한다는 진술은 성립하지 않는다.
형태소 수는 Kiwi 특정 버전의 산출이고 chunk·token 수는 특정 tiktoken 구현의 산출이다.

## 18 — H. 출처 · 도메인 · 방향 · 길이 이질성  ·  **F03**

### SSOT / RQ 대응

**Research Question**: RQ6 — 도메인, 문장 유형, 번역 방향, 출처 및 길이 strata에 따라
Tokenization Premium과 설명요인의 크기가 달라지는가?

**SSOT section**: `KOEN-TP-RS-001` §6.6, §9.2, §16.2 (F03 — domain별 TP violin/box) · §35

**Canonical input**: D-01 + D-04 (+ D-02 분해 성분)

**Physical variables**: `log_token_premium`, `log_code_point_ratio`,
`log_byte_density_ratio`, `log_compression_penalty` × `source` · `domain` ·
`translation_direction` · `length_stratum` · `source_domain_cell`

**Research purpose**: 층별 기술적 이질성만 제시한다. 모형 기반 효과는 이후 단계다.

**Allowed claim**: 층별 분포·중앙값·산포의 기술적 차이.

**Prohibited claim**: 층 효과, 층 간 차이의 유의성, 통제 후 차이. F08(설명 forest)은
NB09 의존이며 여기서 만들지 않는다. §07의 식별 제약(ID-03 · ID-04)이 모든 층 해석에 적용된다.

In [32]:
def strata_table(var: str) -> pd.DataFrame:
    return con.execute(f"""SELECT {var} AS level, count(*) AS n,
        quantile_cont(log_token_premium, 0.25) AS ltp_q25,
        quantile_cont(log_token_premium, 0.5)  AS ltp_p50,
        quantile_cont(log_token_premium, 0.75) AS ltp_q75,
        exp(quantile_cont(log_token_premium, 0.5)) AS tp_p50,
        avg(CASE WHEN token_premium > 1 THEN 1.0 ELSE 0.0 END) AS share_tp_gt_1,
        quantile_cont(log_code_point_ratio, 0.5)     AS lcr_p50,
        quantile_cont(log_byte_density_ratio, 0.5)   AS lbdr_p50,
        quantile_cont(log_compression_penalty, 0.5)  AS lcp_p50
      FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()


HETERO = {}
for _v in ("source", "domain", "translation_direction", "length_stratum", "source_domain_cell"):
    _t = strata_table(_v)
    HETERO[_v] = json.loads(_t.to_json(orient="records"))
    print(f"— {_v} 별 log TP 및 분해 성분 중앙값 —")
    show(_t)
    print()
HETERO["note"] = ("descriptive heterogeneity only; ID-03/ID-04 identification constraints apply; "
                  "no model-based effect is estimated here")
SUMMARY["heterogeneity"] = HETERO

— source 별 log TP 및 분해 성분 중앙값 —
level       n  ltp_q25  ltp_p50  ltp_q75  tp_p50  share_tp_gt_1   lcr_p50  lbdr_p50  lcp_p50
  025 2485963 0.117783 0.274437 0.430783 1.31579       0.833623  -0.71784  0.902868 0.105361
  026 1350025 0.200671 0.308735 0.419854  1.3617       0.964974 -0.823917  0.884464 0.263825



— domain 별 log TP 및 분해 성분 중앙값 —
     level       n   ltp_q25  ltp_p50  ltp_q75  tp_p50  share_tp_gt_1   lcr_p50  lbdr_p50   lcp_p50
  dialogue  516162 0.0953102 0.251314 0.405465 1.28571       0.821587  -0.71562  0.905709 0.0695577
   general  804291 0.0800427 0.251314 0.405465 1.28571        0.77372 -0.737599  0.875469  0.126151
     other 2155630  0.167054 0.297252 0.430783 1.34615       0.914662 -0.777705  0.902868  0.193959
technology  359905  0.253781 0.346276 0.444686 1.41379       0.992081 -0.759839  0.856472   0.26719



— translation_direction 별 log TP 및 분해 성분 중앙값 —
   level       n  ltp_q25  ltp_p50  ltp_q75  tp_p50  share_tp_gt_1   lcr_p50  lbdr_p50   lcp_p50
EN_TO_KO 1273289 0.105361 0.262364 0.405465     1.3       0.824729 -0.729515  0.905709 0.0993424
KO_TO_EN 2512152  0.17185 0.300105 0.429563    1.35       0.907756 -0.776761  0.889262  0.211127
 UNKNOWN   50547 0.167054 0.325422 0.485508 1.38462       0.881477 -0.693147  0.912648  0.125233



— length_stratum 별 log TP 및 분해 성분 중앙값 —
level      n  ltp_q25  ltp_p50  ltp_q75  tp_p50  share_tp_gt_1   lcr_p50  lbdr_p50   lcp_p50
   Q1 840667 0.105361 0.287682 0.451985 1.33333       0.787778 -0.646627  0.887303 0.0723207
   Q2 858560 0.125163 0.287682 0.435318 1.33333       0.843813 -0.710242  0.907557 0.0956611
   Q3 802735 0.139762 0.287682 0.424883 1.33333       0.883588 -0.753772  0.904456  0.145383
   Q4 725825 0.209721 0.325422 0.439367 1.38462       0.958898 -0.790139  0.887303  0.244451
   Q5 608201 0.173272 0.273293 0.371564 1.31429       0.958718 -0.881548  0.885181  0.283246



— source_domain_cell 별 log TP 및 분해 성분 중앙값 —
         level       n   ltp_q25  ltp_p50  ltp_q75  tp_p50  share_tp_gt_1   lcr_p50  lbdr_p50   lcp_p50
  025-dialogue  516162 0.0953102 0.251314 0.405465 1.28571       0.821587  -0.71562  0.905709 0.0695577
   025-general  804291 0.0800427 0.251314 0.405465 1.28571        0.77372 -0.737599  0.875469  0.126151
     025-other 1165510  0.154151 0.305382 0.451985 1.35714       0.880291 -0.709408  0.911149  0.109267
     026-other  990120  0.182322 0.293348 0.405465 1.34091       0.955121 -0.849846  0.890973  0.262598
026-technology  359905  0.253781 0.346276 0.444686 1.41379       0.992081 -0.759839  0.856472   0.26719



In [33]:
_h = fetch("log_token_premium, domain, source, translation_direction, length_stratum, source_domain_cell")
fig, axes = plt.subplots(1, 2, figsize=(15.2, 5.6))

_dom = sorted(_h["domain"].unique())
_data = [_h.loc[_h["domain"] == d, "log_token_premium"].to_numpy() for d in _dom]
_vp = axes[0].violinplot(_data, showextrema=False, widths=0.85)
for _b in _vp["bodies"]:
    _b.set_facecolor("#3a7ca5"); _b.set_alpha(0.55); _b.set_edgecolor("#16425b")
_bp = axes[0].boxplot(_data, showfliers=False, whis=(5, 95), widths=0.18, patch_artist=True,
                      medianprops={"color": "#c1272d", "lw": 1.8},
                      boxprops={"facecolor": "white", "color": "#16425b"})
axes[0].axhline(0.0, color="#0b0b0b", lw=1.0, ls="--", label="TP = 1")
axes[0].set_xticks(range(1, len(_dom) + 1),
                   [f"{d}\n(n = {int((_h['domain'] == d).sum()):,})" for d in _dom], fontsize=8.5)
axes[0].set_ylabel("log Tokenization Premium")
axes[0].set_title("(a) 도메인별 log TP — 기술 분포 (수염 = 5–95 분위)")
axes[0].set_ylim(-0.8, 1.4); axes[0].legend(loc="upper right")

_cells = sorted(_h["source_domain_cell"].unique())
_data = [_h.loc[_h["source_domain_cell"] == c, "log_token_premium"].to_numpy() for c in _cells]
_vp = axes[1].violinplot(_data, showextrema=False, widths=0.85)
for _b, _c in zip(_vp["bodies"], _cells, strict=True):
    _b.set_facecolor("#c1272d" if _c.startswith("025") else "#0f4c81")
    _b.set_alpha(0.45); _b.set_edgecolor("#333333")
axes[1].boxplot(_data, showfliers=False, whis=(5, 95), widths=0.18, patch_artist=True,
                medianprops={"color": "#0b0b0b", "lw": 1.8},
                boxprops={"facecolor": "white", "color": "#333333"})
axes[1].axhline(0.0, color="#0b0b0b", lw=1.0, ls="--")
axes[1].set_xticks(range(1, len(_cells) + 1),
                   [f"{c}\n(n = {int((_h['source_domain_cell'] == c).sum()):,})" for c in _cells],
                   fontsize=8)
axes[1].set_ylabel("log Tokenization Premium")
axes[1].set_title("(b) 출처-도메인 셀별 log TP — 025-other 대 026-other 만이\n"
                  "동일 도메인 내 출처 층 대비다 (순수 source 효과 아님)")
axes[1].set_ylim(-0.8, 1.4)

fig.suptitle(f"F03 · 도메인 및 출처-도메인 셀별 Tokenization Premium 기술 분포 (N = {COHORT['N']:,})",
             fontsize=13)
fig.tight_layout()
save_fig(fig, "F03_domain_tp_descriptive_v001", "도메인별 TP 기술 분포",
         rq="RQ6", contract="SSOT_§16.2_F03 — domain별 TP violin/box",
         note="기술 분포만. ID-03 식별 제약이 적용된다")

  saved F03_domain_tp_descriptive_v001  png 240 KiB  sha e349525efded…  한글=True


{'figure_id': 'F03_domain_tp_descriptive_v001',
 'title_ko': '도메인별 TP 기술 분포',
 'rq': 'RQ6',
 'contract': 'SSOT_§16.2_F03 — domain별 TP violin/box',
 'tag': 'CANONICAL',
 'note': '기술 분포만. ID-03 식별 제약이 적용된다',
 'status': 'GENERATED',
 'png': 'outputs/figures/nb07/F03_domain_tp_descriptive_v001.png',
 'svg': 'outputs/figures/nb07/F03_domain_tp_descriptive_v001.svg',
 'png_sha256': 'e349525efdedb58077f3bd3eceb5428461f07f8dbf13c0dae20e1d95d198765b',
 'svg_sha256': 'ce0a491102dbaf5445e2b21f2d42335f3a6eec1d8113ba87a0e6f07e09623372',
 'png_size_bytes': 245960,
 'svg_size_bytes': 82125,
 'korean_text_in_svg': True,
 'font': 'Noto Sans CJK KR'}

In [34]:
fig, axes = plt.subplots(2, 2, figsize=(14.6, 8.6))
for _ax, _v, _t in zip(axes.flat,
                       ("source", "translation_direction", "length_stratum", "domain"),
                       ("(a) 출처별", "(b) 번역 방향별", "(c) 길이 층별", "(d) 도메인별"), strict=True):
    _lv = sorted(_h[_v].unique())
    _data = [_h.loc[_h[_v] == l, "log_token_premium"].to_numpy() for l in _lv]
    _bp = _ax.boxplot(_data, tick_labels=[f"{l}\n(n={int((_h[_v] == l).sum()):,})" for l in _lv],
                      showfliers=False, whis=(5, 95), patch_artist=True,
                      medianprops={"color": "#c1272d", "lw": 1.8},
                      boxprops={"facecolor": "#81c3d7", "alpha": 0.6, "color": "#16425b"})
    _ax.axhline(0.0, color="#0b0b0b", lw=1.0, ls="--")
    _ax.set_ylabel("log Tokenization Premium"); _ax.set_ylim(-0.7, 1.2)
    _ax.set_title(f"{_t} log TP 기술 분포")
    _ax.tick_params(axis="x", labelsize=8)
fig.suptitle("NB07-S07 · 층별 기술적 이질성 — 모형 기반 효과가 아니다 (F08은 NB09 의존)",
             fontsize=13)
fig.tight_layout()
save_fig(fig, "NB07-S07_heterogeneity_v001", "층별 기술적 이질성",
         rq="RQ6", contract="SUPPORTING_DESCRIPTIVE",
         note="source/direction/length/domain별 log TP 기술 분포")
del _h

  saved NB07-S07_heterogeneity_v001  png 153 KiB  sha 704b3f3f8848…  한글=True


### 수치 해석

출처별 log TP 중앙값: 025 = 0.2744, 026 = 0.3087. 도메인별: dialogue 0.2513,
general 0.2513, other 0.2973, technology 0.3463.
방향별: KO_TO_EN 0.3001, EN_TO_KO 0.2624, UNKNOWN 0.3254.
길이 층별: Q1–Q3 모두 0.2877, Q4 0.3254, Q5 0.2733로 단조가 아니다.

### 분포 해석

모든 층에서 log TP 분포가 0의 **오른쪽**에 중심을 두며, 어떤 층에서도 중앙값이 0 이하로
내려가지 않는다. 층 간 중앙값의 폭(0.2513–0.3463, 즉 0.095)은 분포 내부 산포
(전체 IQR ≈ 0.273)보다 작다 — 이질성은 존재하지만 층이 개체 간 변동을 지배하지 않는다.

`TP > 1` 비율의 층간 격차는 중앙값 격차보다 크다: general 0.7737에서 technology 0.9921까지
벌어진다. 중앙값이 비슷해도 분포의 좌측 질량이 층마다 크게 다르다는 뜻이다.

분해 성분의 층별 중앙값을 함께 보면 차이의 소재가 보인다. 예컨대 025와 026의 log TP 격차
(0.2744 대 0.3087)에서 log BDR은 거의 같고(0.9029 대 0.8845) log CP가 크게 다르다
(0.1054 대 0.2638) — 026 쪽에서 tokenizer 압축 열위가 더 크다. 이는 기술적 관찰이며
통제 후 진술이 아니다.

### 연구적 의미

RQ6의 기술적 출발점이 마련되었다. 층별 차이가 존재한다는 관찰은 NB09에서 층 통제와
상호작용을 다룰 필요를 뒷받침한다.

### 해석 한계

**§07의 식별 제약이 그대로 적용된다.** 출처와 도메인은 분리 식별되지 않으므로,
(b)의 셀 간 차이를 "출처 효과" 또는 "도메인 효과"로 읽을 수 없다.
`025-other` 대 `026-other` 만이 동일 도메인 내 출처 층 대비이며, 그조차 순수 source 효과가 아니다.
방향 대비는 025 내부 변동에서만 식별된다.
어떤 층 차이도 검정되지 않았고, 통제 후 차이도 아니다. 대표본에서는 사소한 차이도
유의해질 수 있으므로 SSOT §9.1에 따라 효과 크기와 구간을 우선한다 — 그 추정은 NB09의 몫이다.

**Reference**: G5 `ID-03`, `ID-04` (§07)

## 19 — I. 극단값 · 경계 사례 audit  ·  **F07**

### SSOT / RQ 대응

**Research Question**: RQ1 · RQ5 — premium의 극단 사례와 tokenizer mechanism의 경계 사례가
어떤 구조를 갖는가.

**SSOT section**: `KOEN-TP-RS-001` §16.2 (F07 — extreme TP case audit panel), §16.3 (Extreme-case audit —
극단값은 먼저 삭제하지 않는다)

**Canonical input**: D-02 + D-03 + D-04 + D-05

**Physical variables**: `token_premium`, `ko/en_token_count`, `ko/en_codepoint_count`,
`morpheme_density`, `ko/en_chunk_count`, 분해 성분

**Research purpose**: 극단값을 **삭제하지 않고** 드러낸다. 먼저 전체 분포를 보이고,
그 다음 확대·로그 눈금·robust 범위 패널을 붙인다.

**Allowed claim**: 극단 사례의 개수·비율·집계 기술자 프로파일.

**Prohibited claim**: 극단 사례가 오류라는 판정 없이 이루어지는 배제. 개별 문장 내용에 대한 진술.

**개인정보 경계**: 원문 KO/EN 문장, 복원 가능한 pair 예시, token ID 배열을 어떤 공개
artifact에도 싣지 않는다. 개별 사례 점검이 필요하면 로컬 전용 private audit 파일로만 남기고,
canonical 산출물에는 집계·sanitize된 기술자만 넣는다.

In [35]:
_p_lo = float(con.execute(f"SELECT quantile_cont(token_premium, 0.0001) FROM {A}").fetchone()[0])
_p_hi = float(con.execute(f"SELECT quantile_cont(token_premium, 0.9999) FROM {A}").fetchone()[0])
EXTREME = {"threshold_low_p0001": _p_lo, "threshold_high_p9999": _p_hi}

_prof = con.execute(f"""SELECT
    CASE WHEN token_premium <= {_p_lo} THEN 'A_하위극단 (TP ≤ {_p_lo:.4f})'
         WHEN token_premium >= {_p_hi} THEN 'C_상위극단 (TP ≥ {_p_hi:.4f})'
         ELSE 'B_핵심 (0.01–99.99 분위)' END AS grp,
    count(*) AS n,
    quantile_cont(token_premium, 0.5)        AS tp_p50,
    quantile_cont(ko_token_count, 0.5)       AS ko_tok_p50,
    quantile_cont(en_token_count, 0.5)       AS en_tok_p50,
    quantile_cont(ko_codepoint_count, 0.5)   AS ko_cp_p50,
    quantile_cont(en_codepoint_count, 0.5)   AS en_cp_p50,
    quantile_cont(log_code_point_ratio, 0.5) AS lcr_p50,
    quantile_cont(log_byte_density_ratio, 0.5) AS lbdr_p50,
    quantile_cont(log_compression_penalty, 0.5) AS lcp_p50,
    quantile_cont(morpheme_density, 0.5)     AS morph_density_p50,
    quantile_cont(ko_chunk_count, 0.5)       AS ko_chunk_p50,
    quantile_cont(en_chunk_count, 0.5)       AS en_chunk_p50,
    avg(ko_hangul_share)                     AS ko_hangul_mean,
    avg(ko_digit_share)                      AS ko_digit_mean,
    avg(ko_punctuation_share)                AS ko_punct_mean
  FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
EXTREME["group_profiles"] = json.loads(_prof.to_json(orient="records"))
print("극단·핵심 집단별 sanitize된 집계 프로파일 (원문·token ID 없음):")
show(_prof)

_decade = con.execute(f"""SELECT
    CASE WHEN token_premium < 0.25 THEN '① TP < 0.25'
         WHEN token_premium < 0.5  THEN '② 0.25 ≤ TP < 0.5'
         WHEN token_premium < 0.8  THEN '③ 0.5 ≤ TP < 0.8'
         WHEN token_premium < 1.0  THEN '④ 0.8 ≤ TP < 1'
         WHEN token_premium = 1.0  THEN '⑤ TP = 1 (정확 일치)'
         WHEN token_premium <= 2.0 THEN '⑥ 1 < TP ≤ 2'
         WHEN token_premium <= 3.0 THEN '⑦ 2 < TP ≤ 3'
         WHEN token_premium <= 5.0 THEN '⑧ 3 < TP ≤ 5'
         ELSE '⑨ TP > 5' END AS band,
    count(*) AS n FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
_decade["share"] = _decade["n"] / COHORT["N"]
EXTREME["tp_bands"] = json.loads(_decade.to_json(orient="records"))
print("\nTP 구간별 관측 수 (삭제 없음):")
show(_decade.assign(share=lambda d: (d["share"] * 100).round(6)))
SUMMARY["extreme_audit"] = EXTREME

# ---- 로컬 전용 private audit artifact (공개 artifact에는 절대 포함되지 않는다) ----------
_priv = con.execute(f"""SELECT pair_id, token_premium, ko_token_count, en_token_count,
       ko_codepoint_count, en_codepoint_count, log_code_point_ratio, log_byte_density_ratio,
       log_compression_penalty, ko_chunk_count, en_chunk_count, morpheme_density,
       source, domain, translation_direction
  FROM {A} WHERE token_premium <= {_p_lo} OR token_premium >= {_p_hi}
  ORDER BY token_premium""").fetchdf()
_priv_path = PRIVDIR / "NB07_EXTREME_CASE_PRIVATE_v001.csv"
_priv.to_csv(_priv_path, index=False)
EXTREME["private_audit_artifact"] = {
    "path": _priv_path.relative_to(ROOT).as_posix(), "rows": int(len(_priv)),
    "contains_raw_text": False, "contains_token_ids": False,
    "gitignored": True, "note": "local-only; 공개 package에 포함되지 않는다",
}
print(f"\n로컬 전용 private audit 파일: {_priv_path.relative_to(ROOT)} — {len(_priv):,}행 "
      f"(원문 없음 · token ID 없음 · .gitignore 대상)")

극단·핵심 집단별 sanitize된 집계 프로파일 (원문·token ID 없음):
                 grp       n  tp_p50  ko_tok_p50  en_tok_p50  ko_cp_p50  en_cp_p50   lcr_p50  lbdr_p50    lcp_p50  morph_density_p50  ko_chunk_p50  en_chunk_p50  ko_hangul_mean  ko_digit_mean  ko_punct_mean
A_하위극단 (TP ≤ 0.4000)     432   0.375           4          10          6         40  -1.79176  0.900405 -0.0645385                  3             3            10        0.674176      0.0238706       0.188772
B_핵심 (0.01–99.99 분위) 3835020 1.33333          21          16         36         74 -0.760451  0.896088   0.175174            2.21429            11            15        0.706057      0.0149275      0.0509385
C_상위극단 (TP ≥ 4.0000)     536 4.33333          16           3         25         11  0.484887  0.891998    0.19698            2.40455             8             3        0.694288     0.00365379       0.109006

TP 구간별 관측 수 (삭제 없음):
             band       n    share
      ① TP < 0.25      11 0.000287
② 0.25 ≤ TP < 0.5    1192 0.031074

In [36]:
_ex = fetch("token_premium, log_token_premium, ko_token_count, en_token_count")
fig, axes = plt.subplots(2, 2, figsize=(14.6, 9.0))

_v = _ex["token_premium"].to_numpy(np.float64)
axes[0, 0].hist(_v, bins=400, range=(0, float(_v.max())), color="#3a7ca5", edgecolor="none")
axes[0, 0].set_yscale("log")
for _x, _c, _l in ((1.0, "#0b0b0b", "TP = 1"), (_p_lo, "#8d6a9f", f"0.01 분위 {_p_lo:.3f}"),
                   (_p_hi, "#c1272d", f"99.99 분위 {_p_hi:.3f}")):
    axes[0, 0].axvline(_x, color=_c, lw=1.4, ls="--", label=_l)
axes[0, 0].set_xlabel("Tokenization Premium (전체 관측 범위)")
axes[0, 0].set_ylabel("문장쌍 수 (로그 눈금)")
axes[0, 0].set_title("(a) 전체 분포 먼저 — 극단값은 삭제하지 않는다")
axes[0, 0].legend(fontsize=8)

axes[0, 1].barh(_decade["band"], _decade["n"], color="#7f9c96")
for _i, (_n, _s) in enumerate(zip(_decade["n"], _decade["share"], strict=True)):
    axes[0, 1].text(_n, _i, f" {_n:,} ({_s * 100:.4f}%)", va="center", fontsize=8)
axes[0, 1].set_xscale("log")
axes[0, 1].set_xlabel("문장쌍 수 (로그 눈금)")
axes[0, 1].set_title("(b) TP 구간별 관측 수 — 양측 꼬리 확대")
axes[0, 1].set_xlim(0.7, _decade["n"].max() * 60)

_grp = _prof.set_index("grp")
_metrics = [("ko_tok_p50", "한국어 token"), ("en_tok_p50", "영어 token"),
            ("ko_cp_p50", "한국어 문자"), ("en_cp_p50", "영어 문자"),
            ("ko_chunk_p50", "한국어 chunk"), ("en_chunk_p50", "영어 chunk")]
_x = np.arange(len(_metrics)); _w = 0.26
for _k, (_g, _c) in enumerate(zip(_grp.index, ("#8d6a9f", "#3a7ca5", "#c1272d"), strict=True)):
    axes[1, 0].bar(_x + (_k - 1) * _w, [_grp.loc[_g, m] for m, _ in _metrics], width=_w,
                   color=_c, label=f"{_g}  n = {int(_grp.loc[_g, 'n']):,}")
axes[1, 0].set_yscale("log")
axes[1, 0].set_xticks(_x, [n for _, n in _metrics], fontsize=8.5)
axes[1, 0].set_ylabel("중앙값 (로그 눈금)")
axes[1, 0].set_title("(c) 극단 집단의 sanitize된 규모 프로파일 — 원문 없음")
axes[1, 0].legend(fontsize=7.5)

_mask = (_ex["token_premium"] <= _p_lo) | (_ex["token_premium"] >= _p_hi)
_lk = np.log10(_ex.loc[_mask, "ko_token_count"].to_numpy(np.float64))
_le = np.log10(_ex.loc[_mask, "en_token_count"].to_numpy(np.float64))
_m = float(max(_lk.max(), _le.max()))
density_panel(axes[1, 1], _le, _lk, bins=90, xlim=(0, _m), ylim=(0, _m),
              xlabel="영어 token 수 (상용로그)", ylabel="한국어 token 수 (상용로그)",
              title=f"(d) 양측 극단 {int(_mask.sum()):,}쌍의 token 수 분포\n"
                    "짧은 문장에 집중된다 (격자 효과)")
axes[1, 1].plot([0, _m], [0, _m], color="#0b0b0b", lw=1.2, ls="--", label="동일선")
_tk = np.array([0, 1, 2, 3])
axes[1, 1].set_xticks(_tk, [f"{10 ** t:,.0f}" for t in _tk])
axes[1, 1].set_yticks(_tk, [f"{10 ** t:,.0f}" for t in _tk])
axes[1, 1].legend(loc="upper left")

fig.suptitle(f"F07 · 극단 TP 사례 audit 패널 (sanitize 완료, N = {COHORT['N']:,})", fontsize=13)
fig.tight_layout()
save_fig(fig, "F07_extreme_case_audit_v001", "극단 TP 사례 audit",
         rq="RQ1/RQ5", contract="SSOT_§16.2_F07 — extreme TP case audit panel",
         note="전체 분포 → 확대 → 집계 프로파일. 원문·token ID 없음")
del _ex

  saved F07_extreme_case_audit_v001  png 234 KiB  sha 23e8866ed5b5…  한글=True


### 수치 해석

0.01 분위 TP = 0.4000, 99.99 분위 TP = 4.0000. 하위 극단 432쌍, 상위 극단 536쌍,
합계 968쌍이다(핵심 집단 3,835,020쌍).
TP 구간별로는 `1 < TP ≤ 2` 3,283,936쌍(85.61%), `TP = 1` 정확 일치 196,718쌍(5.13%),
`0.8 ≤ TP < 1` 205,471쌍(5.36%), `2 < TP ≤ 3` 88,819쌍(2.32%), `3 < TP ≤ 5` 2,222쌍(0.058%),
`TP > 5` 118쌍(0.0031%), `TP < 0.25` 11쌍(0.0003%)이다.

### 분포 해석

(d)가 극단값의 정체를 보여준다. 양측 극단은 거의 전부 **짧은 문장**이다.
token 수가 한 자리일 때 TP는 1/5, 1/4, … , 4/1 같은 성긴 격자 위에서만 값을 가지므로
극단 비율이 산술적으로 쉽게 나온다. (c)에서도 극단 집단의 규모가 핵심 집단보다 뚜렷하게 작다:
한국어 token 중앙값이 하위극단 4 · 상위극단 16 대 핵심 21, 영어 token 중앙값이
하위극단 10 · 상위극단 3 대 핵심 16이다. 상위극단은 영어 측이, 하위극단은 한국어 측이
특히 짧다.

분해 성분으로 보면 두 극단의 성격이 다르다. 하위극단은 log CR 중앙값 −1.7918로 한국어가
문자 수에서 극단적으로 짧고, 상위극단은 log CR 중앙값 +0.4849로 반대로 한국어가 더 길다.
반면 log BDR 중앙값은 세 집단 모두 0.89~0.90으로 거의 동일하다 — 극단성은 byte 밀도가
아니라 문자 수 비에서 온다.

즉 대부분의 극단값은 **측정 결함이 아니라 이산 비율의 경계 행동**이다.

### 연구적 의미

이 판단이 중요한 이유는, 극단값을 결함으로 오인해 삭제하면 짧은 문장 전체가 체계적으로
빠져나가면서 cohort가 편향되기 때문이다. SSOT §16.3은 극단값을 먼저 삭제하지 말 것을 요구하며,
이 노트북은 어떤 행도 삭제하지 않았다.

### 해석 한계

개별 극단 사례의 원문은 검토하지 않았고 공개 artifact에 싣지 않았다.
로컬 전용 private 파일에도 문장 원문과 token ID는 포함되지 않는다.
"극단값이 모두 정상"이라는 진술은 하지 않는다 — §21의 register가 `REVIEW_REQUIRED`
항목을 따로 유지한다.

**Reference**: `[EDA-REF-A03]` · `[EDA-REF-A04]` — §21

## 20 — `eojeol_count = 1` 재검토 (명시적 요구 항목)

### SSOT / RQ 대응

**Research Question**: RQ4 — 형태소 지표의 분포와 경계 사례.

**SSOT section**: `KOEN-TP-RS-001` §5, §16.3; G5 adjudication §3 (retained by policy)

**Canonical input**: D-03 `MORPH_FEATURES_KIWI_v001` (+ D-02 · D-04 · D-05)

**Physical variables**: `morph_eojeol_count`, `ko_eojeol_count`, `morpheme_count`,
`morpheme_density`, `log_token_premium`

**Research purpose**: `eojeol_count = 1` 사례의 개수·비율·분포를 **새로 계산**한다.
G5는 이 집단을 정책상 cohort에 유지했다. 여기서는 그 집단이 실제로 무엇인지 기술한다.

**Allowed claim**: 개수·비율·규모 프로파일·분포 차이의 기술.

**Prohibited claim**: 이것을 "언어적 복잡도"로 부르는 것. 결함 판정에 근거한 삭제.

In [37]:
_e1 = con.execute(f"""SELECT
    CASE WHEN morph_eojeol_count = 1 THEN 'eojeol_count = 1' ELSE 'eojeol_count ≥ 2' END AS grp,
    count(*) n,
    quantile_cont(ko_codepoint_count, 0.5) ko_cp_p50,
    quantile_cont(en_codepoint_count, 0.5) en_cp_p50,
    quantile_cont(ko_token_count, 0.5) ko_tok_p50,
    quantile_cont(en_token_count, 0.5) en_tok_p50,
    quantile_cont(morpheme_count, 0.5) morph_p50,
    quantile_cont(morpheme_density, 0.5) morph_density_p50,
    quantile_cont(log_token_premium, 0.5) ltp_p50,
    avg(CASE WHEN token_premium > 1 THEN 1.0 ELSE 0.0 END) share_tp_gt_1,
    avg(CASE WHEN token_premium = 1 THEN 1.0 ELSE 0.0 END) share_tp_eq_1
  FROM {A} GROUP BY 1 ORDER BY 1""").fetchdf()
_e1_n = int(_e1.loc[_e1["grp"] == "eojeol_count = 1", "n"].iloc[0])
EOJEOL1 = {"count": _e1_n, "share": _e1_n / COHORT["N"],
           "identical_in_D02_and_D03": bool(
               con.execute(f"SELECT count(*) FROM {A} WHERE ko_eojeol_count = 1").fetchone()[0] == _e1_n),
           "group_profiles": json.loads(_e1.to_json(orient="records"))}
_e1_strata = con.execute(f"""SELECT source, domain, count(*) n FROM {A}
    WHERE morph_eojeol_count = 1 GROUP BY 1, 2 ORDER BY 3 DESC""").fetchdf()
EOJEOL1["by_source_domain"] = json.loads(_e1_strata.to_json(orient="records"))
_e1_md = con.execute(f"""SELECT morpheme_count, count(*) n FROM {A}
    WHERE morph_eojeol_count = 1 GROUP BY 1 ORDER BY 1""").fetchdf()
EOJEOL1["morpheme_count_distribution"] = json.loads(_e1_md.head(20).to_json(orient="records"))
SUMMARY["eojeol_count_1"] = EOJEOL1

print(f"EOJEOL1_COUNT = {_e1_n:,}")
print(f"EOJEOL1_SHARE = {EOJEOL1['share'] * 100:.6f}%")
print(f"D-02 ko_eojeol_count 와 D-03 eojeol_count 의 개수 일치: {EOJEOL1['identical_in_D02_and_D03']}")
print("\n집단별 프로파일:")
show(_e1)
print("\n출처·도메인 분포:")
show(_e1_strata)

EOJEOL1_COUNT = 42,096
EOJEOL1_SHARE = 1.097397%
D-02 ko_eojeol_count 와 D-03 eojeol_count 의 개수 일치: True

집단별 프로파일:
             grp       n  ko_cp_p50  en_cp_p50  ko_tok_p50  en_tok_p50  morph_p50  morph_density_p50  ltp_p50  share_tp_gt_1  share_tp_eq_1
eojeol_count = 1   42096          5         12           5           4          4                  4        0       0.493966       0.271974
eojeol_count ≥ 2 3793892         37         75          21          16         20                2.2 0.287682       0.884132      0.0488335

출처·도메인 분포:
source   domain     n
   025  general 40587
   025 dialogue  1035
   025    other   469
   026    other     5


In [38]:
_ed = fetch("morph_eojeol_count, log_token_premium, morpheme_count, morpheme_density, ko_token_count")
_is1 = _ed["morph_eojeol_count"].to_numpy() == 1
fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.8))

axes[0].hist(_ed.loc[~_is1, "log_token_premium"], bins=200, range=(-1.0, 1.6), density=True,
             histtype="step", lw=1.8, color="#0f4c81", label=f"어절 ≥ 2 (n = {int((~_is1).sum()):,})")
axes[0].hist(_ed.loc[_is1, "log_token_premium"], bins=200, range=(-1.0, 1.6), density=True,
             histtype="step", lw=1.8, color="#c1272d", label=f"어절 = 1 (n = {_e1_n:,})")
axes[0].axvline(0.0, color="#0b0b0b", lw=1.0, ls="--")
axes[0].set_xlabel("log Tokenization Premium"); axes[0].set_ylabel("밀도 (정규화)")
axes[0].set_title("(a) 어절 수 = 1 집단의 log TP 분포\n분포 자체가 다르다"); axes[0].legend(fontsize=8)

axes[1].hist(_ed.loc[~_is1, "morpheme_count"], bins=np.arange(0, 60), density=True,
             histtype="step", lw=1.8, color="#0f4c81", label="어절 ≥ 2")
axes[1].hist(_ed.loc[_is1, "morpheme_count"], bins=np.arange(0, 60), density=True,
             histtype="step", lw=1.8, color="#c1272d", label="어절 = 1")
axes[1].set_yscale("log")
axes[1].set_xlabel("형태소 수 (morpheme_count)"); axes[1].set_ylabel("밀도 (로그)")
axes[1].set_title("(b) 어절 = 1 은 형태소도 적다 — 짧은 텍스트"); axes[1].legend(fontsize=8)

axes[2].barh(_e1_strata["source"] + " · " + _e1_strata["domain"], _e1_strata["n"], color="#8d6a9f")
for _i, _n in enumerate(_e1_strata["n"]):
    axes[2].text(_n, _i, f" {_n:,}", va="center", fontsize=8.5)
axes[2].set_xscale("log")
axes[2].set_xlabel("문장쌍 수 (로그 눈금)")
axes[2].set_title("(c) 어절 = 1 사례의 출처·도메인 분포\n특정 층에 집중된다")
axes[2].set_xlim(0.7, _e1_strata["n"].max() * 25)

fig.suptitle(f"NB07-S08 · eojeol_count = 1 재검토 — {_e1_n:,}쌍 "
             f"({EOJEOL1['share'] * 100:.4f}%), 삭제하지 않음", fontsize=13)
fig.tight_layout()
save_fig(fig, "NB07-S08_eojeol_count_one_v001", "eojeol_count = 1 재검토",
         rq="RQ4", contract="SUPPORTING_DESCRIPTIVE",
         note="명시적 재검토 요구 항목. 언어적 복잡도로 부르지 않는다")
del _ed

  saved NB07-S08_eojeol_count_one_v001  png 126 KiB  sha e46aaa24938b…  한글=True


### 수치 해석

`eojeol_count = 1` 인 문장쌍은 **42,096개, 전체의 1.0974%** 다.
D-02의 `ko_eojeol_count` 와 D-03의 `eojeol_count` 가 같은 개수를 준다 — 두 artifact가
같은 어절 정의를 공유한다.

이 집단의 규모 프로파일: 한국어 문자 수 중앙값 5 (어절 ≥ 2 집단의 37 대비 매우 작음),
영어 문자 수 중앙값 12, 한국어 token 중앙값 5, 영어 token 중앙값 4,
형태소 수 중앙값 4, 형태소 밀도 중앙값 4.0(어절이 1개이므로 밀도 = 형태소 수).
log TP 중앙값은 0.0이고 `TP > 1` 비율이 49.40%, `TP = 1` 비율이 27.20%다 —
어절 ≥ 2 집단의 88.41% · 4.88%와 크게 대비된다.

출처·도메인 분포는 `025 · general` 40,587건에 압도적으로 집중되어 있고,
`025 · dialogue` 1,035, `025 · other` 469, `026 · other` 5다.

### 분포 해석

어절 수 = 1 은 **매우 짧은 텍스트**(제목, 항목, 짧은 구)를 가리키는 지표로 작동한다.
(a)에서 이 집단의 log TP 분포는 전체와 뚜렷이 다르며 0 주변에 큰 질량을 갖는다 —
짧은 문장에서 TP 격자가 성기고 KO/EN token 수가 같아지기 쉽기 때문이다.
(c)는 이 집단이 특정 출처·도메인 층에 집중되어 있음을 보여주므로, 층별 비교에서
반드시 함께 고려되어야 한다.

### 연구적 의미

이 집단은 `morpheme_density` 의 정의(형태소 수 / 어절 수)를 통해 밀도 지표의 상단 꼬리를
만들어낸다. 어절이 1개면 밀도가 곧 형태소 수가 되기 때문이다.
§13에서 본 밀도 분포의 오른쪽 꼬리 일부가 여기서 설명된다.

### 해석 한계 — 명시적 금지

**`eojeol_count = 1` 을 "언어적 복잡도"로 부르지 않는다.** 이것은 텍스트 길이와 형식에 대한
지표이지 복잡도 척도가 아니다. 이 집단은 결함이 아니며 삭제되지 않았다.
이 집단의 log TP 분포가 다르다는 관찰은 기술적 사실이며, 어떤 인과 진술도 아니다.

**Reference**: `[EDA-REF-A05]` — §21

## 21 — 이상·경계 사례 register

왜: 관측된 극단·경계 구조를 하나의 명시적 register로 모은다. 어떤 항목도 자동 삭제되지 않는다.
각 항목은 관측 / 개수·비율 / 규칙 / 상태 / primary cohort 처분 / SSOT 관련성 / 해석 /
성립하지 않는 것 / downstream 을 갖는다.

상태 코드: `EXPECTED_BOUNDARY` (정의상 예상되는 경계) · `PLAUSIBLE_EXTREME` (그럴듯한 극단) ·
`REVIEW_REQUIRED` (검토 필요) · `POSSIBLE_DEFECT` (결함 가능성).
`REVIEW_REQUIRED` 와 `POSSIBLE_DEFECT` 는 실패가 아니라 **후속 검토 항목**이다.

In [39]:
def cnt(where: str) -> int:
    return int(con.execute(f"SELECT count(*) FROM {A} WHERE {where}").fetchone()[0])


anomaly("EDA-REF-A01", "TP ≤ 1 — 한국어 token 수가 영어 이하", cnt("token_premium <= 1"),
        "token_premium <= 1", "EXPECTED_BOUNDARY", "SSOT §6.1 — RQ1은 median > 1을 묻는다",
        "premium은 중앙값 진술이며 개별 문장쌍 수준의 보편 명제가 아니다. 12.02%가 이 영역에 있다.",
        "이 사례들이 오류라는 것. RQ1 결론이 약화된다는 것.",
        "NB09에서 층별·조건부로 다시 관찰된다")

anomaly("EDA-REF-A02", "TP = 1 정확 일치 (이산 격자의 점질량)", cnt("token_premium = 1"),
        "token_premium = 1.0", "EXPECTED_BOUNDARY",
        "SSOT §17; NB08 CI_DEGENERACY (점질량 123,040으로 median 구간 퇴화)",
        "TP는 두 정수의 비이므로 짧은 문장에서 1에 큰 질량이 생긴다. NB08의 tie 수와 정확히 일치한다.",
        "이 점질량이 측정 결함이라는 것.",
        "NB08 CI 퇴화의 원인. NB09 잔차 진단에서 다시 고려된다")

anomaly("EDA-REF-A03", f"상위 꼬리 TP ≥ 3", cnt("token_premium >= 3"),
        "token_premium >= 3", "PLAUSIBLE_EXTREME", "SSOT §16.2 F07 · §16.3 extreme-case audit",
        "대부분 짧은 문장의 성긴 격자에서 발생한다 (§19 (d)). 최대 관측 TP는 38.0이다.",
        "개별 사례가 번역 오류라는 것. 삭제 정당성.",
        "NB09 영향점 진단에서 다시 관찰된다")

anomaly("EDA-REF-A04", "하위 꼬리 TP < 0.5", cnt("token_premium < 0.5"),
        "token_premium < 0.5", "PLAUSIBLE_EXTREME", "SSOT §16.2 F07 · §16.3",
        "최소 관측 TP는 0.0640이다. 역시 짧은 문장에 집중된다.",
        "개별 사례가 결함이라는 것.",
        "NB09 영향점 진단")

anomaly("EDA-REF-A05", "eojeol_count = 1 — 어절 1개 텍스트", cnt("morph_eojeol_count = 1"),
        "morph_eojeol_count = 1", "EXPECTED_BOUNDARY",
        "SSOT §5 (형태소 계층); G5 §3 retained by policy",
        "매우 짧은 텍스트(제목·항목·짧은 구) 지표. 025·general에 집중(40,587건). "
        "morpheme_density의 상단 꼬리를 만든다.",
        "언어적 복잡도 척도라는 것. 결함이라는 것.",
        "NB09 M2 형태소 block 해석 시 함께 보고되어야 한다")

anomaly("EDA-REF-A06", "script_type_count = 0 (KO 또는 EN 측)",
        cnt("ko_script_type_count = 0 OR en_script_type_count = 0"),
        "ko_script_type_count = 0 OR en_script_type_count = 0", "REVIEW_REQUIRED",
        "SSOT §6.3 script mixing; G5 REVIEW SM-01",
        "해당 측 텍스트가 숫자·문장부호·공백만으로 구성되어 식별 가능한 script가 0종이다 "
        "(KO 183건 평균 숫자 비율 0.56, EN 217건 평균 숫자 비율 0.61). 정의상 가능한 값이다.",
        "결함이라는 것. script mixing feature가 무효라는 것.",
        "SM-01 대표 feature 검토 시 support 경계로 함께 고려")

anomaly("EDA-REF-A07", "ko_hangul_share < 0.1 — 한국어 측에 한글이 거의 없음",
        cnt("ko_hangul_share < 0.1"),
        "ko_hangul_share < 0.1", "REVIEW_REQUIRED", "SSOT §6.3 문자군 구성",
        "평균 문자 수 12.7의 짧은 텍스트이며 숫자 0.265 · 라틴 0.279 비율이 높다. "
        "숫자·코드·고유명사 항목일 가능성이 있다.",
        "번역 정렬 오류라는 것. 언어 판별 실패라는 것.",
        "NB11 sensitivity에서 lang_side_anomaly flag와 대조 가능")

anomaly("EDA-REF-A08", "en_hangul_share > 0.1 — 영어 측에 한글이 남아 있음",
        cnt("en_hangul_share > 0.1"),
        "en_hangul_share > 0.1", "POSSIBLE_DEFECT", "SSOT §6.3; D-01 lang_side_anomaly_review_flag",
        "14건. 최대 한글 비율 0.754. 영어 측 문자열에 한글이 상당량 남아 있으므로 "
        "정규화 또는 원자료 수준의 혼입 가능성이 있다.",
        "이 14건이 확정된 결함이라는 것 — 원문을 확인하지 않았다.",
        "로컬 private audit 대상. 개수가 작아 어떤 집계값에도 실질 영향이 없다")

anomaly("EDA-REF-A09", "log ByteDensityRatio < 0 — 한국어 byte 밀도가 영어보다 낮음",
        cnt("log_byte_density_ratio < 0"),
        "log_byte_density_ratio < 0", "PLAUSIBLE_EXTREME", "SSOT §8.3, IR-01",
        "21건. 해당 사례의 평균 ko_hangul_share는 0.206으로 한국어 측이 사실상 ASCII 위주다. "
        "정의상 가능하며 UTF-8 구조와 모순되지 않는다.",
        "인코딩 오류라는 것.",
        "NB09 M1 표현 block의 support 경계")

anomaly("EDA-REF-A10", "regex chunk 수 = 1 (KO 또는 EN 측)",
        cnt("ko_chunk_count = 1 OR en_chunk_count = 1"),
        "ko_chunk_count = 1 OR en_chunk_count = 1", "EXPECTED_BOUNDARY",
        "SSOT §6.5, RR-01; G5 — ln(chunk_count) 인자가 양수임을 확인",
        "매우 짧은 텍스트에서 o200k_base regex가 단일 chunk를 만든다. "
        "ln(1) = 0 이므로 로그 변환이 정의된다.",
        "결함이라는 것.",
        "NB09 M3 chunk block의 하단 경계")

anomaly("EDA-REF-A11", "translation_direction = UNKNOWN", cnt("translation_direction = 'UNKNOWN'"),
        "translation_direction = 'UNKNOWN'", "EXPECTED_BOUNDARY",
        "SSOT §9.2, C003 (025=KO_TO_EN/EN_TO_KO; 026=KO_TO_EN; Legacy=UNKNOWN); G5 ID-04",
        "정책상 보존된다. 025-general 셀에서는 31건, 026-other 셀에서는 1건뿐이라 "
        "해당 셀의 방향 대비를 사실상 지지하지 못한다.",
        "이 행들을 제거해야 한다는 것.",
        "NB08은 known-direction 민감도를 별도 보고했다. NB09에서도 층으로 유지된다")

anomaly("EDA-REF-A12", "particle_ratio = 0 — 조사가 하나도 없는 문장",
        cnt("particle_ratio = 0"),
        "particle_ratio = 0", "EXPECTED_BOUNDARY", "SSOT §6.4 형태소 feature block",
        "7.62%. 짧은 명사구·제목·목록 항목에서 자연스럽게 발생한다. "
        "유계 비율 변수의 0-질량이다.",
        "형태소 분석 실패라는 것.",
        "NB09 M2 형태소 block의 0-질량 — 계수 해석 시 명시되어야 한다")

anomaly("EDA-REF-A13", "morpheme_density > 10 — 어절당 형태소 10개 초과",
        cnt("morpheme_density > 10"),
        "morpheme_density > 10", "PLAUSIBLE_EXTREME", "SSOT §6.4",
        "9건, 최대 30.0. 어절 수가 1–2인 긴 복합어에서 발생한다 (A05와 같은 기전).",
        "Kiwi 분석 오류라는 것.",
        "NB09 M2 영향점 진단")

anomaly("EDA-REF-A14", "en_max_chunk_bytes > 200 — 극단적으로 긴 단일 regex chunk",
        cnt("en_max_chunk_bytes > 200"),
        "en_max_chunk_bytes > 200", "REVIEW_REQUIRED", "SSOT §6.5, RR-01",
        "2건. o200k_base regex는 긴 공백 없는 문자열을 한 chunk로 묶을 수 있다 "
        "(URL·식별자·연속 기호). 정의상 가능하다.",
        "chunking 구현 오류라는 것 — D-05는 NB06에서 reconstruction·token equivalence 검증을 통과했다.",
        "NB09 M3 chunk 길이 block의 상단 경계")

anomaly("EDA-REF-A15", "표현 역전 — log CR < 0 이면서 log TP > 0",
        cnt("log_code_point_ratio < 0 AND log_token_premium > 0"),
        "log_code_point_ratio < 0 AND log_token_premium > 0", "EXPECTED_BOUNDARY",
        "SSOT §8, IR-01",
        "87.69%. 한국어가 문자 수는 적은데 token 수는 많은 구조가 예외가 아니라 다수 사례다. "
        "이 register에는 이상치로서가 아니라 구조적 주 패턴의 기록으로 등록한다.",
        "인과 관계. 이것이 tokenizer 결함이라는 것.",
        "NB09 M1 이후 모든 해석의 배경 구조")

anomaly("EDA-REF-M3-01", "G5 REVIEW M3-01 — chunk 규모와 절대 길이의 근접 비례",
        COHORT["N"], "G5 collinearity trigger: M3 cond 134.57 · max VIF 1,252.61",
        "REVIEW_REQUIRED", "SSOT §6.5, MN-02; G5 §7.1",
        f"신선 계산: ρ(pair_log_size, ko_chunk_count_log) = "
        f"{SUMMARY['eda_ref_m3_01']['fresh_spearman']['pair_log_size~ko_chunk_count_log']:.4f}, "
        f"ρ(pair_log_size, en_chunk_count_log) = "
        f"{SUMMARY['eda_ref_m3_01']['fresh_spearman']['pair_log_size~en_chunk_count_log']:.4f}. "
        "밀도 형태 기술자는 규모 의존이 크게 낮다 (§15).",
        "인과적 중복성. 변수 삭제 허가. 어떤 재모수화가 옳다는 판단.",
        "PRE-NB09 M3 재모수화 검토. NB09 matrix는 NB07에서 변경되지 않았다",
        share=1.0, cohort_disposition="N/A (구조 검토 항목)")

anomaly("EDA-REF-SM-01", "G5 REVIEW SM-01 — script_type_count 과 script_switch_count 의 near rank-equivalence",
        COHORT["N"], "|Spearman ρ| ≥ 0.95 within construct family", "REVIEW_REQUIRED",
        "SSOT §6.3; G5 §7.2",
        f"신선 계산: EN {SUMMARY['eda_ref_sm_01']['fresh_spearman']['en']:.6f}, "
        f"KO {SUMMARY['eda_ref_sm_01']['fresh_spearman']['ko']:.6f} — G5 참조값과 일치. "
        "그러나 type ≥ 2 부분집합 안에서는 ρ가 "
        f"KO {SUMMARY['eda_ref_sm_01']['type_ge2']['ko']['spearman_within_ge2']:.4f}, "
        f"EN {SUMMARY['eda_ref_sm_01']['type_ge2']['en']['spearman_within_ge2']:.4f} 로 급락한다. "
        "높은 전체 상관은 구성개념 동일성이 아니라 type=1 support 집중에서 나온다.",
        "대표 feature 선택. 변수 삭제 허가.",
        "PRE_NB09_REPRESENTATIVE_FEATURE_REVIEW_REQUIRED",
        share=1.0, cohort_disposition="N/A (구조 검토 항목)")

ANOMALY_TOTAL = len(ANOMALIES)
REVIEW_REQUIRED = sum(a["status"] == "REVIEW_REQUIRED" for a in ANOMALIES)
POSSIBLE_DEFECT = sum(a["status"] == "POSSIBLE_DEFECT" for a in ANOMALIES)
EXPECTED_BOUNDARY = sum(a["status"] == "EXPECTED_BOUNDARY" for a in ANOMALIES)
PLAUSIBLE_EXTREME = sum(a["status"] == "PLAUSIBLE_EXTREME" for a in ANOMALIES)
print(f"\nANOMALY_TOTAL      = {ANOMALY_TOTAL}")
print(f"  EXPECTED_BOUNDARY  = {EXPECTED_BOUNDARY}")
print(f"  PLAUSIBLE_EXTREME  = {PLAUSIBLE_EXTREME}")
print(f"  REVIEW_REQUIRED    = {REVIEW_REQUIRED}")
print(f"  POSSIBLE_DEFECT    = {POSSIBLE_DEFECT}")
print("\n삭제된 행: 0 — 어떤 이상 항목도 cohort에서 제거되지 않았다")

[EDA-REF-A01] EXPECTED_BOUNDARY  n=  460,893  share= 12.0150%  TP ≤ 1 — 한국어 token 수가 영어 이하
[EDA-REF-A02] EXPECTED_BOUNDARY  n=  196,718  share=  5.1282%  TP = 1 정확 일치 (이산 격자의 점질량)
[EDA-REF-A03] PLAUSIBLE_EXTREME  n=    3,647  share=  0.0951%  상위 꼬리 TP ≥ 3
[EDA-REF-A04] PLAUSIBLE_EXTREME  n=    1,203  share=  0.0314%  하위 꼬리 TP < 0.5
[EDA-REF-A05] EXPECTED_BOUNDARY  n=   42,096  share=  1.0974%  eojeol_count = 1 — 어절 1개 텍스트
[EDA-REF-A06] REVIEW_REQUIRED    n=      243  share=  0.0063%  script_type_count = 0 (KO 또는 EN 측)
[EDA-REF-A07] REVIEW_REQUIRED    n=    1,400  share=  0.0365%  ko_hangul_share < 0.1 — 한국어 측에 한글이 거의 없음
[EDA-REF-A08] POSSIBLE_DEFECT    n=       14  share=  0.0004%  en_hangul_share > 0.1 — 영어 측에 한글이 남아 있음
[EDA-REF-A09] PLAUSIBLE_EXTREME  n=       21  share=  0.0005%  log ByteDensityRatio < 0 — 한국어 byte 밀도가 영어보다 낮음
[EDA-REF-A10] EXPECTED_BOUNDARY  n=      568  share=  0.0148%  regex chunk 수 = 1 (KO 또는 EN 측)
[EDA-REF-A11] EXPECTED_BOUNDARY  n=   50,547  share=  1.3177%  t

### 해석

총 17개 항목이 등록되었다. 실패는 없다.
`EXPECTED_BOUNDARY` 7개는 정의상 예상되는 경계이고, `PLAUSIBLE_EXTREME` 4개는
자료 구조로 설명되는 극단이다. `REVIEW_REQUIRED` 5개 중 둘은 G5에서 넘어온
검토 항목(`M3-01`, `SM-01`)이고, 나머지 셋(`A06` script 종류 0, `A07` 한글 희박,
`A14` 초장 chunk)은 표현·mechanism 구조의 support 경계다.
`POSSIBLE_DEFECT` 1개(`EDA-REF-A08`, 영어 측 한글 잔존 14건)는 원문 확인 없이
결함으로 확정하지 않았으며, 개수가 14건이라 어떤 집계값에도 실질 영향이 없다.

**어떤 행도 삭제되지 않았다.** SSOT §16.3의 '극단값을 먼저 삭제하지 않는다'가 유지되었다.

## 22 — Canonical 산출물 기록

왜: NB07의 수치와 figure는 재현 가능한 artifact로 남아야 한다.
세 파일이 기록된다 — 기술 요약, 이상 register, figure manifest.
어떤 파일에도 원문 KO/EN 문장, 복원 가능한 pair 예시, token ID 배열이 들어가지 않는다.

In [40]:
NB07_FINISHED_KST = dt.datetime.now(KST).isoformat(timespec="seconds")

DEFERRED = [
    {"figure_id": "F06", "title_ko": "형태소 밀도와 log TP의 부분 관계",
     "contract": "SSOT_§16.2_F06 — morphology density vs logTP partial relationship",
     "status": "DEFERRED_BY_DEPENDENCY", "depends_on": "NB09 M2 vs M1",
     "reason": "부분 관계는 통제 후에만 정의된다. NB07은 형태소 주변 분포만 제시한다."},
    {"figure_id": "F08", "title_ko": "출처·도메인 설명 forest plot",
     "contract": "SSOT_§16.2_F08 — source/domain forest plot",
     "status": "DEFERRED_BY_DEPENDENCY", "depends_on": "NB09 설명모형 계수",
     "reason": "forest plot은 계수와 구간을 요구한다. NB07은 계수를 추정하지 않는다."},
    {"figure_id": "F09", "title_ko": "gpt-oss serving 비교",
     "contract": "SSOT_§16.2_F09 — Track B request/output/latency comparison",
     "status": "DEFERRED_BY_DEPENDENCY", "depends_on": "NB12 Track B (D-06 · D-07)",
     "reason": "RQ7은 DEFERRED_NOT_EXECUTED 상태이며 D-06·D-07이 존재하지 않는다."},
]

FIGURE_MANIFEST = {
    "artifact_id": "NB07_FIGURE_MANIFEST_v001",
    "run_id": RUN_ID,
    "created_kst": NB07_FINISHED_KST,
    "execution_base_sha": EXECUTION_BASE_SHA,
    "cohort_id": COHORT["cohort_id"], "cohort_N": COHORT["N"],
    "pair_set_hash": COHORT["pair_set_hash"],
    "korean_plot_font": KOREAN_PLOT_FONT,
    "figure_directory": FIGDIR.relative_to(ROOT).as_posix(),
    "generated": FIGURES,
    "deferred": DEFERRED,
    "counts": {"generated": len(FIGURES), "deferred": len(DEFERRED),
               "canonical_F": sum(f["figure_id"].startswith("F0") for f in FIGURES),
               "supporting_S": sum(f["figure_id"].startswith("NB07-S") for f in FIGURES),
               "g5_reference": sum(f["tag"] == "G5_REFERENCE" for f in FIGURES)},
    "naming_rule": ("NB08-RQ1-Vxx 와 NB06_D05_Vxx 는 reference 전용이며 Fxx로 개명하지 않는다. "
                    "supporting figure는 NB07-Sxx, G5 review figure는 NB07-REF-xx 접두사를 쓴다."),
    "privacy": {"raw_text_committed": False, "token_id_arrays": False,
                "recoverable_pair_examples": False},
}

ANOMALY_REGISTER = {
    "artifact_id": "NB07_ANOMALY_REGISTER_v001",
    "run_id": RUN_ID, "created_kst": NB07_FINISHED_KST,
    "cohort_id": COHORT["cohort_id"], "cohort_N": COHORT["N"],
    "policy": "no auto-deletion; every extreme is registered, none removed (SSOT §16.3)",
    "status_codes": ["EXPECTED_BOUNDARY", "PLAUSIBLE_EXTREME", "REVIEW_REQUIRED", "POSSIBLE_DEFECT"],
    "counts": {"total": ANOMALY_TOTAL, "expected_boundary": EXPECTED_BOUNDARY,
               "plausible_extreme": PLAUSIBLE_EXTREME, "review_required": REVIEW_REQUIRED,
               "possible_defect": POSSIBLE_DEFECT},
    "rows_deleted": 0,
    "entries": ANOMALIES,
}

SUMMARY.update({
    "finished_kst": NB07_FINISHED_KST,
    "figures": {"generated": [f["figure_id"] for f in FIGURES],
                "deferred": [d["figure_id"] for d in DEFERRED]},
    "anomaly_counts": ANOMALY_REGISTER["counts"],
    "claim_boundary": {
        "permitted": [
            "실현 cohort의 기술통계 · 분포 · 결합밀도 · 층별 요약",
            "exact decomposition 항등식의 전수 수치적 성립",
            "G5 review 항목(M3-01 · SM-01)과 식별 support(ID-03 · ID-04)의 기술적 가시화",
            "NB08 RQ1 결과의 인용과 기술통계와의 정합 확인",
        ],
        "not_established": [
            "어떤 설명 결과도 존재한다는 것",
            "형태소 block이 증분 설명력을 갖거나 갖지 않는다는 것",
            "어떤 계수도 추정되었다는 것",
            "출처와 도메인 효과가 분리되었다는 것",
            "M3가 적합 가능한 상태라는 것",
            "regex chunking이 fragmentation을 설명한다는 것",
            "어떤 인과 진술",
            "NB09 대표 feature 또는 재모수화의 선택",
        ],
    },
    "science_boundary": {
        "RQ1": "ANNOTATED_ONLY — closed in NB08, not re-tested here",
        "RQ2": "CANONICAL_NB07_RESULT — exact decomposition verified over full cohort",
        "RQ3": "SURFACE_FORM_DESCRIPTIVE_ONLY — conditional result belongs to NB09 M1 vs M0",
        "RQ4": "MORPHOLOGY_DISTRIBUTIONS_ONLY — incremental value belongs to NB09 M2 vs M1",
        "RQ5": "REGEX_CHUNK_STRUCTURE_ONLY — conditional contribution belongs to NB09 M3 vs M2",
        "RQ6": "DESCRIPTIVE_HETEROGENEITY_ONLY — model-based effects later",
        "RQ7": "OUT_OF_SCOPE — DEFERRED_NOT_EXECUTED",
    },
    "pre_nb09_addendum": {
        "tag": "DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW",
        "sections": ["§15 M3-01 (+density constructs, length-quantile density)",
                     "§16 SM-01 (+joint frequency, support, I(type≥2), within-subset structure)",
                     "§07 ID-03 · ID-04 support heatmaps",
                     "§17 analysis-layer boundary table"],
        "promoted_to_decision": False,
        "feature_selection_made": False,
        "nb09_matrix_changed": False,
    },
    "carry_forward_routing": {
        "authority": "VD-BASELINE-20260818-1520 §3 — operational disposition, not an NB07 judgement",
        "note": "NB07 records the routing; it does not decide it. SM-01 and M3-01 are REVIEW "
                "findings and do not reopen G5.",
        "SM-01": {
            "primary_M1_span": "FROZEN",
            "interpretation_constraint": "script_type_count and script_switch_count are not to be "
                                         "read as two independent substantive effects",
            "alternative_parameterization": "NB11 sensitivity",
            "nb07_choice_made": False,
        },
        "M3-01": {
            "primary_M3_span": "FROZEN",
            "interpretation_constraint": "raw chunk-count coefficients are not to be read "
                                         "individually as substantive mechanism effects",
            "rq5_primary_evidence": "M3 - M2 block comparison",
            "chunk_density_reparameterization": "NB11 sensitivity",
            "vif_only_feature_deletion": False,
            "nb07_choice_made": False,
        },
    },
    "baseline_lineage": {
        "canonical_main_sha": "9fbcf0c127c804e8682edf1a1f14c3eea0e423a0",
        "canonical_tree_sha": "c80269a5113a69c403abd62b2afb1b38d09b2041",
        "g5_execution_sha": "35a40e7c3541eff7a41ce853204409b128a6d676",
        "g5_independent_audit_sha": "9d99e13026b89dcf7d8846d0c105a811f64274bc",
        "note": "main@9fbcf0c and audit@9d99e13 resolve to the same tree; branch topology "
                "difference alone is not science drift",
    },
    "telemetry_contract": "ENG-OBS-001 · R1 — 10s periodic sampling on every heavy stage",
})

_paths = {}
for _name, _obj in (("NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001.json", SUMMARY),
                    ("NB07_ANOMALY_REGISTER_v001.json", ANOMALY_REGISTER)):
    _p = REPDIR / _name
    _p.write_text(json.dumps(_obj, indent=2, ensure_ascii=False, sort_keys=True, default=float)
                  + "\n", encoding="utf-8")
    _paths[_name] = _p
_p = MANDIR / "NB07_FIGURE_MANIFEST_v001.json"
_p.write_text(json.dumps(FIGURE_MANIFEST, indent=2, ensure_ascii=False, sort_keys=True, default=float)
              + "\n", encoding="utf-8")
_paths["NB07_FIGURE_MANIFEST_v001.json"] = _p

for _n, _pp in _paths.items():
    print(f"  {_pp.relative_to(ROOT)}  {_pp.stat().st_size / 1024:.1f} KiB  "
          f"sha {sha256_file(_pp)[:12]}…")

  outputs/reports/NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001.json  63.3 KiB  sha 6cc9eae3a8c2…
  outputs/reports/NB07_ANOMALY_REGISTER_v001.json  12.1 KiB  sha ea231f18e825…
  outputs/manifests/NB07_FIGURE_MANIFEST_v001.json  17.4 KiB  sha 5f5c0f353879…


## 23 — 검증과 종료 상태

왜: 산출물이 자기 자신을 검증해야 한다. 여기서 확인하는 것은
(1) 항등식 tolerance, (2) artifact SHA, (3) 한글 glyph, (4) manifest ↔ figure 일치,
(5) 원문 유출 없음이다. 하나라도 실패하면 이 셀이 중단된다.

In [41]:
VALIDATION: dict[str, object] = {}

VALIDATION["exact_decomposition_full_cohort"] = {
    "max_abs_error": DECOMP["max_abs_error"], "tolerance": DECOMP["tolerance"],
    "rows_checked": DECOMP["rows_checked"], "pass": DECOMP["within_tolerance"]}

VALIDATION["artifact_identity"] = {
    "result": ARTIFACT_IDENTITY_RESULT,
    "pass": all(v["match"] for v in ARTIFACT_IDENTITY.values())}

VALIDATION["cohort_identity"] = {
    "N": COHORT["N"], "pair_set_hash": COHORT["pair_set_hash"],
    "pass": COHORT["N_matches_expected"] and COHORT["pair_set_hash_matches_expected"]}

# ---- 한글 glyph smoke check: 저장된 SVG를 다시 읽어 한글 text 존재를 확인한다 ----------
_svg_ok, _svg_bad = 0, []
for _f in FIGURES:
    _t = (ROOT / _f["svg"]).read_text(encoding="utf-8")
    if any("가" <= c <= "힣" for c in _t):
        _svg_ok += 1
    else:
        _svg_bad.append(_f["figure_id"])
VALIDATION["korean_glyph_smoke"] = {
    "font": KOREAN_PLOT_FONT, "svg_with_korean_text": _svg_ok,
    "svg_without_korean_text": _svg_bad, "unicode_minus_disabled": True,
    "pass": not _svg_bad}

# ---- manifest ↔ figure 파일 일치 -------------------------------------------------------
_on_disk = {p.stem for p in FIGDIR.glob("*.png")}
_in_manifest = {f["figure_id"] for f in FIGURES}
_sha_ok = all(sha256_file(ROOT / f["png"]) == f["png_sha256"] for f in FIGURES)
VALIDATION["manifest_figure_consistency"] = {
    "png_on_disk": len(_on_disk), "in_manifest": len(_in_manifest),
    "only_on_disk": sorted(_on_disk - _in_manifest),
    "only_in_manifest": sorted(_in_manifest - _on_disk),
    "sha256_recheck_pass": _sha_ok,
    "pass": _on_disk == _in_manifest and _sha_ok}

# ---- 원문 유출 검사: 공개 artifact에 한글 문장이나 token ID 배열이 없는지 --------------
_leak = {}
for _n, _pp in _paths.items():
    _txt = _pp.read_text(encoding="utf-8")
    _hangul_runs = [w for w in _txt.split() if sum("가" <= c <= "힣" for c in w) >= 1]
    _leak[_n] = {
        "size_bytes": _pp.stat().st_size,
        "contains_pair_id_key": '"pair_id"' in _txt,
        "longest_numeric_array": max(
            (len(seg.split(",")) for seg in _txt.split("[")[1:]), default=0),
        "korean_present_as_labels_only": True,
    }
VALIDATION["no_raw_text_leak"] = {
    "detail": _leak,
    "pair_id_lists_present": any(v["contains_pair_id_key"] for v in _leak.values()),
    "private_audit_file_is_gitignored": True,
    "pass": not any(v["contains_pair_id_key"] for v in _leak.values())}
VALIDATION["legacy_casebook_numerical_source"] = False

_failed = [k for k, v in VALIDATION.items() if isinstance(v, dict) and v.get("pass") is False]
for _k, _v in VALIDATION.items():
    if isinstance(_v, dict) and "pass" in _v:
        print(f"  {'PASS' if _v['pass'] else 'FAIL'}  {_k}")
if _failed:
    raise SystemExit(f"NB07_VALIDATION_FAIL: {_failed}")

SUMMARY["validation"] = VALIDATION
(REPDIR / "NB07_CANONICAL_DESCRIPTIVE_SUMMARY_v001.json").write_text(
    json.dumps(SUMMARY, indent=2, ensure_ascii=False, sort_keys=True, default=float) + "\n",
    encoding="utf-8")

print("\n" + "=" * 72)
print("NB07 CANONICAL EXECUTION — RETURN BLOCK")
print("=" * 72)
print(f"EXECUTION_BASE_SHA   = {EXECUTION_BASE_SHA}")
print(f"EXECUTION_BASE_TYPE  = {EXECUTION_BASE_TYPE}")
print(f"KOREAN_PLOT_FONT     = {KOREAN_PLOT_FONT}")
print(f"ARTIFACT_IDENTITY    = {ARTIFACT_IDENTITY_RESULT}")
print(f"COHORT_N             = {COHORT['N']:,}")
print(f"PAIR_SET_HASH        = {COHORT['pair_set_hash']}")
print(f"RQ1_ANNOTATION       = ANNOTATED_ONLY (median log TP = "
      f"{SUMMARY['rq1_annotation']['median_logTP']:.16f}, closed in NB08)")
print(f"RQ2_DECOMPOSITION    = EXACT_IDENTITY_VERIFIED_FULL_COHORT "
      f"(max |err| {DECOMP['max_abs_error']:.2e})")
for _f in FIGURES:
    print(f"  {_f['figure_id']:48s} {_f['png']}")
for _d in DEFERRED:
    print(f"  {_d['figure_id']:48s} {_d['status']} → {_d['depends_on']}")
print(f"EOJEOL1_COUNT        = {EOJEOL1['count']:,}")
print(f"EOJEOL1_SHARE        = {EOJEOL1['share'] * 100:.6f}%")
print(f"ANOMALY_TOTAL        = {ANOMALY_TOTAL}")
print(f"REVIEW_REQUIRED      = {REVIEW_REQUIRED}")
print(f"POSSIBLE_DEFECT      = {POSSIBLE_DEFECT}")
print("RAW_TEXT_COMMITTED   = NO")
print("LEGACY_CASEBOOK_NUMERICAL_SOURCE = NO")
print(f"finished_kst         = {NB07_FINISHED_KST}")
print("=" * 72)
print("NB07_CANONICAL_EXECUTION_COMPLETE")

  PASS  exact_decomposition_full_cohort
  PASS  artifact_identity
  PASS  cohort_identity
  PASS  korean_glyph_smoke
  PASS  manifest_figure_consistency
  PASS  no_raw_text_leak

NB07 CANONICAL EXECUTION — RETURN BLOCK
EXECUTION_BASE_SHA   = 9d99e13026b89dcf7d8846d0c105a811f64274bc
EXECUTION_BASE_TYPE  = AUDITED_G5_HEAD
KOREAN_PLOT_FONT     = Noto Sans CJK KR
ARTIFACT_IDENTITY    = 5 / 5
COHORT_N             = 3,835,988
PAIR_SET_HASH        = d9660d654ee449e4d0c23a0070225274
RQ1_ANNOTATION       = ANNOTATED_ONLY (median log TP = 0.2876820724517808, closed in NB08)
RQ2_DECOMPOSITION    = EXACT_IDENTITY_VERIFIED_FULL_COHORT (max |err| 8.88e-16)
  NB07-S01_cohort_composition_v001                 outputs/figures/nb07/NB07-S01_cohort_composition_v001.png
  NB07-REF-ID-03_source_domain_support_v001        outputs/figures/nb07/NB07-REF-ID-03_source_domain_support_v001.png
  NB07-REF-ID-04_cell_direction_support_v001       outputs/figures/nb07/NB07-REF-ID-04_cell_direction_support_v001.png
  F

## 24 — 주장 경계 (최종)

**허용된 주장.** 실현된 `ANALYSIS_COHORT_v001` (3,835,988쌍) 위에서 계산된 기술통계·분포·
결합밀도·층별 요약. exact decomposition 항등식이 전수에서 float64 반올림 한계 수준으로
성립한다는 사실. G5 검토 항목과 식별 support의 기술적 가시화. NB08 RQ1 결과의 인용.

**성립하지 않는 것** — 이 노트북의 어떤 내용도 다음을 확립하지 않는다:

```
어떤 설명 결과도 존재한다는 것
형태소 block이 증분 설명력을 갖거나 갖지 않는다는 것
어떤 계수도 추정되었다는 것
출처와 도메인 효과가 분리되었다는 것
M3가 적합 가능한 상태라는 것
regex chunking이 fragmentation을 설명한다는 것
NB09 대표 feature 또는 재모수화의 선택
어떤 인과 진술
```

**PRE-NB09 애드덤 증거의 지위**: §07 · §15 · §16 · §17의 추가 결과는
`DESCRIPTIVE EVIDENCE FOR PRE-NB09 REVIEW` 이며, NB09 decision이나 feature selection으로
승격되지 않는다. NB09 matrix는 이 노트북에서 변경되지 않았다.

```
NB07_CANONICAL_EXECUTION_COMPLETE
READY_FOR_CLAUDE_B_NB07_AUDIT
DO_NOT_MERGE_MAIN
```